In [ ]:
import os
os.environ['SYNTH_GRIDSEARCH'] = '0'
os.environ['SYNTH_NATIVE_COMPARE'] = '0'
os.environ['SYNTH_NATIVE_LAYOUT'] = '0'
os.environ['SYNTH_NATIVE_SIM'] = '0'
os.environ['SYNTH_TEMPORAL'] = '0'
os.environ['SYNTH_SWEEP'] = '0'
os.environ['SYNTH_BLURSWEEP'] = '0'
os.environ['QR_NATGALLERY'] = '0'
os.environ['DSBUILD_RUN'] = '1'
# Keep the build small and sequence-heavy: Exp204 needs labelled lineages,
# not static pretraining volumes.  This is a feasibility set, not a model.
os.environ['DS_TARGET_GB'] = '0.25'
os.environ['DS_SEQ_LEN'] = '6'
os.environ['DS_SEQ_FRAC'] = '0.99'
os.environ['DS_BUDGET_H'] = '1.0'
os.environ['DS_DIV_RATE'] = '0.05'
os.environ['SYNTH_PLACE_MODE'] = 'shell'
print('EXP204 MINI SYNTHETIC DATASET: lineage feasibility build')


# Biohub Exp204 — capped synthetic lineage builder\n\nPrivate derivative of `josefreitasalvesneto/biohub-synthetic-dataset`. The public output was not attachable on 2026-08-19, so this runs the public generator unchanged except for the bounded configuration in the first cell. It produces a small sequence-heavy output for Exp204 only.\n

# Biohub Synth Gen - Gerador de Dataset Sintetico 3D + Validacao Estatistica

**Competicao:** Biohub - Cell Tracking During Development (Kaggle, US$60k). Detectar/rastrear
nucleos de embriao de zebrafish em microscopia 3D+tempo. So **2 embrioes** rotulados no treino,
rotulos **esparsos** (~2.8%) -> qualquer detector aprendido tende a overfitar. **Estrategia deste
notebook:** gerar um dataset sintetico 3D **100% rotulado** que **representa a realidade** (medida
por teste de hipotese sobre muitas distribuicoes) para treinar um detector com sim-to-real.

## O que este notebook faz
1. **Extrai a verdade real:** GT rotulado + DoG precision-first -> banco de TEMPLATES de celulas
   reais (aparencia 3D autentica) + campos de fundo reais + estatisticas (tamanho, brilho, NN, densidade).
2. **Modela o "liquido":** fundo/tecido real, perfil de profundidade Z, referencia canonica p/ HM,
   distribuicao empirica de LUZ onde as celulas estao, e a TEXTURA RESIDUAL real (alta-frequencia).
3. **Gera volumes** (fundo real + celulas), 100% rotulados em caixas 3D (formato DETR), com pipeline
   fundamentado na literatura (CytoPacq/MitoGen): elipsoide/template -> textura de cromatina ->
   PSF Gaussiana anisotropica -> ruido Poisson-Gaussiano (sCMOS) -> histogram matching -> perfil Z.
4. **Valida por TESTE DE HIPOTESE:** para 8 distribuicoes (intensidade de voxel, media/pico/tamanho/
   contraste por celula, distancia NN, perfil Z, gradiente/textura) roda KS 2-amostras + energy
   distance + Anderson-Darling contra o real. H0 = "mesma distribuicao".

## Evolucao (recursao guiada por dados) — resultados-chave documentados abaixo
- **MODO VENCEDOR = HIBRIDO:** celula de **template REAL** (aparencia/tamanho/textura = reais por
  construcao) + **placement/tecido do param** (o que melhor casa espaco/global). Derrubou as
  marginais celulares: cell_mean 0.44->0.18, cell_peak 0.49->0.21 (KS).
- **Busca exaustiva:** 41 configs de grid + 36 iteracoes de 2 otimizadores hill-climbing + 4 modos
  geradores + levers de modelo (bootstrap de intensidade, jitter, layout real, field-smooth,
  residual real pre/pos-blur, soft-blend de borda, count-mult).

## STATUS ATUAL (180 epocas MINIMAX + 100% do dataset + DIVISAO modelada, held-out anti-leakage)
- **100% DO DATASET:** validacao held-out sobre TODOS os ~199 videos dos 2 embrioes (GEN 140 + REF disjunto),
  nao mais 40 videos. A referencia real mais rica possivel.
- **DIVISAO CELULAR modelada e validada:** distribuicao `div_frac` (fracao de pares proximos = celulas
  dividindo/halteres) entrou no KS (14 distribuicoes). Taxa OTIMA encontrada = 6% (SYNTH_DIVRATE=0.06),
  casando o real (div_frac KS 0.23). Sister-dist e taxa tunaveis pelo otimizador.
- **RESULTADO (100 rodadas, 100% dataset): KS medio 0.34 -> 0.225.** Excelentes: cell_mean 0.07, zprof
  0.07, grad_mag 0.10, slice_var 0.11, count 0.14, local_std 0.16, div_frac 0.23. Gargalo: hi_freq (0.64,
  altissima-freq). Config: MODE=layout + DIVRATE 0.06 + PSMATCH 0.8 (hicut 0.25) + textura 0.30 + tecido 0.50.
--- (historico) 80 epocas anteriores (40 videos): ---
- **VALIDACAO ANTI-LEAKAGE:** os videos que GERAM o synth sao disjuntos dos que VALIDAM o KS (held-out).
  Isso corrigiu um vies: o KS "vazado" (mesmos videos) inflava ~20% (0.72 vazado vs 0.60 honesto).
- **13 distribuicoes testadas** (angulos de comparacao): voxel_int, cell_mean/peak/size/range, nn_um,
  zprof, grad_mag, contrast, local_std, slice_var, count, hi_freq (energia de alta-freq por FFT).
- **METODOS NOVOS que furaram os pisos:** (a) POWER-SPECTRUM MATCHING de Fourier BAND-LIMITED (casa a
  magnitude FFT=textura real na alta-freq, preserva a fase=estrutura) -> grad_mag 0.65->0.17, local_std
  0.69->0.25; (b) HM pos-PSMATCH (restaura o histograma de intensidade sem perder textura) -> voxel_int
  0.40->0.16; (c) OBJETIVO MINIMAX (score = 1-KS_medio - 0.6*pior_KS) -> baixa TODAS juntas, sem sacrificar.
- **RESULTADO (80 epocas minimax, held-out): pior KS 0.58->0.404, KS medio 0.34->0.232.** ~10/13
  distribuicoes com bom match. Config vencedora = MODE=layout + PSMATCH 0.8 (hicut 0.25) + textura 0.45
  + Poisson 450 + soft-blend + HM-fix. Gargalo restante: cell_peak/cell_range (0.40, pico de intensidade).
- **LICAO de LB (conclusiva):** a recursao deixou o synth muito proximo do real (KS medio 0.23), MAS as
  3 fusoes DoG+DETR testadas no LB (ADD 0.82/0.75, SWAP n_pred-fixo 0.827) TODAS < DoG-max puro 0.836.
  O gargalo do leaderboard e' a GENERALIZACAO com so 2 embrioes, NAO a fidelidade do synth. Melhor
  autentico no LB = DoG-max classico + line-fit 0.836; o valor deste gerador e' cientifico/metodologico.

> Secoes: **[A] nucleo do gerador** -> **[B] estudos do real** -> **[C] geracao + validacao** ->
> **[D] SHOWCASE: pipeline passo-a-passo + relatorio de testes de hipotese (held-out anti-leakage)**.

In [ ]:
import os, sys, json, glob, math, subprocess, importlib.util as _ilu
from pathlib import Path
import numpy as np

def _pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=False)
if os.environ.get("BIOHUB_SMOKE", "0") != "1" and any(_ilu.find_spec(m) is None for m in ("tracksdata", "zarr", "blosc2")):
    import importlib.metadata as _md
    try: NPV = _md.version("numpy")
    except Exception: NPV = None
    pk = ["zarr>=3.0.10,<4", "tracksdata", "geff", "blosc2", "scikit-image", "scikit-learn"]
    if NPV: pk.append(f"numpy=={NPV}")
    _pip(*pk)
    if NPV: _pip("--force-reinstall", "--no-deps", f"numpy=={NPV}")

import matplotlib
try:
    from IPython import get_ipython
    if get_ipython() is not None: get_ipython().run_line_magic("matplotlib", "inline")
    else: matplotlib.use("Agg")
except Exception: matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, maximum_filter, zoom as ndzoom, convolve as ndconvolve

VOXEL_UM = np.array([1.625, 0.40625, 0.40625]); DOWNSAMPLE = (1, 4, 4)
EFF_VOXEL = VOXEL_UM * np.array(DOWNSAMPLE)
# RAIO de tolerancia p/ extrair templates do REAL. GRID SEARCH (biohub-radius-sweep, anti-leakage):
#   9.2 (antigo default, r_vox=3) mean_KS 0.4096 = PIOR | 12.0 (r_vox=4) 0.3632 | 20.0 (r_vox=6) 0.3458
# 12.0 escolhido: melhor cell_mean/cell_peak (0.757->0.647 / 0.772->0.620 = as maiores lacunas) e cobre a
# celula de 9.2um SEM invadir vizinhos (NN real ~6-10um). O 20um ganha KS mas capturaria vizinhos inteiros
# no template (artefato) -> ganho provavelmente espurio. Validar 20 vs 12 com proxy-training se importar.
CELL_DIAM_UM = float(os.environ.get("SYNTH_CELL_DIAM", "12.0"))
CELL_R_VOX = max(1, int(round((CELL_DIAM_UM / 2) / EFF_VOXEL[0])))
N_EST_PER_FRAME = int(os.environ.get("SYNTH_NEST", "240"))     # densidade real estimada (EDA)
N_SAMPLES = int(os.environ.get("SYNTH_NSAMPLES", "120"))        # volumes sinteticos (por versao)
SIM_GATE = float(os.environ.get("SYNTH_SIMGATE", "0.70"))       # cada celula >=70% similar a uma real (espaco 3d)
BLUR_SIGMA = float(os.environ.get("SYNTH_BLUR", "0.6"))         # desfoque final (PSF); ajustado pelo sweep
PSF_Z_RATIO = float(os.environ.get("SYNTH_PSFZ", "1.6"))        # PSF ANISOTROPICA: light-sheet borra ~1.6x mais em z que xy
DIV_RATE = float(os.environ.get("SYNTH_DIVRATE", "0.0"))        # taxa de celulas DIVIDINDO (medida no GT real; halteres/2 nucleos)
CELL_MEAN_TARGET = float(os.environ.get("SYNTH_CELLMEAN", "0.30"))  # ALVO: media de intensidade POR CELULA (distribuicao NORMAL)
CELL_MEAN_SIGMA  = float(os.environ.get("SYNTH_CELLSIG", "0.10"))   # desvio da normal (recalibrado do real em runtime)
SMOKE = os.environ.get("BIOHUB_SMOKE", "0") == "1"
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("synth_out")
DSDIR = OUT / "synth_dataset"; (DSDIR).mkdir(parents=True, exist_ok=True)
FIG = OUT / "figuras_synth"; FIG.mkdir(parents=True, exist_ok=True)

## Nucleo (extracao / liquido / geracao / similaridade)

In [ ]:
def pool_xy(f): return f[::DOWNSAMPLE[0], ::DOWNSAMPLE[1], ::DOWNSAMPLE[2]].astype(np.float32)

def dog_response(vol, scales=((1.5, 4.0), (2.2, 5.5))):
    dog = None
    for ss, sl in scales:
        r = gaussian_filter(vol, tuple(ss / EFF_VOXEL)) - gaussian_filter(vol, tuple(sl / EFF_VOXEL))
        dog = r if dog is None else np.maximum(dog, r)
    return dog

def dog_detect_precise(vol, rel_thr=0.35):
    dog = dog_response(vol)
    size = tuple(np.maximum(1, (2 * np.round(3.0 / EFF_VOXEL) + 1)).astype(int))
    mx = maximum_filter(dog, size=size, mode="nearest")
    return np.argwhere((dog == mx) & (dog >= dog.max() * rel_thr)), dog

SPHERICAL = os.environ.get("SYNTH_SPHERICAL", "1") != "0"   # captura esferica (v6) vs cubica

def _sphere_mask(shape, r):
    zz, yy, xx = np.indices(shape); c = np.array(shape) // 2
    d = np.sqrt(((zz-c[0])*EFF_VOXEL[0])**2 + ((yy-c[1])*EFF_VOXEL[1])**2 + ((xx-c[2])*EFF_VOXEL[2])**2)
    return (d <= (r + 0.5) * EFF_VOXEL[0]).astype(np.float32)

def extract_template(vol, zc, yc, xc, r=CELL_R_VOX):
    z0, z1, y0, y1, x0, x1 = zc - r, zc + r + 1, yc - r, yc + r + 1, xc - r, xc + r + 1
    if z0 < 0 or y0 < 0 or x0 < 0 or z1 > vol.shape[0] or y1 > vol.shape[1] or x1 > vol.shape[2]:
        return None
    t = vol[z0:z1, y0:y1, x0:x1].astype(np.float32).copy()
    if SPHERICAL:                                        # mascara ESFERICA -> so a celula, sem cantos
        t = t * _sphere_mask(t.shape, r)
    return t

def build_template_bank(frame_pooled, centroids_pooled, r=CELL_R_VOX, refine=True):
    tb = []
    for (zc, yc, xc) in np.round(np.atleast_2d(centroids_pooled)).astype(int):
        t = extract_template(frame_pooled, zc, yc, xc, r)
        if t is None: continue
        if refine: t = np.maximum(t - np.percentile(t, 20), 0.0)
        if t.max() > 0: tb.append(t)
    return tb

def liquid_model(frames_pooled, centroids_by_frame, cell_r=CELL_R_VOX):
    # v2: guarda os CAMPOS reais por frame (estrutura tecido/meio) -> banco p/ variar por amostra
    bg_vals, fields, field_cens = [], [], []
    for fr, cens in zip(frames_pooled, centroids_by_frame):
        mask = np.ones(fr.shape, bool)
        for (zc, yc, xc) in np.round(np.atleast_2d(cens)).astype(int):
            z0, z1 = max(0, zc - cell_r), min(fr.shape[0], zc + cell_r + 1)
            y0, y1 = max(0, yc - cell_r), min(fr.shape[1], yc + cell_r + 1)
            x0, x1 = max(0, xc - cell_r), min(fr.shape[2], xc + cell_r + 1)
            mask[z0:z1, y0:y1, x0:x1] = False
        bg_vals.append(fr[mask])
        fields.append(gaussian_filter(fr, float(os.environ.get("SYNTH_FIELD_SMOOTH", "4"))).astype(np.float32))  # suavizacao do campo (knob p/ grad_mag)
        # LAYOUT REAL: centroides DENSOS (GT rotulado uniao DoG) -> posicoes reais p/ transferencia de layout
        det = dog_detect_precise(fr)[0]
        allc = np.vstack([np.atleast_2d(cens), det]) if len(np.atleast_2d(cens)) and len(det) else (det if len(det) else np.atleast_2d(cens))
        field_cens.append(np.round(allc).astype(np.int32))
    bg = np.concatenate(bg_vals) if bg_vals else np.array([0.0])
    # PERFIL DE PROFUNDIDADE Z real (atenuacao de microscopia): brilho medio por slice Z
    zp = np.mean([f.mean(axis=(1, 2)) for f in frames_pooled], axis=0) if frames_pooled else np.ones(64)
    z_prof = gaussian_filter(zp.astype(np.float32), 2); z_prof = z_prof / (z_prof.mean() + 1e-6)  # normalizado, suave
    # NN + CONTAGEM + ONDE as celulas estao (nivel de campo/tecido nas deteccoes reais, NAO so no pico saturado)
    real_nn, dog_counts, gt_field = [], [], []
    _nsub = frames_pooled[:: max(1, len(frames_pooled) // 60)][:60]   # ate 60 frames representativos (base maior)
    for fr in _nsub:
        det = dog_detect_precise(fr)[0]; dog_counts.append(len(det))
        d = nn_um(det)
        if len(d): real_nn.append(d)
        fld = gaussian_filter(fr, 5); fnn = fld / (float(fld.max()) + 1e-6); sh = np.array(fr.shape) - 1
        for c in det: gt_field.append(float(fnn[tuple(np.clip(np.round(c).astype(int), 0, sh))]))
    real_nn = np.concatenate(real_nn) if real_nn else np.array([CELL_DIAM_UM * 0.55])
    gt_field = np.array(gt_field) if gt_field else np.array([0.6])
    gt_field_med = float(np.median(gt_field)); gt_field_std = float(np.std(gt_field) + 0.06)
    # DISTRIBUICAO EMPIRICA de LUZ onde as celulas REALMENTE estao -> P(celula | intensidade do campo).
    # Replica a MANEIRA como a luz se concentra no real (nao so a mediana) p/ alocar as celulas sinteticas.
    _lh, _le = np.histogram(gt_field, bins=24, range=(0.0, 1.0), density=True)
    light_bins = (0.5 * (_le[:-1] + _le[1:])).astype(np.float32)
    light_pdf = (gaussian_filter(_lh.astype(np.float32), 1.0)); light_pdf = (light_pdf / (light_pdf.max() + 1e-9)).astype(np.float32)
    # SEPARACAO MINIMA. Modo 'p5' (default): usa o 5o percentil do NN REAL = o quao perto duas celulas
    # de fato chegam. O modo antigo ('median') usava mediana*0.72, que PROIBIA os pares proximos que
    # existem no real -> o relatorio de qualidade mediu NN synth 12-15um contra real 8-9um.
    # DEFAULT = 'median'. A ablacao CONDENOU o modo 'p5': ele NAO melhora o NN (0.779 -> 0.768; quem
    # consertou o NN foi SYNTH_PLACEMENT=real, de 0.479 p/ 0.774) e e' a CAUSA ISOLADA do colapso de
    # textura (voxel_int 0.816 -> 0.463, local_std 0.698 -> 0.447, grad_mag 0.739 -> 0.535, media
    # 0.747 -> 0.706). Mecanismo: celulas mais proximas se FUNDEM sob a PSF -> massa conectada mais
    # brilhante e o DoG ainda detecta MENOS (mediana 74 -> 56). Ligar so' p/ experimento: SYNTH_MINSEP_MODE=p5.
    _nnp5 = float(np.percentile(real_nn, 5))
    if os.environ.get("SYNTH_MINSEP_MODE", "median") == "p5":
        min_sep_vox = float(np.clip(_nnp5 / EFF_VOXEL[0], 2.0, 6.0))
    else:
        min_sep_vox = float(np.clip(np.median(real_nn) * float(os.environ.get("SYNTH_MINSEP_FACTOR", "0.72")) / EFF_VOXEL[0], 3.0, 6.0))
    n_cells_real = int(np.median(dog_counts)) if dog_counts else N_EST_PER_FRAME     # densidade real, nao 240 fixo
    # DISTRIBUICAO EMPIRICA de contagem PAREADA com a extensao do tecido -> field_n_cells faz o
    # mapeamento por quantil (reproduz a variabilidade volume-a-volume, nao so' a mediana). A extensao
    # e' medida no campo SUAVIZADO, o mesmo objeto que `fields` guarda e que a geracao consome.
    _fs = float(os.environ.get("SYNTH_FIELD_SMOOTH", "4"))
    _sf = [gaussian_filter(f, _fs) for f in _nsub]
    _bf_thr = float(np.percentile(np.concatenate([s.ravel()[::7] for s in _sf]), 55)) if _sf else 0.0
    _cnt_bf = [field_extent(s, _bf_thr) for s in _sf]     # limiar GLOBAL -> a extensao varia de fato
    # REFERENCIA CANONICA p/ HM: POOL de frames por INDICE (sem vies de brilho) -> distribuicao = MEDIA real
    # exata (mesmos frames da validacao). Estavel e desacoplada dos knobs estruturais.
    hm_ref = None
    if frames_pooled:
        idx = np.unique(np.linspace(0, len(frames_pooled) - 1, min(16, len(frames_pooled))).astype(int))
        hm_ref = np.concatenate([np.asarray(frames_pooled[i], np.float32) for i in idx], axis=0)
    # GUARDA MEMORIA: cap do banco de campos/layouts (a base pode ter milhares de frames) - subamostra representativa
    CAP = int(os.environ.get("SYNTH_FIELDCAP", "500"))
    if len(fields) > CAP:
        ci = np.linspace(0, len(fields) - 1, CAP).astype(int)
        fields = [fields[i] for i in ci]; field_cens = [field_cens[i] for i in ci]
        frames_pooled = [frames_pooled[i] for i in ci]
    # TEXTURA RESIDUAL REAL (frame - campo suavizado) = alta-freq real p/ injetar no sintetico (ataca grad_mag)
    residuals = [(np.asarray(frames_pooled[i], np.float32) - fields[i]).astype(np.float32) for i in range(len(fields))]
    return dict(mean=float(bg.mean()), std=float(bg.std()), p50=float(np.percentile(bg, 50)), residuals=residuals,
                p99=float(np.percentile(bg, 99)), noise_std=float(np.std(bg) * 0.6),
                fields=fields, field_cens=field_cens,                  # LAYOUT REAL: centroides densos pareados c/ campos
                ref_frames=[np.asarray(f, np.float32) for f in frames_pooled],  # v4: p/ histogram matching
                count_dist=np.asarray(dog_counts, np.float32), bf_dist=np.asarray(_cnt_bf, np.float32), bf_thr=_bf_thr,
                hm_ref_canon=hm_ref, min_sep_vox=min_sep_vox, real_nn_p5_um=_nnp5,
                real_nn_med_um=float(np.median(real_nn)), n_cells_real=n_cells_real,
                gt_field_med=gt_field_med, gt_field_std=gt_field_std,   # ONDE as celulas estao (nivel de campo real)
                light_bins=light_bins, light_pdf=light_pdf,             # distribuicao EMPIRICA de luz->celula (real)
                z_prof=z_prof,                                          # PERFIL DE PROFUNDIDADE Z (atenuacao real)
                shape=frames_pooled[0].shape if frames_pooled else None)

def augment_template(t, rng):
    for ax in range(3):
        if rng.random() < 0.5: t = np.flip(t, ax)
    if rng.random() < 0.5: t = np.rot90(t, rng.integers(1, 4), axes=(1, 2))
    return np.ascontiguousarray(t * rng.uniform(0.7, 1.3))

def field_extent(field, thr=None):
    """Fracao do volume ocupada por tecido (proxy do estagio do desenvolvimento).

    BUG CORRIGIDO: a versao original usava `field > np.percentile(field, 55)`, que por DEFINICAO da'
    sempre ~0.45 -- o percentil e' do proprio campo. Ou seja, a "densidade time-aware" era uma
    CONSTANTE e todo volume sintetico recebia o mesmo alvo de contagem (o Q-Q de `count` saturava).
    Com um limiar GLOBAL (mesmo para todos os campos) a medida volta a discriminar tecido pouco vs
    muito extenso."""
    if thr is None: return float((field > np.percentile(field, 55)).mean())   # compat (constante ~0.45)
    return float((field > thr).mean())

def field_n_cells(field, base=N_EST_PER_FRAME, bg=None):
    """Densidade TIME-AWARE: n de celulas ~ extensao do tecido (estagio do desenvolvimento).

    MODO EMPIRICO (default, quando `bg` traz as distribuicoes): MAPEAMENTO POR QUANTIL -- acha o
    percentil da extensao deste campo entre as extensoes reais e devolve o MESMO percentil da
    distribuicao real de contagens. Isso reproduz de uma vez a MARGINAL (o real vai de ~25 a ~330
    celulas/volume) e a CORRELACAO com o tecido.
    Antes usava-se `int(median(dog_counts))` -- um unico numero -- ainda multiplicado por 0.60. O
    relatorio de qualidade mostrou o efeito: o Q-Q de `count` SATURAVA em ~100 enquanto o real chega a
    330, e o vizinho-mais-proximo saia deslocado (synth 12-15um vs real 8-9um) porque faltavam celulas.
    """
    # DEFAULT = 'empirical', decidido por ABLACAO no relatorio de qualidade (4 rodadas, held-out):
    #   A = config otimizada                      media 0.745 | count 0.500 | hi_freq 0.332 | voxel_int 0.769
    #   B = A + count empirical                   media 0.747 | count 0.542 | hi_freq 0.395 | voxel_int 0.816
    #   C = B + minsep p5                         media 0.706 | count 0.528 | hi_freq 0.344 | voxel_int 0.463
    # B melhora as DUAS piores distribuicoes (count +0.042, hi_freq +0.063) e sobe o pior eixo de
    # 0.332 p/ 0.384, com media neutra (+0.002, dentro do ruido de 1 seed). C mostra que o dano de
    # textura veio do MINSEP, nao daqui. Voltar ao antigo com SYNTH_COUNT_MODE=median.
    if bg is not None and os.environ.get("SYNTH_COUNT_MODE", "empirical") == "empirical":
        cd = bg.get("count_dist"); bd = bg.get("bf_dist"); thr = bg.get("bf_thr")
        if cd is not None and bd is not None and thr is not None and len(cd) >= 5 and len(bd) == len(cd):
            bf = field_extent(field, thr)
            q = float(np.searchsorted(np.sort(bd), bf) / max(len(bd), 1))          # percentil da extensao
            return int(np.clip(np.quantile(np.sort(cd), np.clip(q, 0.0, 1.0)), 22, 700))
    return int(np.clip(base * field_extent(field) / 0.30 * 0.60, 22, 460))         # modo antigo (constante ~0.45)

def connected_frac(vol, thr=0.4):
    """Fracao do volume brilhante que esta no MAIOR componente conectado.
    ~1 = massa conectada (como o tecido real); baixo = teia de blobs espalhados."""
    from scipy.ndimage import label
    m = vol > thr
    tot = int(m.sum())
    if tot < 5: return 0.0
    lab, n = label(m)
    if n == 0: return 0.0
    sizes = np.bincount(lab.ravel())[1:]
    return float(sizes.max() / tot)

def nn_um(coords):
    """Distancia ao vizinho mais proximo (um) de um conjunto de centroides."""
    if len(coords) < 2: return np.array([])
    P = np.asarray(coords, float) * EFF_VOXEL
    D = np.linalg.norm(P[:, None] - P[None], axis=2); np.fill_diagonal(D, np.inf)
    return D.min(1)

def clump_index(coords, shape, cell=8):
    """Indice de aglomeracao: variancia da densidade local / esperado Poisson (>1 = agregado)."""
    if len(coords) < 3: return 1.0
    g = np.zeros((shape[0]//cell+1, shape[1]//cell+1, shape[2]//cell+1))
    for z, y, x in np.asarray(coords, int):
        g[z//cell, y//cell, x//cell] += 1
    mu = g.mean()
    return float(g.var() / mu) if mu > 0 else 1.0

def overlap_audit(boxes, thr=0.3):
    """Fracao de caixas que se SOBREPOEM (IoU 3D > thr) com alguma vizinha. Deve ser ~0 (dados limpos)."""
    b = np.asarray(boxes, float)
    if len(b) < 2: return 0.0, 0.0
    vol = np.prod(np.clip(b[:, 3:] - b[:, :3], 0, None), 1)
    bad = 0; maxiou = 0.0
    for i in range(len(b)):
        lo = np.maximum(b[i, :3], b[:, :3]); hi = np.minimum(b[i, 3:], b[:, 3:])
        inter = np.prod(np.clip(hi - lo, 0, None), 1); inter[i] = 0
        iou = inter / (vol[i] + vol - inter + 1e-9)
        mi = float(iou.max()); maxiou = max(maxiou, mi)
        if mi > thr: bad += 1
    return bad / len(b), maxiou

def place_clustered(field, n_cells, rng, margin=6, gt_med=None, gt_std=None, light_bins=None, light_pdf=None):
    """Colocacao pela DISTRIBUICAO EMPIRICA de luz real: pondera cada voxel por P(celula | intensidade)
    medido nas deteccoes reais (a MANEIRA como a luz se concentra). Fallback = gaussiana em gt_med."""
    fn = field / (float(field.max()) + 1e-6)
    if light_pdf is not None and light_bins is not None:
        emp = np.interp(fn, light_bins, light_pdf).astype(np.float32)      # P(celula|intensidade) EMPIRICO real
        valid = (fn > np.percentile(fn, 20)).astype(np.float32)           # regiao de tecido (nao fundo)
        prob = (0.7 * emp + 0.3 * valid) * valid                          # DISPERSA: mistura empirico c/ uniforme (menos clump)
    elif gt_med is not None:
        prob = np.exp(-((fn - gt_med) / (gt_std)) ** 2)                    # pico no nivel de campo REAL
        prob = prob * (fn > np.percentile(fn, 30))                        # exclui o meio escuro (fundo)
    else:
        prob = np.clip(field - np.percentile(field, 45), 0, None) ** 1.1
    prob[:margin] = 0; prob[-margin:] = 0; prob[:, :margin] = 0; prob[:, -margin:] = 0
    prob[:, :, :margin] = 0; prob[:, :, -margin:] = 0
    s = float(prob.sum())
    if s <= 0: return np.zeros((0, 3), int)
    ps = prob.ravel() / s
    n_cand = min(int(n_cells * 3), int((ps > 0).sum()))
    idx = rng.choice(prob.size, size=n_cand, replace=False, p=ps)
    return np.array(np.unravel_index(idx, field.shape)).T

def field_zprof(field):
    """Perfil Z do CAMPO especifico usado neste volume (gradiente real DAQUELE frame, nao a media global)."""
    fz = gaussian_filter(field.mean(axis=(1, 2)).astype(np.float32), 2); return fz / (fz.mean() + 1e-6)

def apply_z_profile(vol, z_prof):
    """Impoe o PERFIL DE PROFUNDIDADE Z real (atenuacao). RENORMALIZA (nao corta) p/ nao saturar as
    slices profundas boostadas -> o clip destruia o gradiente (as slices boostadas passavam de 1 e eram cortadas)."""
    if z_prof is None or len(z_prof) != vol.shape[0]: return vol
    cur = vol.mean(axis=(1, 2)); cur = cur / (cur.mean() + 1e-6)
    gain = np.clip((z_prof / (cur + 1e-6)).astype(np.float32), 0.35, 3.0)
    vol = vol * gain[:, None, None]
    p = float(np.percentile(vol, 99.5))                                   # renormaliza -> preserva o gradiente
    return np.clip(vol / (p + 1e-6), 0, 1) if p > 0 else np.clip(vol, 0, 1)

# ALVO de intensidade por celula: DATA-DRIVEN. _CELL_MEANS_RT = amostra REAL (bootstrap casa a FORMA
# da distribuicao, incl. assimetria log-normal -> corrige cell_mean/cell_peak que o KS acusou diferir).
_CELL_MU_RT = CELL_MEAN_TARGET; _CELL_SIG_RT = CELL_MEAN_SIGMA; _CELL_MEANS_RT = None
_CELL_RATIO_RT = None   # distribuicao REAL de pico/media por celula (preenchida por measure_cell_stats)
_TEMPLATES_RT = None   # banco de templates de celulas REAIS (modo HIBRIDO: aparencia real + placement/global do param)

_PS_REF = None   # espectro de MAGNITUDE de referencia (real) p/ casamento de textura via Fourier
def match_power_spectrum(vol, bg, rng, w=None):
    """ETAPA NOVA (ataca grad_mag): casa o ESPECTRO DE POTENCIA do volume ao de um frame REAL no dominio
    de Fourier (preserva a FASE=estrutura do synth, adota a MAGNITUDE=textura do real). Blend por peso w."""
    w = float(os.environ.get("SYNTH_PSMATCH", "0.0")) if w is None else w
    if w <= 0: return vol
    refs = bg.get("ref_frames") or bg.get("residuals")
    if not refs: return vol
    ref = np.asarray(refs[int(rng.integers(len(refs)))], np.float32)
    if ref.shape != vol.shape: return vol
    Fv = np.fft.fftshift(np.fft.fftn(vol.astype(np.float32))); mag_v = np.abs(Fv); ph = np.angle(Fv)
    mag_r = np.abs(np.fft.fftshift(np.fft.fftn(ref - ref.mean())))
    # BAND-LIMITED: casa a magnitude SO em ALTA-FREQ (textura); preserva BAIXA-FREQ (estrutura/intensidade)
    shp = vol.shape; c = [(s - 1) / 2.0 for s in shp]; zz, yy, xx = np.indices(shp)
    r = np.sqrt(((zz-c[0])/(shp[0]/2))**2 + ((yy-c[1])/(shp[1]/2))**2 + ((xx-c[2])/(shp[2]/2))**2)
    hicut = float(os.environ.get("SYNTH_PSMATCH_HICUT", "0.35"))
    wr = (1.0 / (1.0 + np.exp(-(r - hicut) * 12))).astype(np.float32) * w   # sigmoide: peso cresce na alta-freq
    mag = mag_v * (1 - wr) + mag_r * wr                            # baixa-freq intacta, alta-freq casa o real (textura)
    out = np.real(np.fft.ifftn(np.fft.ifftshift(mag * np.exp(1j * ph)))).astype(np.float32)
    p = np.percentile(out, 99.5); out = out / (p + 1e-6) if p > 0 else out
    out = np.clip(out, 0, 1)
    if os.environ.get("SYNTH_PSMATCH_HM", "1") == "1":            # HM pos-PSMATCH: restaura o HISTOGRAMA de intensidade
        try:                                                       # (monotonico -> preserva a textura) -> corrige voxel_int
            from skimage.exposure import match_histograms
            rf = np.clip(ref / (np.percentile(ref, 99.5) + 1e-6), 0, 1)
            out = np.clip(match_histograms(out, rf).astype(np.float32), 0, 1)
        except Exception: pass
    return out

def poisson_gaussian_noise(vol, rng):
    """RUIDO DE CAMERA (sCMOS, EMVA1288, da literatura): shot (Poisson, dependente do sinal) + leitura (Gaussiano).
    variancia = sinal/peak_pe + read^2 -> quebra a lisura das celulas, casa o GRAO real. Aplicado no final."""
    pe = float(os.environ.get("SYNTH_POISSON_PE", "220"))            # fotoeletrons no pico (maior=menos ruido)
    rd = float(os.environ.get("SYNTH_READNOISE", "0.004"))
    if pe <= 0: return vol
    lam = np.clip(vol, 0, 1) * pe
    noisy = rng.poisson(lam).astype(np.float32) / pe + rng.normal(0, rd, vol.shape).astype(np.float32)
    return np.clip(noisy, 0, 1)

def retarget_cell_intensity(vol, boxes, rng, mu=None, sigma=None, targets=None):
    """Casa a MEDIA de intensidade POR CELULA a uma NORMAL(mu, sigma) medida no REAL. Peso ESFERICO
    suave (=0 na borda do box -> NAO deixa quadrado escuro no tecido) + clamp de escala (nao cria buraco).

    `targets` (opcional, 1 alvo por box) FIXA o alvo em vez de sortear. Necessario no gerador TEMPORAL:
    sem isso a mesma celula rastreada sorteia um brilho novo a cada frame e PISCA, destruindo a
    identidade de aparencia que o preditor de arestas precisa aprender."""
    if boxes is None or len(boxes) == 0: return vol
    mu = _CELL_MU_RT if mu is None else mu
    sigma = _CELL_SIG_RT if sigma is None else sigma
    for _bi, b in enumerate(np.atleast_2d(boxes).astype(int)):
        z0, y0, x0, z1, y1, x1 = b[:6]
        z0, y0, x0 = max(z0, 0), max(y0, 0), max(x0, 0)
        reg = vol[z0:z1, y0:y1, x0:x1]
        if reg.size < 8: continue
        pk = float(reg.max())
        if pk <= 1e-4: continue
        active = reg > 0.3 * pk                                   # regiao ativa (mesma def de measure_cell_stats/validate)
        cur = float(reg[active].mean()) if active.any() else float(reg.mean())
        if cur <= 1e-4: continue
        if targets is not None and _bi < len(targets):
            tgt = float(np.clip(float(targets[_bi]) * rng.uniform(0.98, 1.02), 0.05, 0.95))  # alvo FIXO da trilha (+-2% de cintilacao real)
        elif _CELL_MEANS_RT is not None and len(_CELL_MEANS_RT) > 5:
            tgt = float(np.clip(rng.choice(_CELL_MEANS_RT) * rng.uniform(0.95, 1.05), 0.05, 0.95))  # BOOTSTRAP do real (casa a forma)
        else:
            tgt = float(np.clip(rng.normal(mu, sigma), 0.05, 0.95))
        # CASAMENTO DO PERFIL (pico/media): um ganho escalar move media e pico juntos e NUNCA corrige a
        # RAZAO entre eles. Um expoente sobre o perfil normalizado muda a razao (>1 afina o pico, <1
        # achata) e depois o ganho recoloca a media no alvo. Ataca cell_peak, cell_range e contrast.
        if os.environ.get("SYNTH_MATCH_PEAK", "0") == "1" and _CELL_RATIO_RT is not None and len(_CELL_RATIO_RT) > 5:
            tgt_ratio = float(rng.choice(_CELL_RATIO_RT))
            base = np.clip(reg / (pk + 1e-6), 0, 1)
            best_g, best_err = 1.0, 1e9
            for g in (0.6, 0.8, 1.0, 1.3, 1.7, 2.2):
                cand = base ** g
                cm = float(cand[active].mean()) if active.any() else float(cand.mean())
                if cm <= 1e-6: continue
                err = abs(float(cand.max()) / cm - tgt_ratio)
                if err < best_err: best_g, best_err = g, err
            if abs(best_g - 1.0) > 1e-6:
                reg = (base ** best_g) * pk
                pk = float(reg.max())
                cur = float(reg[active].mean()) if active.any() else float(reg.mean())
                if cur <= 1e-4: continue
                vol[z0:z1, y0:y1, x0:x1] = reg
        scale = float(np.clip(tgt / cur, 0.6, 1.35))             # CLAMP (iter5: teto menor -> celulas menos 'poppy')
        # peso ESFERICO gaussiano centrado no box -> 0 na borda (tecido intacto, sem quadrado)
        sh = reg.shape; cc = (np.array(sh) - 1) / 2.0
        zz, yy, xx = np.indices(sh)
        d2 = ((zz-cc[0])/(sh[0]/2.0+1e-6))**2 + ((yy-cc[1])/(sh[1]/2.0+1e-6))**2 + ((xx-cc[2])/(sh[2]/2.0+1e-6))**2
        w = np.exp(-2.2 * d2).astype(np.float32)                 # ~1 no centro, ~0 na borda
        vol[z0:z1, y0:y1, x0:x1] = np.clip(reg * (1 + w * (scale - 1)), 0, 1)
    return vol

def gen_volume(shape, bg, templates, n_cells, rng, min_sep=2):
    """v2: campo REAL como base + colocacao das celulas por DENSIDADE (brilho do tecido)
    + brilho da celula herdado do tecido local -> recupera estrutura espacial e intensidade."""
    Z, Y, X = shape
    fields = bg.get("fields")
    if fields:
        field = fields[rng.integers(len(fields))].copy()          # 1 campo real por amostra (varia)
    else:
        field = np.full(shape, bg["p50"], np.float32)
    # densidade REAL x mult (nn_um). SYNTH_NEST so' vale se setado EXPLICITAMENTE: sem isso o
    # `bg['n_cells_real']` medido sobrescreve o default e o knob fica MORTO (o sweep biohub-newvars
    # gastou 4 configs de SYNTH_NEST que deram mean_KS identico ate a 16a casa decimal).
    _base = N_EST_PER_FRAME if os.environ.get("SYNTH_NEST") else bg.get('n_cells_real', N_EST_PER_FRAME)
    n_cells = int(field_n_cells(field, _base, bg) * float(os.environ.get("SYNTH_COUNT_MULT", "1.0")))
    # ruido ESCALADO pelo campo -> meio (escuro) fica escuro como o real (preto), tecido tem textura
    fn = field / max(float(field.max()), 1e-6)
    _res = bg.get("residuals")
    if os.environ.get("SYNTH_REAL_RESIDUAL", "0") == "1" and _res:
        vol = field + _res[int(rng.integers(len(_res)))] * float(os.environ.get("SYNTH_RESIDUAL_W", "1.0"))  # TEXTURA REAL (grad_mag)
    else:
        _nz = rng.normal(0, bg.get("noise_std", bg["std"]), shape).astype(np.float32)
        _ncorr = float(os.environ.get("SYNTH_NOISE_CORR", "0.0"))    # knob p/ correlacionar ruido; default OFF: sweep provou que NAO move o hi_freq
        if _ncorr > 0: _nz = gaussian_filter(_nz, _ncorr)            # causa real do overshoot = low-pass do XY-pooling do REAL que o synth 64^3 nativo nao replica
        vol = field + _nz * (0.15 + 0.85 * fn)
    lo, hi = np.percentile(field, 50), np.percentile(field, 99)     # MATRIZ DE TECIDO -> massa conectada
    vol = vol + 0.38 * np.clip((field - lo) / (hi - lo + 1e-6), 0, 1)
    r = templates[0].shape[0] // 2
    min_sep = bg.get("min_sep_vox", min_sep)                       # piso = NN minima REAL (evita sobreposicao)
    # colocacao AGREGADA (clusters como o tecido real) em vez de Poisson disperso
    coords = place_clustered(field, n_cells, rng, margin=r, gt_med=bg.get('gt_field_med'), gt_std=bg.get('gt_field_std'), light_bins=bg.get('light_bins'), light_pdf=bg.get('light_pdf'))
    centers, boxes = [], []
    for (zc, yc, xc) in coords:
        if len(centers) >= n_cells: break
        msj = min_sep * float(rng.uniform(float(os.environ.get("SYNTH_MINSEP_JLO", "1.0")), float(os.environ.get("SYNTH_MINSEP_JHI", "1.0"))))  # JITTER (iter7): broadena NN
        if centers and np.linalg.norm((np.array(centers) - [zc, yc, xc]) * EFF_VOXEL, axis=1).min() < msj * EFF_VOXEL[0]:
            continue                                               # rejeita centro sobreposto -> caixas distintas
        t = augment_template(templates[rng.integers(len(templates))], rng)
        t = t * (0.4 + 1.0 * field[zc, yc, xc])                    # brilho herdado (menos dominante que o tecido)
        rz, ry, rx = np.array(t.shape) // 2
        vol[zc-rz:zc+rz+1, yc-ry:yc+ry+1, xc-rx:xc+rx+1] = np.maximum(vol[zc-rz:zc+rz+1, yc-ry:yc+ry+1, xc-rx:xc+rx+1], t)
        centers.append([zc, yc, xc]); boxes.append([zc-rz, yc-ry, xc-rx, zc+rz+1, yc+ry+1, xc+rx+1])
    vol = np.clip(vol, 0, None)
    if BLUR_SIGMA > 0: vol = gaussian_filter(vol, (BLUR_SIGMA*PSF_Z_RATIO, BLUR_SIGMA, BLUR_SIGMA))  # PSF ANISOTROPICA final (z>xy)
    m = np.percentile(vol, 99)                                     # nivel do TECIDO -> nucleos SATURAM em 1.0 (pico real)
    vol = np.clip(vol / m, 0, 1) if m > 0 else vol
    # v4: HISTOGRAM MATCHING -> casa a distribuicao de intensidade a de um frame REAL
    ref = bg.get("hm_ref_canon")                                   # referencia CANONICA (brilho mediano) -> estavel
    if ref is not None:
        try:
            from skimage.exposure import match_histograms
            hm = match_histograms(vol, ref).astype(np.float32)
            vol = np.clip(0.05 * vol + 0.95 * hm, 0, 1)   # HM forte a ref fixa -> intensidade ~ real, sem acoplar aos knobs
        except Exception:
            pass
    vol = apply_z_profile(vol, field_zprof(field))               # PERFIL Z do CAMPO especifico (nao a media global -> sem corcova)
    vol = retarget_cell_intensity(vol, boxes, rng)               # media por celula -> Normal data-driven (casa real)
    vol = poisson_gaussian_noise(vol, rng)                       # ruido de camera realista (shot+leitura) -> grao real
    vol = match_power_spectrum(vol, bg, rng)                     # ETAPA NOVA: textura real via Fourier (ataca grad_mag)
    return np.clip(vol, 0, 1).astype(np.float32), np.array(boxes, np.int32), np.array(centers, np.float32)

def rand_rot(rng):
    a, b, c = rng.uniform(0, 2*np.pi, 3)
    ca, sa, cb, sb, cc, sc = np.cos(a), np.sin(a), np.cos(b), np.sin(b), np.cos(c), np.sin(c)
    Rz = np.array([[ca, -sa, 0], [sa, ca, 0], [0, 0, 1]])
    Ry = np.array([[cb, 0, sb], [0, 1, 0], [-sb, 0, cb]])
    Rx = np.array([[1, 0, 0], [0, cc, -sc], [0, sc, cc]])
    return (Rz @ Ry @ Rx).astype(np.float32)

_E3D = np.random.default_rng(1234)
_E3D_FILT = _E3D.standard_normal((16, 3, 3, 3)).astype(np.float32)   # banco fixo de filtros 3d

def embed3d(patch, size=8):
    """Embedding de espaco vetorial 3D: normaliza tamanho+brilho -> banco de filtros 3d aleatorios
    (captura FORMA/ESTRUTURA 3d) -> vetor unitario. Produto interno = similaridade de forma [-1,1]."""
    p = np.asarray(patch, np.float32)
    if p.shape != (size, size, size):
        p = ndzoom(p, [size / s for s in p.shape], order=1)
    m = float(p.max())
    if m > 0: p = p / m
    p = p - p.mean()
    feats = []
    for f in _E3D_FILT:
        r = ndconvolve(p, f, mode="constant")
        feats += [float(r.mean()), float(r.std()), float(np.abs(r).max())]
    v = np.array(feats, np.float32); return v / (np.linalg.norm(v) + 1e-9)

def cell_sim(patch, real_embs):
    """Similaridade (cosseno, 0..1) da celula a celula real MAIS parecida no espaco 3d."""
    if real_embs is None or len(real_embs) == 0: return 1.0
    return float(np.max(real_embs @ embed3d(patch)))

def radius_by_gradient(t, nbins=16):
    """RAIO por BORDA (declive radial mais ingreme) -> robusto a vizinhos e ao tamanho da janela.
    O criterio antigo (volume>meia-altura DENTRO da janela) e enviesado: janela pequena clipa a celula,
    janela grande engloba vizinhos (medido em biohub-radius-profile: r_half/r_bg batem no teto em tecido
    denso; r_grad nao). Retorna raio em VOXELS (efetivo isotropico)."""
    c = np.array(t.shape) // 2
    zz, yy, xx = np.indices(t.shape)
    d = np.sqrt(((zz-c[0])*EFF_VOXEL[0])**2 + ((yy-c[1])*EFF_VOXEL[1])**2 + ((xx-c[2])*EFF_VOXEL[2])**2)
    rmax = float(d.max()); edges = np.linspace(0, rmax, nbins+1); ctr = 0.5*(edges[:-1]+edges[1:])
    prof = np.full(nbins, np.nan, np.float32)
    for i in range(nbins):
        m = (d >= edges[i]) & (d < edges[i+1])
        if m.sum() >= 3: prof[i] = float(t[m].mean())
    ok = ~np.isnan(prof)
    if ok.sum() < 4: return None
    ctr, prof = ctr[ok], prof[ok]
    r_um = float(ctr[int(np.argmin(np.gradient(prof)))])          # borda = declive mais ingreme
    return max(0.8, r_um / EFF_VOXEL[0])                          # um -> voxels efetivos

def measure_cell_stats(templates):
    """Mede a DISTRIBUICAO das celulas reais: tamanho, pico, brilho medio, EMBEDDING 3d, elongacao.
    SYNTH_RADIUS_MODE=halfmax (DEFAULT): volume>meia-altura na janela. Enviesado pela janela, MAS validado.
    SYNTH_RADIUS_MODE=grad: raio por borda radial. Robusto a vizinhos no VOLUME COMPLETO (biohub-radius-profile),
      porem VALIDADO LOCALMENTE COMO ENVIESADO no template pequeno: subestima ~35-50% (o declive mais ingreme de
      uma gaussiana esta em r/sqrt(2), nao na borda) e satura com binning grosseiro (raio 2.0 e 3.0 -> ambos 1.30).
      NAO usar como default sem calibrar. Mantido como opt-in p/ experimentacao."""
    RMODE = os.environ.get("SYNTH_RADIUS_MODE", "halfmax")
    radii, peaks, means, embs, aspect = [], [], [], [], []
    for t in templates:
        pk = float(t.max())
        if pk <= 0: continue
        r_g = radius_by_gradient(t) if RMODE == "grad" else None
        if r_g is None:
            mass = float((t > pk * 0.5).sum())                  # fallback/antigo: volume acima da meia-altura
            r_g = (3 * mass / (4 * np.pi)) ** (1 / 3)
        radii.append(r_g); peaks.append(pk)
        means.append(float(t[t > pk * 0.3].mean()))             # brilho MEDIO da celula (regiao ativa)
        embs.append(embed3d(t))
    radii = np.array(radii) if radii else np.array([CELL_R_VOX], float)
    # RAZAO PICO/MEDIA por celula REAL. O `retarget_cell_intensity` so' casava a MEDIA -- nada no
    # pipeline mirava o PICO, e por isso cell_peak/cell_range/contrast eram os piores eixos de
    # aparencia (0.605/0.605/0.702 no placar). Guardar aqui (unico ponto que ve peaks E means juntos)
    # evita ter que propagar a distribuicao pelos 5 sites que setam _CELL_MEANS_RT.
    global _CELL_RATIO_RT
    if peaks and means:
        _r = np.asarray(peaks, float) / np.maximum(np.asarray(means, float), 1e-6)
        _CELL_RATIO_RT = _r[np.isfinite(_r) & (_r > 1.0) & (_r < 8.0)]
    return dict(radii=radii, r_med=float(np.median(radii)),
                peaks=np.array(peaks) if peaks else np.array([0.5], float),
                means=np.array(means) if means else np.array([0.3], float),
                embs=np.array(embs) if embs else None)

def study_cell_development(stats):
    """ESTUDO no REAL: distribuicao de TAMANHO + como celulas JOVENS (pequenas) diferem das VELHAS."""
    r, pk, mn = stats["radii"], stats["peaks"], stats["means"]
    if len(r) < 4: return
    med = float(np.median(r)); yo = r < med; ol = r >= med
    print("=== ESTUDO DE DESENVOLVIMENTO CELULAR (real) ===")
    print(f"  raio(vox): min {r.min():.2f} | mediana {med:.2f} | max {r.max():.2f}")
    print(f"  JOVENS (r<med): n={int(yo.sum())} brilho_med {mn[yo].mean():.3f} pico {pk[yo].mean():.3f}")
    print(f"  VELHAS (r>=med): n={int(ol.sum())} brilho_med {mn[ol].mean():.3f} pico {pk[ol].mean():.3f}")
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].hist(r, bins=25, color="purple", alpha=.8); ax[0].axvline(med, ls="--", c="k", label=f"mediana {med:.1f}")
    ax[0].set_title("Distribuicao de TAMANHO (raio vox)"); ax[0].set_xlabel("raio"); ax[0].legend()
    ax[1].scatter(r, mn, s=10, alpha=.5); ax[1].set_title("Tamanho vs brilho medio"); ax[1].set_xlabel("raio"); ax[1].set_ylabel("brilho")
    ax[2].scatter(r, pk, s=10, alpha=.5, c="orange"); ax[2].set_title("Tamanho vs pico de intensidade"); ax[2].set_xlabel("raio")
    plt.suptitle("Desenvolvimento celular: JOVENS (pequenas) vs VELHAS (grandes) - REAL", weight="bold")
    plt.savefig(FIG / "cell_development.png", dpi=180, bbox_inches="tight"); plt.show()

def fbm_texture_3d(shape, rng, H=None, octaves=None, lacunarity=2.0, base=2):
    """Ruido fractal-Browniano 3D (fBM): textura de cromatina FASE-CORRETA e estruturada.
    Somatorio de oitavas de value-noise; amplitude ~ gain^o (gain=2^-H), frequencia ~ lacunarity^o.
    H=0.7 casa a estatistica de cromatina real (P(f) ~ f^-3.4). Aplicado ANTES da PSF -> o blur
    optico borra a textura como sinal real. Este e o elo que o PSMATCH (so magnitude) nao capturava:
    o casamento de espectro corrige a magnitude mas mantem a FASE sintetica; o fBM injeta estrutura
    de alta-freq fase-correlacionada (o que travava o hi_freq em KS~0.5)."""
    H = float(os.environ.get("SYNTH_FBM_H", "0.85")) if H is None else H  # 0.85 tunado no biohub-gs-fbm (0.7 overshoot: hi_freq 0.96)
    octaves = int(os.environ.get("SYNTH_FBM_OCT", "6")) if octaves is None else octaves
    gain = 2.0 ** (-H)                                                # 0.62 p/ H=0.7
    field = np.zeros(shape, np.float32); amp = 1.0; tot = 0.0
    for o in range(octaves):
        res = tuple(min(int(base * (lacunarity ** o)), d) for d in shape)
        res = tuple(max(2, r) for r in res)
        small = rng.standard_normal(res).astype(np.float32)
        oct_n = small if small.shape == tuple(shape) else \
            ndzoom(small, [shape[i] / small.shape[i] for i in range(3)], order=1)
        field += amp * oct_n; tot += amp; amp *= gain
    field /= (tot + 1e-9)
    rng_ = float(field.max() - field.min()) + 1e-6
    return ((field - field.min()) / rng_).astype(np.float32)          # [0,1] fBM normalizado


def cell_texture(shape, rng, strength=None):
    """TEXTURA INTERNA de cromatina. Agora fBM multi-oitava (H=0.7, fase-correta) em vez de 1 escala.
    Multiplicador ~[1-s, 1+s]. Aplicado no interior do nucleo ANTES do blur/PSF."""
    s = float(os.environ.get("SYNTH_TEXTURE", "0.30")) if strength is None else strength
    if s <= 0: return np.ones(shape, np.float32)
    if os.environ.get("SYNTH_FBM", "1") == "1":
        tex = fbm_texture_3d(shape, rng)                             # [0,1] fase-correta multi-escala
    else:
        small = rng.standard_normal(tuple(max(2, d // 2) for d in shape)).astype(np.float32)
        tex = ndzoom(small, [shape[i] / small.shape[i] for i in range(3)], order=1)
        tex = gaussian_filter(tex, 0.7); rng_ = float(tex.max() - tex.min()) + 1e-6
        tex = (tex - tex.min()) / rng_
    return (1 + s * (tex - 0.5) * 2).astype(np.float32)              # granularidade de cromatina

def gen_param_cell(rng, stats, stage=None, shape_kind=None):
    """Celula PARAMETRICA: amostra (tamanho, brilho) COERENTES de UMA celula real (preserva a
    relacao tamanho-brilho medida). Estagio (jovem/velha), variacao ELIPTICA e TRANSICAO suave."""
    i = rng.integers(len(stats["radii"]))                          # UMA celula real -> size+brilho coerentes
    r = float(stats["radii"][i]); inten = float(stats["means"][i]) * max(0.2, rng.normal(1.3, 0.08))  # brilho REAL (~0.65x menor, cell-mean casa 0.29) + normal estreito
    rmed = float(stats.get("r_med", r))
    # ESTAGIO explicito (galeria) OU implicito no tamanho amostrado (jovens pequenas / velhas grandes)
    if stage == "early": r *= rng.uniform(0.55, 0.75); inten *= rng.uniform(0.7, 0.9)
    elif stage == "dev": r *= rng.uniform(1.0, 1.2)
    young = r < rmed
    # FORMA ELIPTICA: nucleos reais raramente sao esferas perfeitas. TODOS tem anisotropia leve;
    # uma fracao (velhas/em divisao) sao claramente alongados. Eixos = 3 semi-eixos independentes.
    p_ellip = 0.30 if young else 0.60                                 # a maioria e' eliptica em algum grau
    aniso = rng.uniform(0.82, 1.22, 3)                               # anisotropia base (todo nucleo)
    if shape_kind == "ellip" or (shape_kind is None and rng.random() < p_ellip):
        aniso[rng.integers(3)] *= rng.uniform(1.5, 2.2)             # 1 eixo claramente alongado (eliptico marcado)
    axes = np.maximum(r * aniso, 0.6)
    size = int(np.ceil(axes.max()) * 2) + 5                           # +5: margem p/ borda
    zz, yy, xx = np.indices((size, size, size)); c = size // 2
    P = np.stack([zz - c, yy - c, xx - c], -1).astype(np.float32) @ rand_rot(rng).T
    r_norm = np.sqrt((P[..., 0] / axes[0]) ** 2 + (P[..., 1] / axes[1]) ** 2 + (P[..., 2] / axes[2]) ** 2)  # =1 na superficie
    # PERFIL = SUPER-GAUSSIANA (disco PREENCHIDO, borda NITIDA), NAO gaussiana. A sonda mediu o nucleo
    # real: intensidade ~0.63 do centro a meia-distancia (topo chapado) e cai a 0 na borda -- uma
    # gaussiana ja estaria em 0.40. exp(-(r_norm)^n) com n alto = plato no miolo + queda abrupta na borda.
    n = float(os.environ.get("SYNTH_CELL_SHARPNESS", "7"))
    cell = (np.exp(-(r_norm ** n)) * inten).astype(np.float32)
    cell = cell * cell_texture(cell.shape, rng)                      # TEXTURA de cromatina (nao esfera lisa)
    _bl = float(os.environ.get("SYNTH_CELL_EDGEBLUR", "0.25"))       # borda quase nitida (nao gaussiana larga)
    return gaussian_filter(cell, _bl) if _bl > 0 else cell

def add_divisions(vol, boxes, centers, field, stats, rng):
    """GERACAO DE CELULAS (divisao): fracao DIV_RATE das celulas gera um 2o nucleo proximo (haltere)
    = celulas nascendo, na TAXA real medida no lineage GT. Modela a morfologia mitotica rara."""
    dr = float(os.environ.get("SYNTH_DIVRATE", str(DIV_RATE)))     # taxa de divisao TUNAVEL (otimizador)
    if dr <= 0 or not centers: return vol, boxes, centers
    Z, Y, X = vol.shape; R = CELL_R_VOX
    _slo = float(os.environ.get("SYNTH_SISTER_LO", "1.2")); _shi = float(os.environ.get("SYNTH_SISTER_HI", "1.8"))  # dist irmas
    for (zc, yc, xc) in list(centers):
        if rng.random() >= dr: continue
        off = rng.normal(0, 1, 3); off = off / (np.linalg.norm(off) + 1e-6) * rng.uniform(_slo, _shi) * R
        dz, dy, dx = int(zc + off[0]), int(yc + off[1]), int(xc + off[2])
        # CELULA EMERGENTE (recem-dividida) = JOVEM: menor, mais redonda, brilho de celula nova (estagio early)
        cell = gen_param_cell(rng, stats, stage="early")
        cell = cell * (0.96 + 0.05 * field[np.clip(dz,0,Z-1), np.clip(dy,0,Y-1), np.clip(dx,0,X-1)])
        rz, ry, rx = np.array(cell.shape) // 2
        if dz-rz < 0 or dy-ry < 0 or dx-rx < 0 or dz+rz+1 > Z or dy+ry+1 > Y or dx+rx+1 > X: continue
        vol[dz-rz:dz+rz+1, dy-ry:dy+ry+1, dx-rx:dx+rx+1] = np.maximum(vol[dz-rz:dz+rz+1, dy-ry:dy+ry+1, dx-rx:dx+rx+1], cell)
        # tambem ENCOLHE levemente a celula-mae (acabou de dividir) -> par mae-filha realista
        centers.append([dz, dy, dx]); boxes.append([dz-R, dy-R, dx-R, dz+R+1, dy+R+1, dx+R+1])
    return vol, boxes, centers

def gen_volume_param(shape, bg, stats, n_cells, rng, min_sep=2):
    """VERSAO D: celulas 100% parametricas (sem colar templates). Fundo real + colocacao por densidade."""
    Z, Y, X = shape
    fields = bg.get("fields"); _fi = int(rng.integers(len(fields))) if fields else 0
    field = fields[_fi].copy() if fields else np.full(shape, bg["p50"], np.float32)
    # densidade REAL x mult (nn_um). SYNTH_NEST so' vale se setado EXPLICITAMENTE: sem isso o
    # `bg['n_cells_real']` medido sobrescreve o default e o knob fica MORTO (o sweep biohub-newvars
    # gastou 4 configs de SYNTH_NEST que deram mean_KS identico ate a 16a casa decimal).
    _base = N_EST_PER_FRAME if os.environ.get("SYNTH_NEST") else bg.get('n_cells_real', N_EST_PER_FRAME)
    n_cells = int(field_n_cells(field, _base, bg) * float(os.environ.get("SYNTH_COUNT_MULT", "1.0")))
    fn = field / max(float(field.max()), 1e-6)
    _res = bg.get("residuals")
    # MODELO FISICO CORRETO (SYNTH_DARK_MEDIUM=1): a base e' o LIQUIDO ESCURO onde o embriao esta
    # imerso -- NAO o frame real suavizado (que ja contem o brilho medio das celulas -> nuvem clara).
    # As CELULAS sao as fontes de luz; o brilho do ambiente EMERGE da difusao da PSF delas, nao de uma
    # luz de fundo pintada. Sem isso o fundo fica cor-de-celula e a imagem borra (defeito visto no
    # showcase: real = bolinhas nitidas no escuro, synth = borrao). O `field` ainda decide ONDE as
    # celulas vao (placement), mas NAO e' mais o fundo visual.
    DARK = os.environ.get("SYNTH_DARK_MEDIUM", "0") == "1"
    if DARK:
        med = float(os.environ.get("SYNTH_MEDIUM_LEVEL", "0.04"))    # autofluorescencia do liquido (baixa)
        _nz = rng.normal(0, bg.get("noise_std", bg["std"]), shape).astype(np.float32)
        vol = np.clip(med + _nz * 0.5, 0, None).astype(np.float32)   # liquido escuro + ruido; SEM tecido
        tissue = np.zeros(shape, np.float32)
    elif os.environ.get("SYNTH_REAL_RESIDUAL", "0") == "1" and _res:
        vol = field + _res[int(rng.integers(len(_res)))] * float(os.environ.get("SYNTH_RESIDUAL_W", "1.0"))  # TEXTURA REAL (grad_mag)
    else:
        _nz = rng.normal(0, bg.get("noise_std", bg["std"]), shape).astype(np.float32)
        _ncorr = float(os.environ.get("SYNTH_NOISE_CORR", "0.0"))    # knob p/ correlacionar ruido; default OFF: sweep provou que NAO move o hi_freq
        if _ncorr > 0: _nz = gaussian_filter(_nz, _ncorr)            # causa real do overshoot = low-pass do XY-pooling do REAL que o synth 64^3 nativo nao replica
        vol = field + _nz * (0.15 + 0.85 * fn)
    # MATRIZ DE TECIDO brilhante: preenche o espaco entre celulas -> MASSA CONECTADA (nao buracos escuros)
    if not DARK:
        lo, hi = np.percentile(field, 50), np.percentile(field, 99)
        tissue = np.clip((field - lo) / (hi - lo + 1e-6), 0, 1)
    vol = vol + float(os.environ.get("SYNTH_TISSUE", "0.55")) * tissue  # TECIDO mais brilhante -> celulas embutem (menos contraste)
    # BLUR DESACOPLADO: o pipeline aplicava UM filtro global no fim, borrando fundo e celulas juntos.
    # Isso criava uma TENSAO artificial medida no relatorio: subir o blur consertava `hi_freq` (a textura
    # do fundo domina a alta-frequencia) mas afundava `count` (celulas borradas se fundem e o DoG
    # detecta menos). Com SYNTH_DECOUPLE_BLUR=1 as celulas vao para uma camada propria: o FUNDO leva
    # SYNTH_BG_BLUR e as CELULAS levam SYNTH_BLUR (a PSF optica de verdade), cada um no seu valor.
    _dec = os.environ.get("SYNTH_DECOUPLE_BLUR", "0") == "1"
    cell_layer = np.zeros_like(vol) if _dec else None
    dst = cell_layer if _dec else vol                              # onde as celulas sao compostas
    min_sep = bg.get("min_sep_vox", min_sep)                       # piso = NN minima REAL (evita sobreposicao)
    fcs = bg.get("field_cens")
    if os.environ.get("SYNTH_PLACEMENT", "field") == "real" and fcs and _fi < len(fcs) and len(fcs[_fi]) >= 3:
        # PLACEMENT REAL: usa as POSICOES DETECTADAS reais deste campo -> distribuicao NN casa o real por construcao
        rc = dedup_cens(np.atleast_2d(fcs[_fi]).astype(np.float32), min_sep)
        hi = np.array(shape, np.float32) - CELL_R_VOX - 1
        rc = np.clip(rc, CELL_R_VOX, hi)                              # dentro dos limites (evita IndexError na borda)
        m = int(rng.integers(max(1, int(0.85*len(rc))), len(rc)+1))   # subamostra leve (variacao)
        coords = rc[rng.choice(len(rc), size=min(m, len(rc)), replace=False)].astype(int)
    else:
        coords = place_clustered(field, n_cells, rng, margin=6, gt_med=bg.get('gt_field_med'), gt_std=bg.get('gt_field_std'), light_bins=bg.get('light_bins'), light_pdf=bg.get('light_pdf'))  # onde as celulas REAIS estao
    centers, boxes, centers_sim = [], [], []
    for (zc, yc, xc) in coords:
        if len(centers) >= n_cells: break
        msj = min_sep * float(rng.uniform(float(os.environ.get("SYNTH_MINSEP_JLO", "1.0")), float(os.environ.get("SYNTH_MINSEP_JHI", "1.0"))))  # JITTER (iter7): broadena NN
        if centers and np.linalg.norm((np.array(centers) - [zc, yc, xc]) * EFF_VOXEL, axis=1).min() < msj * EFF_VOXEL[0]:
            continue                                               # rejeita centro sobreposto
        if os.environ.get("SYNTH_CELL_SOURCE", "param") == "template" and _TEMPLATES_RT:
            # HIBRIDO: celula = TEMPLATE REAL aumentado (aparencia/tamanho/textura REAIS) no placement do param
            best = augment_template(_TEMPLATES_RT[rng.integers(len(_TEMPLATES_RT))], rng); centers_sim.append(1.0)
        else:
            # GATE >=70%: reamostra ate a celula ser >=70% similar a uma celula REAL (espaco 3d)
            cell = gen_param_cell(rng, stats); best = cell; bs = cell_sim(cell, stats.get("embs"))
            for _t in range(5):
                if bs >= SIM_GATE: break
                cand = gen_param_cell(rng, stats); cs = cell_sim(cand, stats.get("embs"))
                if cs > bs: best, bs = cand, cs
            centers_sim.append(bs)
        cell = best * (0.96 + 0.05 * field[zc, yc, xc])           # brilho quase-constante herdado do tecido
        rz, ry, rx = np.array(cell.shape) // 2
        if zc-rz < 0 or yc-ry < 0 or xc-rx < 0 or zc+rz+1 > Z or yc+ry+1 > Y or xc+rx+1 > X:
            continue
        sub = dst[zc-rz:zc+rz+1, yc-ry:yc+ry+1, xc-rx:xc+rx+1]
        _sb = float(os.environ.get("SYNTH_SOFT_BLEND", "0.0"))  # BORDA suave celula-tecido (ataca grad_mag): alpha=cell
        if _sb > 0:
            a = np.clip(cell / (float(cell.max()) + 1e-6), 0, 1) * _sb   # alpha proporcional a' intensidade da celula
            dst[zc-rz:zc+rz+1, yc-ry:yc+ry+1, xc-rx:xc+rx+1] = np.maximum(sub, (1-a)*sub + a*cell)  # transicao suave, sem borda dura
        else:
            dst[zc-rz:zc+rz+1, yc-ry:yc+ry+1, xc-rx:xc+rx+1] = np.maximum(sub, cell)
        R = CELL_R_VOX                                          # caixa TIGHT ±R (convencao do GT real), nao o padding do array
        centers.append([zc, yc, xc]); boxes.append([zc-R, yc-R, xc-R, zc+R+1, yc+R+1, xc+R+1])
    # DIVISOES (taxa real medida): fracao das celulas GERA um 2o nucleo proximo (haltere) = celulas NASCENDO
    dst, boxes, centers = add_divisions(dst, boxes, centers, field, stats, rng)
    if _dec:
        # FUNDO e CELULAS com PSFs independentes, depois compostos. Rompe a tensao hi_freq x count.
        _bgb = float(os.environ.get("SYNTH_BG_BLUR", str(BLUR_SIGMA)))
        if _bgb > 0: vol = gaussian_filter(vol, (_bgb * PSF_Z_RATIO, _bgb, _bgb))
        if BLUR_SIGMA > 0: dst = gaussian_filter(dst, (BLUR_SIGMA * PSF_Z_RATIO, BLUR_SIGMA, BLUR_SIGMA))
        vol = np.maximum(vol, dst)
    vol = finish_volume(vol, boxes, field, bg, rng, skip_blur=_dec)   # blur ja aplicado por camada
    return vol, np.array(boxes, np.int32), np.array(centers, np.float32)

def finish_volume(vol, boxes, field, bg, rng, targets=None, skip_blur=False):
    """FINALIZACAO otica/camera (ordem convergida pelos otimizadores): PSF -> residual -> normalizacao
    -> HM -> perfil Z -> retarget de intensidade/celula -> ruido Poisson-Gauss -> power-spectrum match.
    Extraida de gen_volume_param p/ ser FONTE UNICA: o gerador temporal (synth_lineage) chama esta
    mesma funcao, senao os frames de uma sequencia sairiam fora da distribuicao ja calibrada."""
    vol = np.clip(vol, 0, None)
    if BLUR_SIGMA > 0 and not skip_blur:                                # skip: modo desacoplado ja borrou por camada
        vol = gaussian_filter(vol, (BLUR_SIGMA*PSF_Z_RATIO, BLUR_SIGMA, BLUR_SIGMA))  # PSF ANISOTROPICA (z>xy)
    if os.environ.get("SYNTH_RESIDUAL_POST", "0") == "1" and bg.get("residuals"):
        _rp = bg["residuals"][int(rng.integers(len(bg["residuals"])))]      # residual REAL apos o blur -> textura sobrevive (grad_mag)
        vol = vol + _rp * float(os.environ.get("SYNTH_RESIDUAL_POST_W", "0.7"))
    mm = np.percentile(vol, 99.6)                                  # p99.6 -> nucleos NAO saturam totalmente (menos branco puro)
    vol = np.clip(vol / mm, 0, 1) if mm > 0 else vol
    # HISTOGRAM MATCHING: forca o histograma do volume a igualar um frame REAL (nuvem clara). No modo
    # DARK isso RE-ERGUERIA o fundo escuro que acabamos de criar (o real contem o brilho medio das
    # celulas) -> destruiria a crispness. Por isso e' DESLIGADO quando SYNTH_DARK_MEDIUM=1: no modelo
    # fisico correto, a distribuicao de intensidade EMERGE das celulas+PSF, nao e' imposta.
    ref = bg.get("hm_ref_canon")
    if ref is not None and os.environ.get("SYNTH_DARK_MEDIUM", "0") != "1":
        try:
            from skimage.exposure import match_histograms
            vol = np.clip(0.05 * vol + 0.95 * match_histograms(vol, ref).astype(np.float32), 0, 1)
        except Exception:
            pass
    vol = apply_z_profile(vol, field_zprof(field))               # PERFIL Z do CAMPO especifico (nao a media global -> sem corcova)
    vol = retarget_cell_intensity(vol, boxes, rng, targets=targets)   # media por celula -> Normal data-driven (casa real)
    vol = poisson_gaussian_noise(vol, rng)                       # ruido de camera realista (shot+leitura) -> grao real
    vol = match_power_spectrum(vol, bg, rng)                     # ETAPA NOVA: textura real via Fourier (ataca grad_mag)
    return np.clip(vol, 0, 1).astype(np.float32)

def dedup_cens(cens, min_sep_vox):
    """Remove centroides duplicados (GT + DoG da mesma celula) mais proximos que min_sep."""
    out = []
    for c in np.atleast_2d(cens).astype(np.float32):
        if not out or np.linalg.norm((np.array(out) - c) * EFF_VOXEL, axis=1).min() >= min_sep_vox * EFF_VOXEL[0]:
            out.append(c.tolist())
    return np.array(out) if out else np.zeros((0, 3))

def gen_volume_layout(shape, bg, stats, rng, real_bg=True):
    """TRANSFERENCIA DE LAYOUT (Fable #1): celulas sinteticas nas POSICOES REAIS (centroides densos).
    Layout/densidade/agrupamento 100% REAIS -> ZERO erro de colocacao (o matador do sim-to-real).
    Rotulos densos+perfeitos = os proprios centroides. real_bg=True: fundo=campo real (max fidelidade)."""
    Z, Y, X = shape
    fcs = bg.get("field_cens"); fields = bg.get("fields")
    if not fcs or not fields: return gen_volume_param(shape, bg, stats, N_EST_PER_FRAME, rng)
    i = rng.integers(len(fcs)); field = fields[i].copy()
    cens = dedup_cens(fcs[i], bg.get("min_sep_vox", 3))          # posicoes REAIS (sem duplicar a mesma celula)
    fn = field / max(float(field.max()), 1e-6)
    _res = bg.get("residuals")
    if os.environ.get("SYNTH_REAL_RESIDUAL", "0") == "1" and _res:
        vol = field + _res[int(rng.integers(len(_res)))] * float(os.environ.get("SYNTH_RESIDUAL_W", "1.0"))  # TEXTURA REAL (grad_mag)
    else:
        _nz = rng.normal(0, bg.get("noise_std", bg["std"]), shape).astype(np.float32)
        _ncorr = float(os.environ.get("SYNTH_NOISE_CORR", "0.0"))    # knob p/ correlacionar ruido; default OFF: sweep provou que NAO move o hi_freq
        if _ncorr > 0: _nz = gaussian_filter(_nz, _ncorr)            # causa real do overshoot = low-pass do XY-pooling do REAL que o synth 64^3 nativo nao replica
        vol = field + _nz * (0.15 + 0.85 * fn)
    if not real_bg:                                             # IP-safe: adiciona matriz de tecido sintetica
        lo, hi = np.percentile(field, 50), np.percentile(field, 99)
        vol = vol + 0.38 * np.clip((field - lo) / (hi - lo + 1e-6), 0, 1)
    R = CELL_R_VOX; boxes, centers = [], []
    for (zc, yc, xc) in np.round(cens).astype(int):
        zc, yc, xc = int(np.clip(zc, R, Z-R-1)), int(np.clip(yc, R, Y-R-1)), int(np.clip(xc, R, X-R-1))
        cell = gen_param_cell(rng, stats) * (0.96 + 0.05 * field[zc, yc, xc])   # quase-constante -> preserva a normal
        rz, ry, rx = np.array(cell.shape) // 2
        if zc-rz < 0 or yc-ry < 0 or xc-rx < 0 or zc+rz+1 > Z or yc+ry+1 > Y or xc+rx+1 > X: continue
        vol[zc-rz:zc+rz+1, yc-ry:yc+ry+1, xc-rx:xc+rx+1] = np.maximum(vol[zc-rz:zc+rz+1, yc-ry:yc+ry+1, xc-rx:xc+rx+1], cell)
        boxes.append([zc-R, yc-R, xc-R, zc+R+1, yc+R+1, xc+R+1]); centers.append([zc, yc, xc])
    vol = np.clip(vol, 0, None)
    if BLUR_SIGMA > 0: vol = gaussian_filter(vol, (BLUR_SIGMA*PSF_Z_RATIO, BLUR_SIGMA, BLUR_SIGMA))
    mm = np.percentile(vol, 99); vol = np.clip(vol / mm, 0, 1) if mm > 0 else vol
    ref = bg.get("hm_ref_canon")
    if ref is not None:                                        # HM sempre -> intensidade casa o real (tight)
        try:
            from skimage.exposure import match_histograms
            vol = np.clip(0.05 * vol + 0.95 * match_histograms(vol, ref).astype(np.float32), 0, 1)
        except Exception: pass
    vol = apply_z_profile(vol, field_zprof(field))
    vol = retarget_cell_intensity(vol, boxes, rng)               # media por celula -> Normal data-driven (casa real)
    vol = poisson_gaussian_noise(vol, rng)                       # ruido de camera realista (shot+leitura) -> grao real
    vol = match_power_spectrum(vol, bg, rng)                     # ETAPA NOVA: textura real via Fourier (ataca grad_mag)
    return np.clip(vol, 0, 1).astype(np.float32), np.array(boxes, np.int32), np.array(centers, np.float32)

def render_at(field, positions, stats, bg, rng):
    """Renderiza celulas sinteticas NAS posicoes dadas + finaliza (blur/HM/z). Base do temporal."""
    Z, Y, X = field.shape; R = CELL_R_VOX
    fn = field / max(float(field.max()), 1e-6)
    vol = field + rng.normal(0, bg.get("noise_std", bg["std"]), field.shape).astype(np.float32) * (0.15 + 0.85 * fn)
    boxes = []
    for (zc, yc, xc) in np.round(positions).astype(int):
        zc, yc, xc = int(np.clip(zc, R, Z-R-1)), int(np.clip(yc, R, Y-R-1)), int(np.clip(xc, R, X-R-1))
        cell = gen_param_cell(rng, stats) * (0.96 + 0.05 * field[zc, yc, xc]); rz, ry, rx = np.array(cell.shape) // 2
        if zc-rz < 0 or yc-ry < 0 or xc-rx < 0 or zc+rz+1 > Z or yc+ry+1 > Y or xc+rx+1 > X: continue
        vol[zc-rz:zc+rz+1, yc-ry:yc+ry+1, xc-rx:xc+rx+1] = np.maximum(vol[zc-rz:zc+rz+1, yc-ry:yc+ry+1, xc-rx:xc+rx+1], cell)
        boxes.append([zc-R, yc-R, xc-R, zc+R+1, yc+R+1, xc+R+1])
    vol = np.clip(vol, 0, None)
    if BLUR_SIGMA > 0: vol = gaussian_filter(vol, (BLUR_SIGMA*PSF_Z_RATIO, BLUR_SIGMA, BLUR_SIGMA))
    mm = np.percentile(vol, 99); vol = np.clip(vol / mm, 0, 1) if mm > 0 else vol
    ref = bg.get("hm_ref_canon")
    if ref is not None:
        try:
            from skimage.exposure import match_histograms
            vol = np.clip(0.05 * vol + 0.95 * match_histograms(vol, ref).astype(np.float32), 0, 1)
        except Exception: pass
    vol = apply_z_profile(vol, field_zprof(field))
    return np.clip(vol, 0, 1).astype(np.float32), np.array(boxes, np.int32)

def resolve_collisions(pos, minsep_vox, iters=150, shape=None):
    """REPULSAO robusta: itera ATE nenhuma celula ficar < min_sep. Clip INTERNO a cada passo (borda nao
    reempurra p/ colisao). 150 iters convergem mesmo na densidade real alta -> ZERO choque/contato."""
    gate = minsep_vox * EFF_VOXEL[0]
    hi = (np.array(shape) - CELL_R_VOX - 1) if shape is not None else None
    for _ in range(iters):
        D = pos[:, None] - pos[None]; dm = np.linalg.norm(D * EFF_VOXEL, axis=2); np.fill_diagonal(dm, 1e9)
        if float(dm.min()) >= gate: break                       # resolvido -> para
        ii, jj = np.where(dm < gate)
        for a, b in zip(ii, jj):
            if a < b:
                v = D[a, b].astype(np.float32); nv = float(np.linalg.norm(v))
                if nv < 1e-4: v = (pos[a] - pos.mean(0)); nv = float(np.linalg.norm(v)) + 1e-4   # coincidentes: separa radial
                push = (v / nv) * ((gate - dm[a, b]) / EFF_VOXEL[0]) * 0.55
                pos[a] += push; pos[b] -= push
        if hi is not None: pos = np.clip(pos, CELL_R_VOX, hi)    # clip DENTRO do loop (borda nao reempurra)
    return pos

def gen_sequence(bg, stats, rng, motion, T=8):
    """SEQUENCIA TEMPORAL: celulas movem com PERSISTENCIA direcional + velocidade REAL, com REPULSAO
    anti-colisao. Cada celula mantem seu ID (rastreavel). Retorna [(vol, boxes, ids, pos)]."""
    fields = bg.get("fields"); fcs = bg.get("field_cens")
    if not fcs or not fields: return []
    i = rng.integers(len(fcs)); field = fields[i]
    pos = dedup_cens(fcs[i], bg.get("min_sep_vox", 3)).astype(np.float32)
    if len(pos) < 3: return []
    ids = np.arange(len(pos)); shp = np.array(field.shape)
    speed = motion["speed_med"] / EFF_VOXEL[0]                    # voxels/frame (real)
    persist = motion["persist"]; minsep = bg.get("min_sep_vox", 3)
    vel = rng.normal(0, speed, (len(pos), 3)).astype(np.float32)
    frames = []
    for t in range(T):
        vol, boxes = render_at(field, pos, stats, bg, rng)
        frames.append((vol, boxes, ids.copy(), pos.copy()))
        # MOVE: velocidade persistente (direcao real) + ruido, calibrada a velocidade real
        nv = persist * vel + (1 - persist) * rng.normal(0, speed, (len(pos), 3)).astype(np.float32)
        sp = np.linalg.norm(nv, axis=1, keepdims=True)
        nv = nv / (sp + 1e-6) * speed * rng.uniform(0.4, 1.3, (len(pos), 1))   # velocidade real com variacao
        pos = pos + nv; vel = nv
        pos = np.clip(pos, CELL_R_VOX, shp - CELL_R_VOX - 1)
        pos = resolve_collisions(pos, minsep, shape=shp)         # EVITA CHOQUE/CONTATO (por ultimo, com clip interno)
    return frames

def visualize_motion(bg, stats, rng, motion):
    """Visualiza a MOVIMENTACAO temporal: trajetorias + verificacao de COLISAO (dist minima por frame)."""
    frames = gen_sequence(bg, stats, rng, motion, T=8)
    if not frames: print("  [motion] sem sequencia"); return
    traj = np.array([f[3] for f in frames])                      # (T, N, 3) voxels
    contact = CELL_DIAM_UM * 0.5                                 # 1 raio = contato
    mind = []
    for f in frames:
        p = f[3]; D = np.linalg.norm((p[:, None] - p[None]) * EFF_VOXEL, axis=2); np.fill_diagonal(D, 1e9)
        mind.append(float(D.min()) if len(p) > 1 else 99.0)
    fig, ax = plt.subplots(1, 3, figsize=(18, 5.5))
    for n in range(traj.shape[1]):
        ax[0].plot(traj[:, n, 2], traj[:, n, 1], "-", lw=0.7, alpha=0.5)
    ax[0].scatter(traj[0, :, 2], traj[0, :, 1], s=10, c="lime", label="inicio")
    ax[0].scatter(traj[-1, :, 2], traj[-1, :, 1], s=10, c="red", label="fim")
    ax[0].set_title("Trajetorias das celulas (movimento realista)"); ax[0].invert_yaxis(); ax[0].legend()
    ax[1].plot(range(len(frames)), mind, "-o"); ax[1].axhline(contact, ls="--", c="r", label=f"contato ({contact:.1f}um)")
    ax[1].set_title("Distancia MINIMA entre celulas/frame (colisao?)"); ax[1].set_xlabel("frame"); ax[1].set_ylabel("um"); ax[1].legend(); ax[1].set_ylim(0, None)
    ax[2].imshow(frames[-1][0].max(0), cmap="magma"); ax[2].set_title(f"frame final (MIP-Z)"); ax[2].axis("off")
    plt.suptitle(f"Movimentacao temporal: {motion['speed_med']:.1f}um/frame, persistencia {motion['persist']:.2f} | dist min {min(mind):.1f}um "
                 f"({'SEM colisao' if min(mind) >= contact*0.8 else 'ATENCAO'})", weight="bold")
    plt.savefig(FIG / "movimentacao.png", dpi=170, bbox_inches="tight"); plt.show()
    print(f"  [motion] dist minima entre celulas ao longo do tempo = {min(mind):.1f}um (contato {contact:.1f}um) -> "
          f"{'SEM colisao/contato' if min(mind) >= contact*0.8 else 'ATENCAO: colisao'}")

def patch_embedding(patch):
    p = patch.astype(np.float32); c = np.array(p.shape) // 2
    zz, yy, xx = np.indices(p.shape)
    d = np.sqrt(((zz-c[0])*EFF_VOXEL[0])**2 + ((yy-c[1])*EFF_VOXEL[1])**2 + ((xx-c[2])*EFF_VOXEL[2])**2)
    bins = np.linspace(0, d.max() + 1e-6, 6)
    radial = [p[(d >= bins[i]) & (d < bins[i+1])].mean() if ((d >= bins[i]) & (d < bins[i+1])).any() else 0.0 for i in range(5)]
    gz, gy, gx = np.gradient(p); gmag = np.sqrt(gz**2 + gy**2 + gx**2)
    return np.array(radial + [p.mean(), p.std(), p.max(), np.percentile(p, 90), gmag.mean(), gmag.std(), float((p > p.mean()).mean())], np.float32)

def embed_set(ps): return np.stack([patch_embedding(p) for p in ps]) if ps else np.zeros((0, 12))

def frechet_distance(A, B):
    from scipy.linalg import sqrtm
    mu1, mu2 = A.mean(0), B.mean(0)
    s1 = np.cov(A, rowvar=False) + 1e-6*np.eye(A.shape[1]); s2 = np.cov(B, rowvar=False) + 1e-6*np.eye(B.shape[1])
    cm = sqrtm(s1 @ s2); cm = cm.real if np.iscomplexobj(cm) else cm
    return float(np.sum((mu1-mu2)**2) + np.trace(s1 + s2 - 2*cm))

def mmd_rbf(A, B, gamma=None):
    from sklearn.metrics.pairwise import rbf_kernel
    g = gamma or 1.0/max(A.shape[1], 1)
    return float(rbf_kernel(A, A, g).mean() + rbf_kernel(B, B, g).mean() - 2*rbf_kernel(A, B, g).mean())

def wass_intensity(a, b):
    from scipy.stats import wasserstein_distance
    return float(wasserstein_distance(a, b))

def ks_stat(a, b):
    from scipy.stats import ks_2samp
    return float(ks_2samp(a, b).statistic)               # 0 = distribuicoes iguais

def _radial_power(img):
    F = np.abs(np.fft.fftshift(np.fft.fft2(img.astype(np.float32)))) ** 2
    c = np.array(img.shape) // 2; yy, xx = np.indices(img.shape)
    r = np.sqrt((yy - c[0]) ** 2 + (xx - c[1]) ** 2).astype(int)
    return np.log(np.bincount(r.ravel(), F.ravel()) / np.maximum(np.bincount(r.ravel()), 1) + 1e-9)

def spectral_dist(a_img, b_img):
    pa, pb = _radial_power(a_img), _radial_power(b_img); n = min(len(pa), len(pb))
    return float(np.abs(pa[:n] - pb[:n]).mean())         # distancia do espectro de potencia (textura)

def ssim_mip(a_vol, b_vol):
    try:
        from skimage.metrics import structural_similarity as ssim
        return float(ssim(a_vol.max(0), b_vol.max(0), data_range=1.0))   # 1 = identico
    except Exception:
        return float("nan")

def grad_wass(a_vol, b_vol):
    ga = np.sqrt(sum(g ** 2 for g in np.gradient(a_vol))); gb = np.sqrt(sum(g ** 2 for g in np.gradient(b_vol)))
    return wass_intensity(ga.ravel()[:200000], gb.ravel()[:200000])      # textura de bordas

def embed_set_3d(patches, n=300):
    """Conjunto de embeddings 3d (forma/estrutura) de patches de celula."""
    return np.stack([embed3d(p) for p in patches[:n]]) if patches else np.zeros((0, 48))

def sim3d_analysis(real_patches, synth_patches):
    """Similaridade no ESPACO VETORIAL 3D: Frechet entre distribuicoes + % de celulas synth >=70%
    similares a alguma celula real + mediana da similaridade."""
    A = embed_set_3d(real_patches); B = embed_set_3d(synth_patches)
    if len(A) < 3 or len(B) < 3: return {}, np.array([])
    sims = np.array([float(np.max(A @ B[i])) for i in range(len(B))])     # cada synth vs real mais parecida
    return dict(frechet3d=frechet_distance(A, B), mmd3d=mmd_rbf(A, B),
                sim3d_median=float(np.median(sims)), sim3d_ge70=float((sims >= 0.70).mean()),
                sim3d_min=float(sims.min())), sims

def blur_sweep(real_frames, bg, stats, rng, sigmas=(0.0, 0.25, 0.4, 0.55, 0.7, 0.85)):
    """Testa desfoques (PSF) e mede similaridade de TEXTURA (espectral + gradiente) -> escolhe o ideal."""
    global BLUR_SIGMA
    rmip = real_frames[0].max(0); rows = []
    for s in sigmas:
        BLUR_SIGMA = s
        vols = [gen_volume_param(bg["shape"], bg, stats, N_EST_PER_FRAME, rng)[0] for _ in range(3)]
        sp = float(np.mean([spectral_dist(rmip, v.max(0)) for v in vols]))     # textura (freq)
        gr = float(np.mean([grad_wass(real_frames[0], v) for v in vols]))      # bordas
        rows.append((float(s), sp, gr, sp + gr))                              # score combinado
    best = min(rows, key=lambda r: r[3]); BLUR_SIGMA = best[0]
    print(f"[blur sweep] sigma ideal = {best[0]:.2f} (espectral {best[1]:.3f} + grad {best[2]:.3f})")
    fig, ax = plt.subplots(1, 1, figsize=(7, 4))
    ax.plot([r[0] for r in rows], [r[1] for r in rows], "-o", label="espectral (textura)")
    ax.plot([r[0] for r in rows], [r[2] for r in rows], "-o", label="gradiente (bordas)")
    ax.plot([r[0] for r in rows], [r[3] for r in rows], "-o", lw=2, label="combinado")
    ax.axvline(best[0], ls="--", c="k", lw=1, label=f"ideal sigma={best[0]:.2f}")
    ax.set_title("Sweep de desfoque (PSF): similaridade vs sigma"); ax.set_xlabel("sigma gaussiano"); ax.legend()
    plt.savefig(FIG / "blur_sweep.png", dpi=180, bbox_inches="tight"); plt.show()
    return rows, best[0]

def gallery_stages(stats, rng):
    """Galeria: celula parametrica por ESTAGIO (early/dev) x FORMA (round/eliptica) + similaridade 3d."""
    combos = [("early", "round"), ("early", "ellip"), ("dev", "round"), ("dev", "ellip")]
    fig, ax = plt.subplots(2, 4, figsize=(16, 8))
    for j, (st, sk) in enumerate(combos):
        cell = gen_param_cell(rng, stats, stage=st, shape_kind=sk); cc = cell.shape[0] // 2
        s = cell_sim(cell, stats.get("embs"))
        ax[0, j].imshow(cell.max(0), cmap="magma"); ax[0, j].set_title(f"{st} / {sk}\nsim3d={s:.2f}"); ax[0, j].axis("off")
        ax[1, j].imshow(cell[cc], cmap="magma"); ax[1, j].set_title("corte central"); ax[1, j].axis("off")
    plt.suptitle("Celulas parametricas: ESTAGIO (iniciando/desenvolvida) x FORMA (round/ELIPTICA)", weight="bold")
    plt.savefig(FIG / "gallery_stages.png", dpi=160, bbox_inches="tight"); plt.show()

## Dados reais + split por video (sem leakage)

In [ ]:
def find_train_dir():
    for b in [Path("/kaggle/input/competitions/biohub-cell-tracking-during-development/train"),
              Path("/kaggle/input/biohub-cell-tracking-during-development/train")]:
        if b.exists() and any(b.glob("*.zarr")): return b
    for p in Path("/kaggle/input").rglob("train"):
        if p.is_dir() and any(p.glob("*.zarr")): return p
    return None

def read_frame(zp, t):
    meta = json.loads((zp/"0"/"zarr.json").read_text()); shape = tuple(int(v) for v in meta["shape"])
    dtype = np.dtype(meta["data_type"]); fs = shape[1:]
    try:
        import blosc2
        a = np.frombuffer(blosc2.decompress((zp/"0"/"c"/str(t)/"0"/"0"/"0").read_bytes()), dtype=dtype)
        if a.size == int(np.prod(fs)): return a.reshape(fs).copy(), shape
    except Exception: pass
    import zarr
    return np.asarray(zarr.open(zp/"0", mode="r")[t]), shape

def norm(f):
    lo, hi = np.percentile(f, 1), np.percentile(f, 99.5)
    return np.clip((f - lo)/max(hi - lo, 1e-6), 0, 1).astype(np.float32)

def read_geff_nodes(gp):
    import tracksdata as td
    g = td.graph.IndexedRXGraph.from_geff(gp); g = g[0] if isinstance(g, tuple) else g
    na = g.node_attrs(attr_keys=["t", "z", "y", "x"])
    by_t = {}
    for r in na.iter_rows(named=True):
        by_t.setdefault(int(r["t"]), []).append([float(r["z"]), float(r["y"]), float(r["x"])])
    return {t: np.array(v) for t, v in by_t.items()}

def geff_for(td_, name):
    for c in [td_/f"{name}.geff", td_/"gt"/f"{name}.geff"]:
        if c.exists(): return c
    h = list(td_.rglob(f"{name}*.geff")); return h[0] if h else None

def measure_division_rate(train_dir, videos):
    """TAXA DE GERACAO DE CELULAS (divisoes): fracao de nos que se dividem (2 filhos) no lineage GT.
    Alimenta o termo 0.1*division_jaccard da metrica. Robusto a variacoes da API do tracksdata."""
    import tracksdata as td
    from collections import Counter
    div, tot = 0, 0
    for name in videos:
        gp = geff_for(train_dir, name)
        if gp is None: continue
        try:
            g = td.graph.IndexedRXGraph.from_geff(gp); g = g[0] if isinstance(g, tuple) else g
            srcs = []
            try:
                ea = g.edge_attrs(attr_keys=[])
                for r in ea.iter_rows(named=True):
                    srcs.append(r.get("source_id", r.get("source", r.get("src", r.get("u")))))
            except Exception:
                for e in g.graph.edge_list(): srcs.append(e[0])   # fallback rustworkx
            oc = Counter([s for s in srcs if s is not None])
            div += sum(1 for v in oc.values() if v >= 2)
            tot += (g.num_nodes() if callable(getattr(g, "num_nodes", None)) else len(g.node_attrs(attr_keys=["t"]).to_numpy()))
        except Exception as e:
            print(f"  [div rate] {name}: {e}")
    rate = div / max(tot, 1)
    print(f"=== TAXA DE GERACAO (divisoes reais no lineage GT) ===")
    print(f"  {div} divisoes em {tot} nos -> taxa = {rate*100:.2f}% das celulas se dividem")
    return float(rate)

def measure_motion(train_dir, videos, gate_um=7.0):
    """MOVIMENTACAO REAL: velocidade (um/frame) + PERSISTENCIA direcional (cos entre velocidades
    consecutivas da mesma celula). Matching NN frame-a-frame do GT (nao precisa das arestas)."""
    speeds, coss = [], []
    for name in videos:
        gp = geff_for(train_dir, name)
        if gp is None: continue
        try: nodes = read_geff_nodes(gp)
        except Exception: continue
        ts = sorted(nodes); prev_vel = {}
        for a, b in zip(ts, ts[1:]):
            if b != a + 1: continue
            Pa = np.atleast_2d(nodes[a]).astype(np.float32) * EFF_VOXEL
            Pb = np.atleast_2d(nodes[b]).astype(np.float32) * EFF_VOXEL
            if not len(Pa) or not len(Pb): continue
            cur_vel = {}
            for ia, pa in enumerate(Pa):
                d = np.linalg.norm(Pb - pa, axis=1); j = int(d.argmin())
                if d[j] <= gate_um:
                    v = Pb[j] - pa; sp = float(np.linalg.norm(v)); speeds.append(sp)
                    if sp > 0.3: cur_vel[ia] = v / sp
                    if ia in prev_vel and sp > 0.3:                      # persistencia direcional
                        coss.append(float(np.dot(prev_vel[ia], v / sp)))
            prev_vel = {j: cur_vel[ia] for ia, (pa) in enumerate(Pa) for j in [int(np.linalg.norm(Pb - pa, axis=1).argmin())] if ia in cur_vel}
    speeds = np.array(speeds) if speeds else np.array([2.3]); coss = np.array(coss) if coss else np.array([0.4])
    m = dict(speed_med=float(np.median(speeds)), speed_p90=float(np.percentile(speeds, 90)),
             persist=float(np.clip(np.mean(coss), 0.0, 0.95)))
    print(f"=== MOVIMENTACAO REAL (GT) ===")
    print(f"  velocidade: mediana {m['speed_med']:.2f} um/frame (p90 {m['speed_p90']:.2f}) | persistencia direcional {m['persist']:.2f} (1=reto, 0=aleatorio)")
    return m

## Pipeline principal (roda no Kaggle; SMOKE usa dado sintetico)

In [ ]:
def collect_real(train_dir, videos, frames_per=int(os.environ.get("SYNTH_FRAMESPER", "8"))):
    """Coleta templates (GT), centroides e frames p/ liquido. frames_per MAIOR = referencia real mais rica."""
    templates, frames, cens_list, gt_boxes = [], [], [], []
    for name in videos:
        zp = train_dir/f"{name}.zarr"; gp = geff_for(train_dir, name)
        if gp is None: continue
        try: nodes = read_geff_nodes(gp)
        except Exception: continue
        ts = sorted(nodes)[:: max(1, len(nodes)//frames_per)][:frames_per]
        for t in ts:
            fr = norm(read_frame(zp, t)[0]); frp = pool_xy(fr)
            cens = nodes[t] / np.array(DOWNSAMPLE)           # centroides no espaco pooled
            templates += build_template_bank(frp, cens)
            frames.append(frp); cens_list.append(cens)
    return templates, frames, cens_list

def save_sample(split, ver, idx, vol, boxes):
    d = DSDIR/ver/split; d.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(d/f"{idx:04d}.npz", volume=vol.astype(np.float32), boxes=boxes.astype(np.int32),
                        labels=np.zeros(len(boxes), np.int64))  # classe 0 = celula (formato DETR)

def gen_realbased_variation(frames, cens_list, bg, stats, rng):
    """DADO REAL-BASED: pega uma CAPTURA REAL (frame pooled) + suas deteccoes reais e AGLOMERA em variacoes
    (rotacao 90, flips, jitter de gama/intensidade, mistura de 2 regioes reais). Movimento/captura = reais."""
    i = rng.integers(len(frames))
    vol = np.asarray(frames[i], np.float32).copy()
    cen = np.round(np.atleast_2d(cens_list[i])).astype(np.float32)
    # AGLOMERA: mistura (maximo) com outra regiao real -> mais celulas reais por imagem (varias imagens numa)
    if rng.random() < 0.6 and len(frames) > 1:
        j = rng.integers(len(frames)); vol = np.maximum(vol, np.asarray(frames[j], np.float32))
        cen = np.vstack([cen, np.round(np.atleast_2d(cens_list[j])).astype(np.float32)]) if len(cens_list[j]) else cen
    Z, Y, X = vol.shape
    # rotacao 90 em XY (multipla) + flips: transforma coords junto (captura real, so reorienta)
    k = int(rng.integers(4)); vol = np.rot90(vol, k, axes=(1, 2))
    if k:
        for _ in range(k): cen[:, [1, 2]] = np.stack([cen[:, 2], (Y - 1) - cen[:, 1]], 1)
    if rng.random() < 0.5: vol = vol[:, ::-1]; cen[:, 1] = (Y - 1) - cen[:, 1]
    if rng.random() < 0.5: vol = vol[:, :, ::-1]; cen[:, 2] = (X - 1) - cen[:, 2]
    vol = np.clip(vol ** float(rng.uniform(0.85, 1.15)) * float(rng.uniform(0.9, 1.1)), 0, 1)  # jitter gama/intensidade
    vol = np.ascontiguousarray(vol, np.float32)
    R = CELL_R_VOX; sh = np.array(vol.shape) - 1
    cen = cen[(cen[:, 0] >= R) & (cen[:, 1] >= R) & (cen[:, 2] >= R) &
              (cen[:, 0] < sh[0]-R) & (cen[:, 1] < sh[1]-R) & (cen[:, 2] < sh[2]-R)]
    boxes = np.array([[c[0]-R, c[1]-R, c[2]-R, c[0]+R+1, c[1]+R+1, c[2]+R+1] for c in cen], np.int32)
    return vol, boxes

def run_scale(train_dir, tr, rng):
    """ESCALA: muitos volumes UNICOS sem figuras (throughput). SYNTH_REALBASED=1 usa captura real+aug.
    SYNTH_CHUNK varia a seed (chunks != dados). Para em SYNTH_MAXGB (limite de output do Kaggle)."""
    import time
    realbased = os.environ.get("SYNTH_REALBASED", "0") == "1"
    n = int(os.environ.get("SYNTH_NSCALE", "13000")); chunk = int(os.environ.get("SYNTH_CHUNK", "0"))
    maxgb = float(os.environ.get("SYNTH_MAXGB", "15.0"))
    NV = int(os.environ.get("SYNTH_REFVIDEOS", str(len(tr))))
    templates, frames, cens = collect_real(train_dir, tr[:NV])
    bg = liquid_model(frames, cens); stats = measure_cell_stats(templates); shape = frames[0].shape
    global _CELL_MU_RT, _CELL_SIG_RT, _CELL_MEANS_RT, _TEMPLATES_RT
    _mn = np.asarray(stats["means"], float)
    _CELL_MU_RT = float(np.clip(np.median(_mn), 0.1, 0.9)); _CELL_SIG_RT = float(np.clip(np.std(_mn), 0.03, 0.20)); _CELL_MEANS_RT = _mn
    _TEMPLATES_RT = templates                                    # banco real p/ o modo HIBRIDO (SYNTH_CELL_SOURCE=template)
    if os.environ.get("SYNTH_CELL_SOURCE", "param") == "template":
        print(f"  MODO HIBRIDO: celulas de template REAL ({len(templates)} no banco) + placement param", flush=True)
    global DIV_RATE;
    if DIV_RATE <= 0: DIV_RATE = measure_division_rate(train_dir, tr[:min(NV, len(tr))])
    d = DSDIR / ("R_realbased" if realbased else "D_scale"); d.mkdir(parents=True, exist_ok=True)
    rs = np.random.default_rng(10007 * (chunk + 1))
    print(f"=== ESCALA {'REAL-BASED' if realbased else 'D_param unico'} chunk={chunk} alvo={n} vols max={maxgb}GB "
          f"| intensidade real media={_CELL_MU_RT:.3f} ===", flush=True)
    t0 = time.time()
    for i in range(n):
        if realbased: vol, boxes = gen_realbased_variation(frames, cens, bg, stats, rs)
        else: vol, boxes, _ = gen_volume_param(shape, bg, stats, N_EST_PER_FRAME, rs)
        if len(boxes) == 0: continue
        np.savez(d / f"{chunk:02d}_{i:05d}.npz", volume=vol.astype(np.float32),
                 boxes=np.asarray(boxes, np.int32), labels=np.zeros(len(boxes), np.int64))
        if i % 500 == 0 and i > 0:
            gb = sum(f.stat().st_size for f in d.glob('*.npz')) / 1e9
            print(f"  {i}/{n} | {gb:.1f}GB | {(time.time()-t0)/i:.2f}s/vol | proj {(time.time()-t0)/i*n/3600:.1f}h", flush=True)
            if gb >= maxgb: print(f"  ATINGIU {gb:.1f}GB (limite) -> parando em {i} vols", flush=True); break
    gb = sum(f.stat().st_size for f in d.glob('*.npz')) / 1e9
    print(f"=== ESCALA CONCLUIDA: {len(list(d.glob('*.npz')))} vols, {gb:.2f}GB em {d} ===", flush=True)

def run_split(train_dir, split, videos, rng, n_samples):
    templates, frames, cens = collect_real(train_dir, videos)
    if len(templates) < 5:
        print(f"  [{split}] templates insuficientes ({len(templates)})"); return None
    bg = liquid_model(frames, cens); shape = frames[0].shape
    stats = measure_cell_stats(templates)
    global _CELL_MU_RT, _CELL_SIG_RT, _CELL_MEANS_RT, _TEMPLATES_RT
    _mn = np.asarray(stats["means"], float)
    _CELL_MU_RT = float(np.clip(np.median(_mn), 0.1, 0.9)); _CELL_SIG_RT = float(np.clip(np.std(_mn), 0.03, 0.20)); _CELL_MEANS_RT = _mn
    _TEMPLATES_RT = templates                                    # banco real p/ o modo HIBRIDO (D_param vira hibrido se SYNTH_CELL_SOURCE=template)
    _hyb = os.environ.get("SYNTH_CELL_SOURCE", "param") == "template"
    print(f"  [{split}] {len(templates)} templates, {len(frames)} frames | liquido mean={bg['mean']:.3f}{' | D_param=HIBRIDO (template real)' if _hyb else ''}")
    # VERSAO A: sintetico por TEMPLATES reais colados
    for i in range(n_samples):
        vol, boxes, _ = gen_volume(shape, bg, templates, N_EST_PER_FRAME, rng)
        save_sample(split, "A_synth", i, vol, boxes)
    # VERSAO D: parametrica OU hibrida (celula template real) conforme SYNTH_CELL_SOURCE
    for i in range(n_samples):
        vol, boxes, _ = gen_volume_param(shape, bg, stats, N_EST_PER_FRAME, rng)
        save_sample(split, "D_param", i, vol, boxes)
    # VERSAO E: TRANSFERENCIA DE LAYOUT (Fable) - aparencia sintetica nas POSICOES REAIS (zero erro de colocacao)
    for i in range(n_samples):
        vol, boxes, _ = gen_volume_layout(shape, bg, stats, rng, real_bg=True)
        if len(boxes): save_sample(split, "E_layout", i, vol, boxes)
    # VERSAO B: so real rotulado (frame real + caixas do GT)
    for i, (fr, cs) in enumerate(zip(frames, cens)):
        boxes = np.array([[int(z)-CELL_R_VOX, int(y)-CELL_R_VOX, int(x)-CELL_R_VOX,
                           int(z)+CELL_R_VOX+1, int(y)+CELL_R_VOX+1, int(x)+CELL_R_VOX+1] for z, y, x in cs], np.int32)
        save_sample(split, "B_real", i, fr, boxes)
    # VERSAO C: real + DoG-precision-100% (GT + DoG confiavel)
    for i, (fr, cs) in enumerate(zip(frames, cens)):
        det, _ = dog_detect_precise(fr)
        allc = np.vstack([cs, det]) if len(det) else cs
        boxes = np.array([[int(z)-CELL_R_VOX, int(y)-CELL_R_VOX, int(x)-CELL_R_VOX,
                           int(z)+CELL_R_VOX+1, int(y)+CELL_R_VOX+1, int(x)+CELL_R_VOX+1] for z, y, x in allc], np.int32)
        save_sample(split, "C_real_dog", i, fr, boxes)
    return dict(templates=templates, frames=frames, cens=cens, bg=bg, shape=shape)

def validate(real_frames, real_cens, synth_dir, split="train", n=40, version="A_synth"):
    """Similaridade real vs sintetico (Frechet/MMD/Wasserstein/KS/espectral/gradiente)."""
    real_patches = []
    for fr, cs in zip(real_frames, real_cens): real_patches += build_template_bank(fr, cs)
    synth_patches, synth_vols, synth_cens, synth_boxes = [], [], [], []
    for f in sorted(glob.glob(str(DSDIR/version/split/"*.npz")))[:n]:
        d = np.load(f); synth_vols.append(d["volume"])
        cen = (d["boxes"][:, :3] + d["boxes"][:, 3:]) / 2
        synth_patches += build_template_bank(d["volume"], cen)
        synth_cens.append(cen); synth_boxes.append(d["boxes"])
    A, B = embed_set(real_patches[:400]), embed_set(synth_patches[:400])
    if len(A) < 5 or len(B) < 5: return {}
    # ANALISE NO ESPACO VETORIAL 3D: Frechet + % de celulas synth >=70% similares a uma real
    sim3d, sims_arr = sim3d_analysis(real_patches[:400], synth_patches[:400])
    # figura: distribuicao de similaridade 3d (cada celula synth vs real mais parecida)
    if len(sims_arr):
        fig, ax = plt.subplots(1, 2, figsize=(13, 4))
        ax[0].hist(sims_arr, bins=30, color="teal", alpha=.8)
        ax[0].axvline(0.70, ls="--", c="r", lw=1.5, label=f">=70%: {(sims_arr>=0.70).mean()*100:.0f}%")
        ax[0].set_title(f"Similaridade 3D de cada celula synth a uma REAL ({version})"); ax[0].set_xlabel("cosseno"); ax[0].legend()
        # BRILHO MEDIO POR CELULA: mede a REGIAO ATIVA (>0.3 do pico) p/ casar o alvo Normal(0.30)
        rm = np.array([float(p[p > 0.3*p.max()].mean()) for p in real_patches[:300] if p.max() > 0])
        sm = np.array([float(p[p > 0.3*p.max()].mean()) for p in synth_patches[:300] if p.max() > 0])
        try:
            from scipy.stats import shapiro
            sp = float(shapiro(sm[:200]).pvalue) if len(sm) >= 8 else 0.0
        except Exception: sp = -1.0
        ax[1].hist(rm, bins=25, alpha=.5, density=True, label=f"real (media {rm.mean():.2f})", color="steelblue")
        ax[1].hist(sm, bins=25, alpha=.5, density=True, label=f"synth (media {sm.mean():.2f})", color="darkorange")
        ax[1].axvline(CELL_MEAN_TARGET, ls="--", c="red", lw=2, label=f"ALVO {CELL_MEAN_TARGET:.2f}")
        xs = np.linspace(0, max(rm.max(), sm.max(), 0.6), 100)   # curvas NORMAIS ajustadas
        ax[1].plot(xs, np.exp(-0.5*((xs-rm.mean())/(rm.std()+1e-6))**2)/((rm.std()+1e-6)*np.sqrt(2*np.pi)), "b-", lw=2)
        ax[1].plot(xs, np.exp(-0.5*((xs-sm.mean())/(sm.std()+1e-6))**2)/((sm.std()+1e-6)*np.sqrt(2*np.pi)), "-", c="darkorange", lw=2)
        ax[1].set_title(f"Brilho/celula: synth media {sm.mean():.3f} std {sm.std():.3f} | Shapiro-normal p={sp:.3f}"); ax[1].legend(fontsize=8)
        plt.suptitle(f"Espaco vetorial 3D + brilho da celula - {version}", weight="bold")
        plt.savefig(FIG/f"sim3d_{version}.png", dpi=180, bbox_inches="tight"); plt.show()
    # amostra TODOS os frames representativamente (nao truncar no 1o frame -> media real correta)
    rcat = np.concatenate([f.ravel()[::max(1, f.size // 12000)] for f in real_frames])
    scat = np.concatenate([v.ravel()[::max(1, v.size // 8000)] for v in synth_vols])
    metrics = dict(frechet_fid3d=frechet_distance(A, B), mmd=mmd_rbf(A, B),
                   wasserstein_intensity=wass_intensity(rcat, scat),
                   # + METRICAS: KS, SSIM, espectral (textura), gradiente (bordas)
                   ks_intensity=ks_stat(rcat, scat),
                   spectral_dist=spectral_dist(real_frames[0].max(0), synth_vols[0].max(0)),
                   grad_wasserstein=grad_wass(real_frames[0], synth_vols[0]),
                   # ESTRUTURAIS: organizacao espacial que o Frechet ignora
                   bright_frac_real=float((rcat > 0.4).mean()), bright_frac_synth=float((scat > 0.4).mean()),
                   mean_int_real=float(rcat.mean()), mean_int_synth=float(scat.mean()),
                   n_real_patches=len(real_patches), n_synth_patches=len(synth_patches))
    metrics.update(sim3d); metrics["blur_sigma"] = float(BLUR_SIGMA)
    # ===== AGRUPAMENTO ESPACIAL: como as celulas se distribuem/aglomeram =====
    real_det = [dog_detect_precise(fr)[0] for fr in real_frames[:8]]              # posicoes reais densas (DoG)
    rnn = np.concatenate([nn_um(d) for d in real_det if len(d) > 1]) if real_det else np.array([])
    snn = np.concatenate([nn_um(c) for c in synth_cens if len(c) > 1]) if synth_cens else np.array([])
    rclump = np.mean([clump_index(d, real_frames[0].shape) for d in real_det if len(d) > 2]) if real_det else 1.0
    sclump = np.mean([clump_index(c, synth_vols[0].shape) for c in synth_cens if len(c) > 2]) if synth_cens else 1.0
    # AUDITORIA DE SOBREPOSICAO (alerta do usuario): fracao de caixas com IoU>0.3
    ov = [overlap_audit(b) for b in synth_boxes if len(b) > 1]
    R = CELL_R_VOX                                                                 # sobreposicao REAL: mesmas caixas ±R nas deteccoes reais
    ovr = [overlap_audit(np.array([[z-R, y-R, x-R, z+R+1, y+R+1, x+R+1] for z, y, x in d])) for d in real_det if len(d) > 1]
    metrics.update(overlap_frac_real=float(np.mean([o[0] for o in ovr])) if ovr else 0.0,
                   overlap_maxiou_real=float(np.max([o[1] for o in ovr])) if ovr else 0.0,
                   nn_real_med_um=float(np.median(rnn)) if len(rnn) else 0.0,
                   nn_synth_med_um=float(np.median(snn)) if len(snn) else 0.0,
                   nn_real_p5_um=float(np.percentile(rnn, 5)) if len(rnn) else 0.0,
                   nn_synth_p5_um=float(np.percentile(snn, 5)) if len(snn) else 0.0,
                   clump_real=float(rclump), clump_synth=float(sclump),
                   overlap_frac_synth=float(np.mean([o[0] for o in ov])) if ov else 0.0,
                   overlap_maxiou_synth=float(np.max([o[1] for o in ov])) if ov else 0.0)
    # figura de agrupamento: NN (vizinho + proximo) + aglomeracao
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    if len(rnn): ax[0].hist(rnn, bins=40, alpha=.6, density=True, label=f"real (med {np.median(rnn):.1f}um)")
    if len(snn): ax[0].hist(snn, bins=40, alpha=.6, density=True, label=f"sintetico (med {np.median(snn):.1f}um)")
    ax[0].axvline(CELL_DIAM_UM, ls="--", c="k", lw=1, label=f"1 diametro ({CELL_DIAM_UM:.1f}um)")
    ax[0].set_title("Distancia ao vizinho + proximo (agrupamento)"); ax[0].set_xlabel("um"); ax[0].legend()
    ax[1].bar([0, 1], [rclump, sclump], color=["steelblue", "orange"]); ax[1].set_xticks([0, 1])
    ax[1].set_xticklabels(["real", "sintetico"]); ax[1].set_title("Indice de aglomeracao (>1 = agregado)")
    plt.suptitle(f"Agrupamento espacial - {version}", weight="bold")
    plt.savefig(FIG/f"agrupamento_{version}.png", dpi=190, bbox_inches="tight"); plt.show()
    # ===== NOVO PONTO DE VISTA: conectividade (massa vs teia) + perfil de profundidade Z =====
    cfr = float(np.mean([connected_frac(f) for f in real_frames[:8]]))
    cfs = float(np.mean([connected_frac(v) for v in synth_vols[:8]]))
    zr = np.mean([f.mean(axis=(1, 2)) for f in real_frames], axis=0)      # TODOS os frames (mesma populacao do synth)
    zs = np.mean([v.mean(axis=(1, 2)) for v in synth_vols], axis=0)
    metrics.update(connected_frac_real=cfr, connected_frac_synth=cfs)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].bar([0, 1], [cfr, cfs], color=["steelblue", "orange"]); ax[0].set_xticks([0, 1])
    ax[0].set_xticklabels([f"real\n{cfr:.2f}", f"sintetico\n{cfs:.2f}"])
    ax[0].set_title("Conectividade: fracao no maior componente (1=massa)"); ax[0].set_ylim(0, 1)
    ax[1].plot(zr, label="real", lw=2); ax[1].plot(zs, label="sintetico", lw=2)
    ax[1].set_title("Perfil de intensidade por profundidade Z"); ax[1].set_xlabel("slice Z"); ax[1].legend()
    plt.suptitle(f"Conectividade + profundidade - {version}", weight="bold")
    plt.savefig(FIG/f"conectividade_{version}.png", dpi=190, bbox_inches="tight"); plt.show()
    # figura: real vs sintetico, multi-angulo (MIP z/y/x)
    rv = real_frames[0]; sv = synth_vols[0]
    fig, ax = plt.subplots(2, 3, figsize=(16, 10))
    for row, (V, tag) in enumerate([(rv, "REAL"), (sv, "SINTETICO")]):
        for col, (proj, pn) in enumerate([(V.max(0), "MIP-Z"), (V.max(1), "MIP-Y"), (V.max(2), "MIP-X")]):
            ax[row, col].imshow(proj, cmap="magma", aspect="auto"); ax[row, col].set_title(f"{tag} {pn}"); ax[row, col].axis("off")
    plt.suptitle("Real vs Sintetico (multi-angulo)", fontsize=15, weight="bold")
    plt.savefig(FIG/"real_vs_synth_mip.png", dpi=200, bbox_inches="tight"); plt.show()
    # histogramas de intensidade + tamanho de celula
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].hist(rcat, bins=60, alpha=.6, density=True, label="real")      # amostra de TODOS os frames (representativo)
    ax[0].hist(scat, bins=60, alpha=.6, density=True, label="sintetico"); ax[0].set_title("Intensidade (todos os frames)"); ax[0].legend(); ax[0].set_yscale("log")
    ax[1].hist([p.mean() for p in real_patches[:300]], bins=30, alpha=.6, density=True, label="real")
    ax[1].hist([p.mean() for p in synth_patches[:300]], bins=30, alpha=.6, density=True, label="sintetico"); ax[1].set_title("Intensidade media da celula"); ax[1].legend()
    plt.savefig(FIG/"dist_real_vs_synth.png", dpi=200, bbox_inches="tight"); plt.show()
    return metrics

def visualize_rich(real_ref, split="train", version="A_synth"):
    """MUITAS visualizacoes: celula individual, dados detectados (caixas), multi-angulo."""
    import matplotlib.patches as mpatches
    from mpl_toolkits.mplot3d import Axes3D  # noqa
    vtag = "_" + version
    sf = sorted(glob.glob(str(DSDIR/version/split/"*.npz")))[:8]
    synth = [np.load(f) for f in sf]
    rframes, rcens = real_ref["frames"], real_ref["cens"]

    # --- VIZ 1: galeria de CELULAS individuais (real vs sintetico), corte central + MIP ---
    rp = []
    for fr, cs in zip(rframes, rcens): rp += build_template_bank(fr, cs, refine=False)
    spatch = []
    for d in synth:
        ctr = ((d["boxes"][:, :3] + d["boxes"][:, 3:]) / 2)
        spatch += build_template_bank(d["volume"], ctr, refine=False)
    rng = np.random.default_rng(1)
    fig, ax = plt.subplots(4, 10, figsize=(22, 9))
    for col in range(10):
        rc = rp[rng.integers(len(rp))]; sc = spatch[rng.integers(len(spatch))]
        ax[0, col].imshow(rc[rc.shape[0]//2], cmap="magma"); ax[0, col].axis("off")
        ax[1, col].imshow(rc.max(0), cmap="magma"); ax[1, col].axis("off")
        ax[2, col].imshow(sc[sc.shape[0]//2], cmap="magma"); ax[2, col].axis("off")
        ax[3, col].imshow(sc.max(0), cmap="magma"); ax[3, col].axis("off")
    for r, lab in [(0, "REAL corte"), (1, "REAL MIP"), (2, "SINT corte"), (3, "SINT MIP")]:
        ax[r, 0].set_ylabel(lab, rotation=0, ha="right", va="center", fontsize=11)
    plt.suptitle(f"Galeria de celulas [{version}]: REAL vs SINTETICO (corte central + MIP)", fontsize=15, weight="bold")
    plt.savefig(FIG/f"viz1_galeria_celulas{vtag}.png", dpi=180, bbox_inches="tight"); plt.show()

    # --- VIZ 2: DADOS DETECTADOS - caixas sobre o volume (o que o 3D-DETR ve) ---
    fig, ax = plt.subplots(2, 4, figsize=(22, 11)); ax = ax.ravel()
    for k in range(min(8, len(synth))):
        d = synth[k]; vol = d["volume"]; boxes = d["boxes"]
        ax[k].imshow(vol.max(0), cmap="gray")
        for b in boxes:
            ax[k].add_patch(mpatches.Rectangle((b[2], b[1]), b[5]-b[2], b[4]-b[1], fill=False, edgecolor="lime", lw=0.6))
        ax[k].set_title(f"sintetico #{k}: {len(boxes)} celulas rotuladas"); ax[k].axis("off")
    plt.suptitle("Dados DETECTAVEIS: caixas 3D rotuladas (MIP-Z) - entrada do 3D-DETR", fontsize=15, weight="bold")
    plt.savefig(FIG/f"viz2_deteccoes_caixas{vtag}.png", dpi=200, bbox_inches="tight"); plt.show()

    # --- VIZ 3: MULTI-ANGULO (real e sintetico): MIP z/y/x + 3 cortes ortogonais ---
    for tag, V, C in [("REAL", rframes[0], rcens[0]), ("SINTETICO", synth[0]["volume"],
                       (synth[0]["boxes"][:, :3] + synth[0]["boxes"][:, 3:]) / 2)]:
        fig, ax = plt.subplots(2, 3, figsize=(17, 11))
        for c, (proj, pn) in enumerate([(V.max(0), "MIP-Z"), (V.max(1), "MIP-Y"), (V.max(2), "MIP-X")]):
            ax[0, c].imshow(proj, cmap="magma", aspect="auto"); ax[0, c].set_title(f"{tag} {pn}"); ax[0, c].axis("off")
        zc, yc, xc = np.array(V.shape) // 2
        for c, (sl, pn) in enumerate([(V[zc], f"corte Z={zc}"), (V[:, yc], f"corte Y={yc}"), (V[:, :, xc], f"corte X={xc}")]):
            ax[1, c].imshow(sl, cmap="magma", aspect="auto"); ax[1, c].set_title(pn); ax[1, c].axis("off")
        plt.suptitle(f"{tag} - multi-angulo (projecoes + cortes ortogonais)", fontsize=15, weight="bold")
        plt.savefig(FIG/f"viz3_multiangulo_{tag}{vtag}.png", dpi=190, bbox_inches="tight"); plt.show()

    # --- VIZ 4: nuvem 3D dos centroides detectaveis (varios angulos de camera) ---
    d = synth[0]; ctr = (d["boxes"][:, :3] + d["boxes"][:, 3:]) / 2
    fig = plt.figure(figsize=(20, 6))
    for i, (az, el) in enumerate([(30, 20), (120, 30), (210, 15), (300, 45)]):
        a = fig.add_subplot(1, 4, i+1, projection="3d")
        a.scatter(ctr[:, 2]*EFF_VOXEL[2], ctr[:, 1]*EFF_VOXEL[1], ctr[:, 0]*EFF_VOXEL[0], s=14, c=ctr[:, 0], cmap="viridis")
        a.view_init(elev=el, azim=az); a.set_title(f"azim={az} elev={el}"); a.set_xlabel("X(um)"); a.set_ylabel("Y(um)")
    plt.suptitle("Centroides 3D detectaveis (varios angulos de camera)", fontsize=15, weight="bold")
    plt.savefig(FIG/f"viz4_3d_angulos{vtag}.png", dpi=170, bbox_inches="tight"); plt.show()

    # --- VIZ 5: varredura de cortes em Z com caixas (dados detectaveis por profundidade) ---
    d = synth[1] if len(synth) > 1 else synth[0]; vol = d["volume"]; boxes = d["boxes"]
    zs = np.linspace(vol.shape[0]*0.2, vol.shape[0]*0.8, 6).astype(int)
    fig, ax = plt.subplots(1, 6, figsize=(24, 4.2))
    for c, z in enumerate(zs):
        ax[c].imshow(vol[z], cmap="gray"); ax[c].set_title(f"Z={z}"); ax[c].axis("off")
        for b in boxes:
            if b[0] <= z < b[3]:
                ax[c].add_patch(mpatches.Rectangle((b[2], b[1]), b[5]-b[2], b[4]-b[1], fill=False, edgecolor="cyan", lw=0.7))
    plt.suptitle("Varredura em Z com caixas (celulas detectaveis por profundidade)", fontsize=14, weight="bold")
    plt.savefig(FIG/f"viz5_cortes_z_caixas{vtag}.png", dpi=190, bbox_inches="tight"); plt.show()
    print("visualizacoes ricas salvas:", len(list(FIG.glob("viz*.png"))), "figuras")

def sweep_capture(train_dir, videos):
    """ESTUDO: raio de captura (2..5) x esferico/cubico -> qual e' mais similar ao real.
    Grounded no EDA (diametro celular ~9.2um -> raio ~2.8 voxels pooled)."""
    global CELL_R_VOX, SPHERICAL
    frames, cens = [], []
    for name in videos:
        zp = train_dir / f"{name}.zarr"; gp = geff_for(train_dir, name)
        if gp is None: continue
        try: nodes = read_geff_nodes(gp)
        except Exception: continue
        for t in sorted(nodes)[:: max(1, len(nodes)//3)][:3]:
            frames.append(pool_xy(norm(read_frame(zp, t)[0]))); cens.append(nodes[t] / np.array(DOWNSAMPLE))
    rcat = np.concatenate([f.ravel() for f in frames])[:200000]
    r_real = float((rcat > 0.4).mean())
    rng = np.random.default_rng(0); rows = []
    r0, s0 = CELL_R_VOX, SPHERICAL
    print(f"EDA: raio esperado ~2.8 vox (diam 9.2um). real bright_frac={r_real:.3f}")
    for r in [2, 3, 4, 5]:
        for sph in [True, False]:
            CELL_R_VOX, SPHERICAL = r, sph
            templates = []
            for fr, cs in zip(frames, cens): templates += build_template_bank(fr, cs)
            if len(templates) < 5: continue
            bg = liquid_model(frames, cens, cell_r=r)
            vols, rp, sp = [], [], []
            for _ in range(10):
                v, _, ctr = gen_volume(frames[0].shape, bg, templates, N_EST_PER_FRAME, rng)
                vols.append(v); sp += build_template_bank(v, ctr, refine=False)
            for fr, cs in zip(frames, cens): rp += build_template_bank(fr, cs, refine=False)
            scat = np.concatenate([v.ravel() for v in vols])[:200000]
            A, B = embed_set(rp[:300]), embed_set(sp[:300])
            fd = frechet_distance(A, B) if len(A) > 4 and len(B) > 4 else 9
            ks = ks_stat(rcat, scat); bf = abs(r_real - float((scat > 0.4).mean()))
            spec = spectral_dist(frames[0].max(0), vols[0].max(0))
            score = fd + ks + 0.3 * spec + bf     # rebalanceado: Frechet(celula)+KS+espectral, sem vies
            rows.append((r, "esf" if sph else "cubo", fd, ks, bf, spec, score))
    CELL_R_VOX, SPHERICAL = r0, s0
    rows.sort(key=lambda x: x[-1])
    print("\n  raio forma | Frechet   KS    d-bright spectral | SCORE (menor=melhor)")
    for r, sh, fd, ks, bf, sc_, score in rows:
        print(f"   {r:>2}  {sh:4s} | {fd:6.4f}  {ks:5.3f}  {bf:6.4f}   {sc_:6.3f}  | {score:.4f}")
    best = rows[0]
    print(f"\n  >>> MELHOR CAPTURA: raio={best[0]} forma={best[1]} (score {best[-1]:.4f})")
    return best[0], best[1] == "esf"

def temporal_study(train_dir, videos):
    """Como as celulas mudam AO LONGO DO TEMPO (inicio/meio/fim do desenvolvimento)."""
    binkeys = ["inicio", "meio", "fim"]; bins = {k: [] for k in binkeys}
    for name in videos:
        zp = train_dir / f"{name}.zarr"; gp = geff_for(train_dir, name)
        if gp is None: continue
        try: nodes = read_geff_nodes(gp)
        except Exception: continue
        ts = sorted(nodes); T = (ts[-1] + 1) if ts else 100
        for t in ts[:: max(1, len(ts) // 9)][:9]:
            b = 0 if t < T / 3 else (1 if t < 2 * T / 3 else 2)
            fr = pool_xy(norm(read_frame(zp, t)[0])); cens = nodes[t] / np.array(DOWNSAMPLE)
            sh = np.array(fr.shape) - 1
            ci = np.mean([fr[tuple(np.round(c).astype(int).clip(0, sh))] for c in cens]) if len(cens) else 0.0
            bins[binkeys[b]].append((len(cens), float((fr > 0.4).mean()), float(ci)))
    print("=== ESTUDO TEMPORAL (celulas ao longo do desenvolvimento) ===")
    print(f"  {'periodo':8s} {'cels/frame':>11s} {'tecido%':>9s} {'intensidade':>12s}")
    rows = []
    for k in binkeys:
        v = np.array(bins[k]) if bins[k] else np.zeros((1, 3))
        print(f"  {k:8s} {v[:,0].mean():11.1f} {v[:,1].mean()*100:8.1f}% {v[:,2].mean():12.3f}")
        rows.append((k, float(v[:, 0].mean()), float(v[:, 1].mean()), float(v[:, 2].mean())))
    fig, ax = plt.subplots(1, 3, figsize=(15, 4)); x = range(len(rows))
    for a, idx, tit, col in [(ax[0], 1, "Celulas/frame (rotuladas)", "steelblue"),
                             (ax[1], 2, "Extensao do tecido", "orange"), (ax[2], 3, "Intensidade da celula", "green")]:
        a.bar(x, [r[idx] for r in rows], color=col); a.set_xticks(list(x)); a.set_xticklabels([r[0] for r in rows]); a.set_title(tit)
    plt.suptitle("Evolucao das celulas ao longo do desenvolvimento (REAL)", weight="bold")
    plt.savefig(FIG / "temporal_estudo.png", dpi=180, bbox_inches="tight"); plt.show()
    json.dump(rows, open(OUT / "temporal.json", "w"), default=float)
    return rows

## [D] SHOWCASE — documentacao visual do gerador
Tres saidas ricas: **(1)** o PIPELINE passo-a-passo (como um volume e' construido),
**(2)** GALERIA de celulas real vs sintetica, **(3)** RELATORIO de testes de hipotese
(8 distribuicoes: KS + energy distance + Anderson-Darling, com veredito por distribuicao).

In [ ]:
def showcase_pipeline_steps(bg, stats, templates, rng):
    """VISUALIZA o pipeline de geracao PASSO A PASSO (MIP-Z de cada estagio) -> documenta COMO o
    gerador constroi um volume 100% rotulado a partir do dado real. Modo hibrido (celula template real)."""
    fields = bg.get("fields"); _fi = int(rng.integers(len(fields)))
    field = fields[_fi].copy(); Z, Y, X = field.shape
    stages = [("1. Campo real (tecido)", field.copy())]
    fn = field / max(float(field.max()), 1e-6)
    res = bg.get("residuals")
    vol = field + (res[_fi] if res else rng.normal(0, bg.get("noise_std", bg["std"]), field.shape).astype(np.float32) * (0.15 + 0.85 * fn))
    stages.append(("2. + textura/ruido real", vol.copy()))
    lo, hi = np.percentile(field, 50), np.percentile(field, 99)
    vol = vol + 0.55 * np.clip((field - lo) / (hi - lo + 1e-6), 0, 1)
    stages.append(("3. + matriz de tecido", vol.copy()))
    # celulas de template real nas posicoes por densidade
    n_cells = field_n_cells(field, bg.get('n_cells_real', N_EST_PER_FRAME), bg)
    coords = place_clustered(field, n_cells, rng, margin=6, gt_med=bg.get('gt_field_med'), gt_std=bg.get('gt_field_std'), light_bins=bg.get('light_bins'), light_pdf=bg.get('light_pdf'))
    boxes = []
    for (zc, yc, xc) in coords[:n_cells]:
        cell = augment_template(templates[rng.integers(len(templates))], rng) if templates else gen_param_cell(rng, stats)
        rz, ry, rx = np.array(cell.shape) // 2
        if zc-rz < 0 or yc-ry < 0 or xc-rx < 0 or zc+rz+1 > Z or yc+ry+1 > Y or xc+rx+1 > X: continue
        vol[zc-rz:zc+rz+1, yc-ry:yc+ry+1, xc-rx:xc+rx+1] = np.maximum(vol[zc-rz:zc+rz+1, yc-ry:yc+ry+1, xc-rx:xc+rx+1], cell)
        boxes.append([zc-CELL_R_VOX, yc-CELL_R_VOX, xc-CELL_R_VOX, zc+CELL_R_VOX+1, yc+CELL_R_VOX+1, xc+CELL_R_VOX+1])
    stages.append((f"4. + {len(boxes)} celulas (template real)", vol.copy()))
    vol = gaussian_filter(vol, (BLUR_SIGMA*PSF_Z_RATIO, BLUR_SIGMA, BLUR_SIGMA))
    stages.append(("5. + PSF anisotropica (blur z>xy)", vol.copy()))
    m = np.percentile(vol, 99.6); vol = np.clip(vol / m, 0, 1) if m > 0 else vol
    ref = bg.get("hm_ref_canon")
    if ref is not None:
        try:
            from skimage.exposure import match_histograms
            vol = np.clip(0.05 * vol + 0.95 * match_histograms(vol, ref).astype(np.float32), 0, 1)
        except Exception: pass
    stages.append(("6. + histogram matching (real)", vol.copy()))
    vol = poisson_gaussian_noise(apply_z_profile(vol, field_zprof(field)), rng)
    stages.append(("7. + perfil Z + ruido camera", vol.copy()))
    fig, ax = plt.subplots(2, 4, figsize=(18, 9))
    for i, (title, v) in enumerate(stages):
        a = ax[i // 4, i % 4]; a.imshow(np.clip(v, 0, 1).max(0), cmap="magma"); a.set_title(title, fontsize=10); a.axis("off")
    # painel 8: final com caixas GT
    a = ax[1, 3]; a.imshow(np.clip(vol, 0, 1).max(0), cmap="magma")
    for b in boxes[:120]:
        a.add_patch(plt.Rectangle((b[2], b[1]), b[5]-b[2], b[4]-b[1], fill=False, ec="lime", lw=0.5))
    a.set_title(f"8. FINAL 100% rotulado ({len(boxes)} caixas GT)", fontsize=10); a.axis("off")
    plt.suptitle("PIPELINE DE GERACAO passo-a-passo (MIP-Z) — do campo real ao volume rotulado", weight="bold", fontsize=14)
    plt.tight_layout(); plt.savefig(FIG / "showcase_pipeline.png", dpi=140, bbox_inches="tight"); plt.show()
    print(f"  showcase pipeline -> {FIG/'showcase_pipeline.png'}")

def showcase_stats_report(real_frames, real_cens, synth_dir, version="D_param", split="train"):
    """RELATORIO de testes de hipotese: 8 distribuicoes real vs sintetico com KS + energy + Anderson-Darling
    e veredito. Documenta QUANTITATIVAMENTE o quao perto o sintetico esta do real."""
    from scipy.stats import ks_2samp, energy_distance
    try: from scipy.stats import anderson_ksamp
    except Exception: anderson_ksamp = None
    def dists(frames, tmpl, dets):
        D = {}
        D['voxel_int'] = np.concatenate([f.ravel()[::9] for f in frames]).astype(np.float32)
        D['cell_mean'] = np.array([float(t[t > 0.3*t.max()].mean()) for t in tmpl if t.max() > 0], np.float32)
        D['cell_peak'] = np.array([float(t.max()) for t in tmpl if t.max() > 0], np.float32)
        D['cell_size'] = np.array([float((t > 0.5*t.max()).sum()) for t in tmpl if t.max() > 0], np.float32)
        nn = [nn_um(d) for d in dets if len(d) > 1]; D['nn_um'] = np.concatenate(nn).astype(np.float32) if nn else np.zeros(2, np.float32)
        D['grad_mag'] = np.concatenate([np.sqrt(sum(g**2 for g in np.gradient(f))).ravel()[::15] for f in frames]).astype(np.float32)
        D['contrast'] = np.array([float(t.max() - t[t > 0].mean()) for t in tmpl if t.max() > 0], np.float32)
        return D
    real_t = []
    for fr, cs in zip(real_frames, real_cens): real_t += build_template_bank(fr, cs)
    real_d = [dog_detect_precise(f)[0] for f in real_frames]
    R = dists(real_frames, real_t, real_d)
    sv, st, sd = [], [], []
    for f in sorted(glob.glob(str(DSDIR/version/split/"*.npz")))[:40]:
        dnpz = np.load(f); sv.append(dnpz["volume"])
        cen = (dnpz["boxes"][:, :3] + dnpz["boxes"][:, 3:]) / 2
        st += build_template_bank(dnpz["volume"], cen); sd.append(dog_detect_precise(dnpz["volume"])[0])
    if len(sv) < 3: print("  showcase_stats: sem synth suficiente"); return
    S = dists(sv, st, sd); dk = list(R)
    print("\n  === RELATORIO DE TESTES DE HIPOTESE (real vs sintetico) ===")
    print(f"  {'distribuicao':12s} {'KS':>6} {'p':>7} {'energy':>8} {'AD_p':>6}  veredito")
    rows = {}
    for k in dk:
        a, b = np.asarray(R[k])[:6000], np.asarray(S.get(k, []))[:6000]
        if len(b) < 5: continue
        ks = ks_2samp(a, b); adp = float('nan')
        if anderson_ksamp is not None:
            try:
                import warnings
                with warnings.catch_warnings(): warnings.simplefilter("ignore"); adp = float(anderson_ksamp([a[:1500], b[:1500]]).significance_level)
            except Exception: pass
        ver = "OK (casa)" if ks.statistic < 0.25 else ("razoavel" if ks.statistic < 0.4 else "PISO estrutural")
        rows[k] = (ks.statistic, ks.pvalue, float(energy_distance(a, b)), adp, ver, a, b)
        print(f"  {k:12s} {ks.statistic:6.3f} {ks.pvalue:7.4f} {rows[k][2]:8.4f} {adp:6.3f}  {ver}")
    npass = sum(1 for k in rows if rows[k][0] < 0.25)
    print(f"  >> {npass}/{len(rows)} distribuicoes com bom match (KS<0.25)")
    fig, ax = plt.subplots(2, 4, figsize=(18, 9))
    for i, k in enumerate(dk):
        a2 = ax[i // 4, i % 4]
        if k not in rows: a2.axis("off"); continue
        ks_s, ks_p, en, adp, ver, a, b = rows[k]
        rng_ = (float(min(a.min(), b.min())), float(max(a.max(), b.max())))
        a2.hist(a, bins=40, range=rng_, density=True, alpha=.55, color="steelblue", label="real")
        a2.hist(b, bins=40, range=rng_, density=True, alpha=.55, color="darkorange", label="synth")
        col = "green" if ks_s < 0.25 else ("orange" if ks_s < 0.4 else "red")
        a2.set_title(f"{k}\nKS={ks_s:.3f} energy={en:.3f} [{ver}]", fontsize=9, color=col)
        a2.legend(fontsize=7); a2.tick_params(labelsize=7)
    plt.suptitle(f"RELATORIO DE TESTES: sintetico ({version}) vs REAL — {npass}/{len(rows)} distribuicoes casam (KS<0.25)", weight="bold", fontsize=14)
    plt.tight_layout(); plt.savefig(FIG / f"showcase_testes_{version}.png", dpi=140, bbox_inches="tight"); plt.show()
    print(f"  showcase testes -> {FIG/f'showcase_testes_{version}.png'}")

def main():
    td_ = find_train_dir(); assert td_, "train dir nao encontrado"
    if os.environ.get("SYNTH_TEMPORAL", "1") != "0":
        _tv = sorted(p.stem for p in td_.glob("*.zarr"))
        _tb = {}
        for _v in _tv: _tb.setdefault(_v.split("_")[0], []).append(_v)
        temporal_study(td_, sum([vs[:5] for vs in _tb.values()], []))
    if os.environ.get("SYNTH_SWEEP", "1") != "0":
        global CELL_R_VOX, SPHERICAL
        print("=== ESTUDO DE CAPTURA (tamanho x forma) ===")
        _allv = sorted(p.stem for p in td_.glob("*.zarr")); _byE = {}
        for _v in _allv: _byE.setdefault(_v.split("_")[0], []).append(_v)
        vids0 = sum([vs[:6] for vs in _byE.values()], [])   # representativo: 6 de cada embriao
        sweep_capture(td_, vids0)   # ESTUDO (documentacao); metricas de tamanho sao ruidosas
        # DECISAO grounded no EDA (diam 9.2um -> raio 3) + estudo (esferico > cubico); raio=4 oversizeia caixas
        CELL_R_VOX, SPHERICAL = 3, True
        print(f"\nCaptura FINAL: raio=3 ESFERICA (EDA + estudo confirmou esferico>cubico)\n")
    import random
    vids = sorted(p.stem for p in td_.glob("*.zarr"))
    by_emb = {}
    for v in vids: by_emb.setdefault(v.split("_")[0], []).append(v)
    rng0 = random.Random(1234); tr, va, te = [], [], []
    for emb, vs in by_emb.items():
        vs = sorted(vs); rng0.shuffle(vs); n = len(vs)
        ntr, nva = int(n*0.7), int(n*0.15)
        tr += vs[:ntr]; va += vs[ntr:ntr+nva]; te += vs[ntr+nva:]
    print(f"split por video (sem leakage): train={len(tr)} val={len(va)} test={len(te)}")
    json.dump({"train": tr, "val": va, "test": te}, open(OUT/"synth_split.json", "w"))
    rng = np.random.default_rng(0)
    if os.environ.get("SYNTH_SCALE", "0") == "1":                 # MODO ESCALA: gera em massa, sem figuras
        run_scale(td_, tr, rng); return
    caps = {"train": N_SAMPLES, "val": N_SAMPLES//4, "test": N_SAMPLES//4}
    # SWEEP DE DESFOQUE (PSF): mede similaridade de textura vs sigma -> fixa o BLUR_SIGMA ideal p/ tudo
    NV = int(os.environ.get("SYNTH_REFVIDEOS", str(len(tr))))     # BASE INTEIRA: todos os videos do treino como referencia
    global DIV_RATE
    DIV_RATE = measure_division_rate(td_, tr[:min(NV, len(tr))])  # TAXA DE GERACAO real (divisoes) -> aplica no sintetico
    MOTION = measure_motion(td_, tr[:min(NV, len(tr))])          # MOVIMENTACAO real (velocidade + persistencia direcional)
    # ALVO DE INTENSIDADE/CELULA DATA-DRIVEN: casa a media+sigma REAIS (nao um 0.30 fixo abaixo do real).
    global _CELL_MU_RT, _CELL_SIG_RT, _CELL_MEANS_RT
    _t0, _, _ = collect_real(td_, tr[:8])
    if len(_t0) >= 5:
        _mn = np.asarray(measure_cell_stats(_t0)["means"], float)
        _CELL_MU_RT = float(np.clip(np.median(_mn), 0.1, 0.9)); _CELL_SIG_RT = float(np.clip(np.std(_mn), 0.03, 0.20)); _CELL_MEANS_RT = _mn
        print(f"ALVO intensidade/celula (regiao ativa) DATA-DRIVEN do REAL: media={_CELL_MU_RT:.3f} sigma={_CELL_SIG_RT:.3f}  "
              f"(=> full-patch ~{_CELL_MU_RT*0.55:.2f}, o '0.30' do user)")
    if os.environ.get("SYNTH_CELLMEAN"):                          # user pode FORCAR valor literal via env
        _CELL_MU_RT = CELL_MEAN_TARGET; print(f"  (SYNTH_CELLMEAN setado -> forcando literal {_CELL_MU_RT})")
    if os.environ.get("SYNTH_BLURSWEEP", "1") != "0":
        _t, _f, _c = collect_real(td_, tr[:12])
        if len(_t) >= 5:
            _st = measure_cell_stats(_t); study_cell_development(_st)
            _bg = liquid_model(_f, _c); blur_sweep(_f, _bg, _st, rng); gallery_stages(_st, rng)
            visualize_motion(_bg, _st, rng, MOTION)             # trajetorias realistas + verificacao de colisao
    real_ref = None
    for split, vs in [("train", tr[:NV]), ("val", va[:max(8, NV//5)]), ("test", te[:max(8, NV//5)])]:
        print(f"gerando {split}...")
        r = run_split(td_, split, vs, rng, caps[split])
        if split == "train" and r: real_ref = r
    print("=== VALIDACAO: A_synth (TEMPLATE) vs D_param (PARAMETRICO) ===")
    results = {}
    for ver in ["A_synth", "D_param", "E_layout"]:
        m = validate(real_ref["frames"], real_ref["cens"], DSDIR, "train", version=ver)
        results[ver] = m
        print(f"  --- {ver} ---")
        for k, v in m.items(): print(f"    {k}: {round(v,5) if isinstance(v,float) else v}")
        if m:
            print(f"    >> AGRUPAMENTO  NN real={m.get('nn_real_med_um',0):.1f}um synth={m.get('nn_synth_med_um',0):.1f}um | "
                  f"aglom real={m.get('clump_real',0):.1f} synth={m.get('clump_synth',0):.1f}")
            print(f"    >> CONECTIVIDADE (massa vs teia)  real={m.get('connected_frac_real',0):.2f} "
                  f"synth={m.get('connected_frac_synth',0):.2f}  (alvo: igualar o real)")
            print(f"    >> ESPACO 3D  Frechet3d={m.get('frechet3d',0):.4f} | celulas synth >=70% similares a uma REAL: "
                  f"{m.get('sim3d_ge70',0)*100:.1f}% (mediana {m.get('sim3d_median',0):.2f}, min {m.get('sim3d_min',0):.2f}) | blur sigma={m.get('blur_sigma',0):.2f}")
            print(f"    >> SOBREPOSICAO IoU>0.3  synth={m.get('overlap_frac_synth',0)*100:.1f}% "
                  f"real={m.get('overlap_frac_real',0)*100:.1f}% | maxIoU synth={m.get('overlap_maxiou_synth',0):.2f} "
                  f"real={m.get('overlap_maxiou_real',0):.2f}  (alvo: IGUALAR o real, nao zerar)")
    json.dump(results, open(OUT/"similaridade.json", "w"), indent=2, default=float)
    print("=== VISUALIZACOES RICAS ===")
    if real_ref:
        visualize_rich(real_ref, "train", "A_synth")
        visualize_rich(real_ref, "train", "D_param")
        print("=== [D] SHOWCASE (pipeline passo-a-passo + relatorio de testes de hipotese HELD-OUT) ===")
        try: showcase_pipeline_steps(real_ref["bg"], real_ref.get("stats") or measure_cell_stats(real_ref["templates"]), real_ref["templates"], np.random.default_rng(7))
        except Exception as ex: print("  showcase_pipeline erro:", ex)
        try:
            # ANTI-LEAKAGE: valida o synth (gerado do TRAIN) contra frames REAIS de TEST (held-out, disjunto)
            _htv = te[:max(6, NV//5)] if te else va[:6]
            print(f"  [ANTI-LEAKAGE] validacao held-out contra TEST={_htv[:3]}...({len(_htv)}) - disjunto do train que gerou")
            _hf, _hc = [], []
            for _nm in _htv:
                _zp = td_/f"{_nm}.zarr"; _gp = geff_for(td_, _nm)
                if _gp is None: continue
                try: _nodes = read_geff_nodes(_gp)
                except Exception: continue
                for _tt in sorted(_nodes)[::max(1, len(_nodes)//6)][:6]:
                    _hf.append(pool_xy(norm(read_frame(_zp, _tt)[0]))); _hc.append(_nodes[_tt]/np.array(DOWNSAMPLE))
            if len(_hf) >= 3: showcase_stats_report(_hf, _hc, DSDIR, version="D_param")
            else: showcase_stats_report(real_ref["frames"], real_ref["cens"], DSDIR, version="D_param")
        except Exception as ex: print("  showcase_stats erro:", ex)
    n_files = len(glob.glob(str(DSDIR/"**"/"*.npz"), recursive=True))
    print(f"\nDataset gerado: {n_files} arquivos .npz em {DSDIR} (versoes A_synth/B_real/C_real_dog)")

if not SMOKE and os.environ.get("SYNTH_GRIDSEARCH", "0") != "1":   # gridsearch importa as funcoes sem rodar main
    main()

In [ ]:
if SMOKE:
    rng = np.random.default_rng(0)
    Z, Y, X = 64, 64, 64
    real = (rng.random((Z, Y, X)) * 0.15).astype(np.float32)
    cens = rng.integers([5, 5, 5], [Z-5, Y-5, X-5], size=(40, 3))
    zz, yy, xx = np.indices((Z, Y, X))
    for c in cens: real += (0.8*np.exp(-(((zz-c[0])**2+(yy-c[1])**2+(xx-c[2])**2)/(2*2.2**2)))).astype(np.float32)
    real = np.clip(real, 0, 1)
    tb = build_template_bank(real, cens); bg = liquid_model([real], [cens])
    vol, boxes, ctr = gen_volume((Z, Y, X), bg, tb, 40, rng)
    A = embed_set(build_template_bank(real, cens)); B = embed_set(build_template_bank(vol, ctr))
    print("SMOKE: templates", len(tb), "| boxes", len(boxes),
          "| Frechet", round(frechet_distance(A, B), 4), "| MMD", round(mmd_rbf(A, B), 5))
    print("SMOKE OK")

# GERADOR NATIVO: gera na resolucao do MICROSCOPIO (256x256) e DEPOIS poola (como o real)

**A virada (pedido do user):** ate agora o gerador criava direto na resolucao POOLED (64^3). Mas o
real e' capturado NATIVO (Z,256,256) e so' entao o pipeline poola 4x em XY -> (64,64,64) p/ o DoG.
Gerar direto no pooled NUNCA modela a transicao nativa->pooled (o borrao que o pooling cria).

**Aqui:** geramos NATIVO -- fundo escuro (liquido) + esferas FISICAS (nitidas, preenchidas, borda
dura) + PSF optica anisotropica + ruido de camera -- e entao aplicamos o MESMO pool_xy 4x do real.
O pooled resultante tem a MESMA assinatura nativa->pooled do real (a sonda mostrou que na nativa a
celula e' um disco nitido e no pooled vira um borrao de ~5px; agora isso emerge do pooling, nao e'
imposto). Voxel fisico: z=1.625, x=y=0.40625 um -> a esfera fisica e' ANISOTROPICA em voxels nativos
(achatada em z, larga em xy); o pool 4x em xy a torna ~isotropica, como o real.

In [ ]:
import os, json
import numpy as np
from scipy.ndimage import gaussian_filter

# voxel fisico NATIVO (antes do pool)
VOXEL_NATIVE = np.array([1.625, 0.40625, 0.40625])
POOL = 4                                                # XY downsample do pipeline (DOWNSAMPLE=(1,4,4))


def _sphere_native(r_um, rng, sharp=7, ellip=0.5, frac=None):
    """Esfera FISICA de raio r_um, amostrada na grade NATIVA anisotropica (achatada em z). Perfil
    super-gaussiano (disco preenchido, borda nitida) + variacao eliptica.
    `frac` = deslocamento SUB-VOXEL (p - round(p)). Sem ele a celula so' pode ficar centrada num voxel
    INTEIRO e o rotulo de centroide sai quantizado (erro sistematico de ate 0.5 voxel nativo, que o
    detector aprende como ruido no alvo). Com ele o perfil e' amostrado deslocado e o centroide
    verdadeiro (float) e' o rotulo."""
    axes_vox = r_um / VOXEL_NATIVE                       # semi-eixos em VOXELS nativos (z pequeno, xy grande)
    aniso = rng.uniform(0.82, 1.20, 3)
    if rng.random() < ellip: aniso[rng.integers(3)] *= rng.uniform(1.4, 2.0)   # eliptica marcada
    axes = np.maximum(axes_vox * aniso, 0.6)
    size = (np.ceil(axes) * 2 + 3).astype(int)          # bounding box por eixo
    zz, yy, xx = np.indices(tuple(size)); c = size // 2
    fz, fy, fx = (0.0, 0.0, 0.0) if frac is None else (float(frac[0]), float(frac[1]), float(frac[2]))
    # rotacao so' no plano XY (a anisotropia z e' fisica, nao rotaciona junto)
    th = rng.uniform(0, np.pi); ct, stt = np.cos(th), np.sin(th)
    dy = (yy - c[1] - fy); dx = (xx - c[2] - fx); dz = (zz - c[0] - fz)
    ry = ct*dy - stt*dx; rx = stt*dy + ct*dx
    rn = np.sqrt((dz/axes[0])**2 + (ry/axes[1])**2 + (rx/axes[2])**2)
    return np.exp(-(rn**sharp)).astype(np.float32)


# ============================================================================================
# CASCA DO EMBRIAO (algoritmo NOVO) -- colocacao GERATIVA com a estrutura espacial do real.
#
# PROBLEMA que ele resolve: o processo de Thomas (clusters aleatorios) espalha grumos pelo volume
# inteiro; o real tem uma FAIXA CURVA de tecido com VAZIO ESCURO de um lado (a blastoderme sobre o
# vitelo atravessando o campo). Copiar as posicoes reais (`cens_native`) casa a distribuicao por
# construcao mas NAO e' gerativo -- reusa os poucos frames reais.
# SOLUCAO: ajustar a GEOMETRIA (nao as posicoes) da casca a partir do real -- centro/raio de uma
# esfera + espessura -- e depois AMOSTRAR cascas novas. Sai estrutura coerente e inedita.
#
# Ajuste algebrico de esfera (Pratt/Coope): p/ pontos p_i, |p_i|^2 = 2 p_i . c + (R^2 - |c|^2),
# linear em [c, k] com k = R^2-|c|^2 -> minimos quadrados. R = sqrt(k + |c|^2).
# LIMITE PLANO: embriao grande => R >> campo de visao e a esfera vira um PLANO; o ajuste fica mal
# condicionado, entao caimos p/ PCA (normal = autovetor de menor variancia) -- e' o caso R->inf.
# ============================================================================================

def fit_shell(pts_vox, voxel=None):
    """Ajusta a casca (esfera OU plano) a pontos em voxels NATIVOS. Retorna dict com geometria em um.
    `pts_vox` = deteccoes/centroides reais (N,3) em (z,y,x) voxel nativo."""
    voxel = VOXEL_NATIVE if voxel is None else np.asarray(voxel, float)
    P = np.atleast_2d(np.asarray(pts_vox, float)) * voxel                    # -> um (isotropico)
    n = len(P)
    if n < 12: return dict(kind="none", n=n)
    c0 = P.mean(0); Q = P - c0
    # --- tentativa ESFERA ---
    A = np.concatenate([2.0*Q, np.ones((n, 1))], 1)
    b = (Q**2).sum(1)
    try:
        sol, *_ = np.linalg.lstsq(A, b, rcond=None)
        cc = sol[:3]; k = sol[3]
        R2 = k + float(cc @ cc)
        R = float(np.sqrt(R2)) if R2 > 0 else -1.0
        center = c0 + cc
        rad = np.linalg.norm(P - center, axis=1)
        resid = float(np.std(rad - R)) if R > 0 else np.inf
    except Exception:
        R, resid, center = -1.0, np.inf, c0
    # --- alternativa PLANO (limite R->inf) via PCA ---
    U, S, Vt = np.linalg.svd(Q - Q.mean(0), full_matrices=False)
    normal = Vt[-1] / (np.linalg.norm(Vt[-1]) + 1e-12)
    off = (P - c0) @ normal
    resid_plane = float(np.std(off))
    # extensao lateral do tecido (p/ saber se ele cobre todo o campo ou so' uma faixa)
    span = np.percentile(P, [2, 98], axis=0)
    # ESCOLHA: esfera so' vence se for bem condicionada (R finito, nao absurdo) E ajustar melhor
    fov = float(np.linalg.norm(P.max(0) - P.min(0)) + 1e-9)
    # PISO FISICO DE RAIO (defeito encontrado NA FIGURA, invisivel nas metricas KS): o ajuste devolvia
    # R=50um num campo de 104um -- esfera MENOR que o campo de visao, que cabe inteira dentro do volume.
    # Celulas numa casca dessas formam uma BOLA OCA e a projecao em Z sai como ANEL com centro escuro.
    # O real NUNCA tem vazio central: a borda dele e' quase reta, ou seja R >> campo. O ajuste estava
    # SUPERAJUSTANDO curvatura local de poucas deteccoes. Agora a esfera so' e' aceita se for
    # fisicamente plausivel (raio >= 1.5x o campo); abaixo disso usamos o PLANO (limite R->inf).
    R_MIN = float(os.environ.get("SYNTH_SHELL_RMIN_FOV", "1.5")) * fov
    sphere_ok = (R > R_MIN) and (R < 60.0 * fov) and np.isfinite(resid) and (resid < 0.92 * resid_plane)
    if sphere_ok:
        return dict(kind="sphere", center_um=center, R_um=R, thick_um=max(resid, 1.0),
                    n=n, resid=resid, resid_plane=resid_plane, span_um=span)
    return dict(kind="plane", point_um=c0, normal=normal, thick_um=max(resid_plane, 1.0),
                n=n, resid=float(resid) if np.isfinite(resid) else -1.0, resid_plane=resid_plane, span_um=span)


def _density_field(rng, shape_zyx, smooth_vox=14.0, contrast=1.0):
    """Campo de densidade suave em [0,1]: regioes mais/menos povoadas ao longo do tecido (o real nao
    e' uniforme dentro da faixa). Ruido branco borrado = campo de baixa frequencia."""
    f = rng.normal(0, 1, shape_zyx).astype(np.float32)
    f = gaussian_filter(f, smooth_vox)
    f -= f.min(); f /= (f.max() + 1e-9)
    return np.clip(0.5 + contrast * (f - 0.5), 0.0, 1.0)


def sample_shell_positions(n_cells, shape_zyx, rng, shell=None, r_lo=2.5, r_hi=4.5, csf=1.0,
                           jitter=0.25, dens_contrast=1.0):
    """Coloca `n_cells` numa CASCA (esfera/plano) atravessando o volume, com gradiente de densidade e
    checagem de colisao. `shell` = saida de fit_shell (geometria medida no real); None -> sorteia uma.
    Retorna (pos_voxel (N,3), radii_um (N,))."""
    Z, Y, X = shape_zyx
    lo = np.array([4.0, 12.0, 12.0]); hi = np.array([Z-4.0, Y-12.0, X-12.0])
    vox = VOXEL_NATIVE
    ext_um = (np.array([Z, Y, X]) * vox)                                     # campo em um
    # ---- geometria: usa a medida OU sorteia uma plausivel (variacao entre embrioes/frames) ----
    if shell is None or shell.get("kind") not in ("sphere", "plane"):
        kind = "sphere" if rng.random() < 0.65 else "plane"
        R = float(rng.uniform(1.2, 4.0) * float(ext_um.max()))               # curvatura suave
        nrm = rng.normal(0, 1, 3); nrm /= np.linalg.norm(nrm) + 1e-9
        cen = ext_um * 0.5 + nrm * R * float(rng.uniform(0.75, 1.05))        # centro FORA do campo
        shell = dict(kind=kind, center_um=cen, R_um=R, point_um=ext_um*rng.uniform(0.35, 0.65, 3),
                     normal=nrm, thick_um=float(rng.uniform(6.0, 16.0)))
    else:
        shell = dict(shell)
        # PERTURBA a geometria medida -> casca NOVA (gerativo), nao a mesma do frame real
        j = float(jitter)
        if shell["kind"] == "sphere":
            shell["R_um"] = float(shell["R_um"] * rng.normal(1.0, 0.10*j/0.25 if j else 0.0)) if j else shell["R_um"]
            shell["center_um"] = np.asarray(shell["center_um"], float) + rng.normal(0, j*12.0, 3)
        else:
            nrm = np.asarray(shell["normal"], float) + rng.normal(0, j*0.18, 3)
            shell["normal"] = nrm / (np.linalg.norm(nrm) + 1e-9)
            shell["point_um"] = np.asarray(shell["point_um"], float) + rng.normal(0, j*10.0, 3)
        # REGIME DE BORDA (defeito visto na figura V01): o real tem DOIS regimes -- campos DENTRO do
        # tecido, que preenchem o quadro, e campos na BORDA do embriao, com uma faixa de celulas e um
        # grande vazio escuro do outro lado. Nossa casca vinha da calibracao (videos de interior), com
        # a normal ao longo de Z -> a laje fica paralela a imagem e SEMPRE cobre tudo na projecao.
        # Para sair a FAIXA, a normal precisa estar NO PLANO da imagem (laje atravessando de lado).
        # Aqui uma fracao dos volumes recebe essa reorientacao, reproduzindo o regime de borda.
        if rng.random() < float(os.environ.get("SYNTH_EDGE_FRAC", "0.35")):
            ang = rng.uniform(0, 2*np.pi)
            n_edge = np.array([rng.uniform(-0.25, 0.25), np.cos(ang), np.sin(ang)], float)
            shell = dict(shell, kind="plane", normal=n_edge/(np.linalg.norm(n_edge)+1e-9),
                         point_um=np.array([Z, Y, X], float)*VOXEL_NATIVE*rng.uniform(0.25, 0.75, 3),
                         thick_um=float(rng.uniform(14.0, 34.0)))
        shell["thick_um"] = float(max(3.0, shell["thick_um"] * rng.normal(1.0, 0.15)))
    thick = float(shell["thick_um"])
    dens = _density_field(rng, (Z, Y, X), smooth_vox=float(os.environ.get("SYNTH_SHELL_DENS_SMOOTH", "14")),
                          contrast=dens_contrast)
    # ---- MAPA DE PROBABILIDADE vetorizado (substitui rejeicao pura, que era fragil e lenta):
    # peso(voxel) = gaussiana(distancia a' superficie) * densidade. Amostrar deste mapa GARANTE que as
    # celulas caiam na casca e cobre o volume todo, sem estourar tentativas.
    gz, gy, gx = np.meshgrid(np.arange(Z, dtype=np.float32), np.arange(Y, dtype=np.float32),
                             np.arange(X, dtype=np.float32), indexing="ij")
    QZ, QY, QX = gz*vox[0], gy*vox[1], gx*vox[2]

    def _dist_map(sh):
        if sh["kind"] == "sphere":
            c = np.asarray(sh["center_um"], float)
            return np.abs(np.sqrt((QZ-c[0])**2 + (QY-c[1])**2 + (QX-c[2])**2) - float(sh["R_um"]))
        p0 = np.asarray(sh["point_um"], float); nn = np.asarray(sh["normal"], float)
        return np.abs((QZ-p0[0])*nn[0] + (QY-p0[1])*nn[1] + (QX-p0[2])*nn[2])

    D = _dist_map(shell)
    # RE-ANCORAGEM: se a casca nao cruza o volume (tudo longe da superficie), ela produziria ZERO
    # celulas. Desloca a geometria p/ que a superficie passe pelo volume, preservando a CURVATURA.
    if float(D.min()) > 1.5 * thick:
        ctr_um = np.array([Z, Y, X], float) * vox * 0.5
        if shell["kind"] == "sphere":
            c = np.asarray(shell["center_um"], float)
            u = (ctr_um - c); nu = np.linalg.norm(u) + 1e-9
            shell["center_um"] = ctr_um - (u/nu) * float(shell["R_um"])   # superficie passa no centro
        else:
            shell["point_um"] = ctr_um
        D = _dist_map(shell)
    W = np.exp(-0.5 * (D / (0.5*thick + 1e-9))**2) * dens
    # zera bordas (celula precisa caber inteira no volume)
    W[: int(lo[0]), :, :] = 0; W[int(hi[0]):, :, :] = 0
    W[:, : int(lo[1]), :] = 0; W[:, int(hi[1]):, :] = 0
    W[:, :, : int(lo[2])] = 0; W[:, :, int(hi[2]):] = 0
    tot = float(W.sum())
    if tot <= 0:                                                          # degenerado -> uniforme
        W = np.zeros_like(W); W[int(lo[0]):int(hi[0]), int(lo[1]):int(hi[1]), int(lo[2]):int(hi[2])] = 1.0
        tot = float(W.sum())
    p = (W.ravel() / tot).astype(np.float64)
    # OVERSAMPLE: sorteia mais candidatos do que o necessario, pois a colisao vai descartar parte
    ncand = int(min(len(p), max(8 * n_cells, 4000)))
    idx = rng.choice(len(p), size=ncand, replace=True, p=p)
    cz, cyx = np.divmod(idx, Y * X); cy, cx = np.divmod(cyx, X)
    cand = np.stack([cz, cy, cx], 1).astype(np.float32)
    cand += rng.uniform(-0.5, 0.5, cand.shape)                            # jitter sub-voxel
    cand = np.clip(cand, lo, hi)
    # aceitacao gulosa com checagem de colisao (mesma regra do resto do gerador)
    cand_r = rng.uniform(r_lo, r_hi, len(cand)).astype(np.float32)
    pos, radii = [], []
    for p_i, r in zip(cand, cand_r):
        if len(pos) >= n_cells: break
        if pos:
            dd = np.linalg.norm((np.array(pos) - p_i) * vox, axis=1)
            if (dd < csf * (r + np.array(radii))).any(): continue
        pos.append(p_i); radii.append(float(r))
    return np.array(pos, np.float32).reshape(-1, 3), np.array(radii, np.float32)


def gen_volume_native(zshape=64, xyshape=256, n_cells=200, rng=None, cens_native=None,
                      medium=None, min_sep_um=None, shell=None):
    """Gera um volume NATIVO (zshape, xyshape, xyshape): fundo escuro + esferas fisicas + PSF + ruido.
    Retorna (vol_native, vol_pooled, centroides_native). cens_native (opcional): posicoes reais."""
    if rng is None: rng = np.random.default_rng(0)
    medium = float(os.environ.get("SYNTH_MEDIUM_LEVEL", "0.04")) if medium is None else medium
    # RAIO calibrado por DETECTABILIDADE (nao a olho): com 2.5-4.5 (diam 7um) o DoG achava so' 0.811
    # das celulas COLOCADAS, enquanto no REAL ele acha 0.91-0.94 -> o sintetico era ~20% mais DIFICIL
    # que a realidade (ensina o detector a esperar celulas menores do que existem). Com 3.5-5.5
    # (diam 9um) o recall vai a 0.913 = casa o real, E bate a EDA (diametro real ~9.5um). Raios maiores
    # (diam 10-11um) dao recall 0.94-0.96, o que ULTRAPASSA o real -- sintetico mais facil que a
    # realidade e' pior p/ treino. A separacao que o user queria e' preservada: NN 11.6um vs real ~10-11.
    r_lo = float(os.environ.get("SYNTH_CELL_RUM_LO", "3.5"))     # raio (um)
    r_hi = float(os.environ.get("SYNTH_CELL_RUM_HI", "5.5"))
    sharp = float(os.environ.get("SYNTH_CELL_SHARPNESS", "4.5"))   # calibrado: brilho relativo pico/media do real
    psf_xy = float(os.environ.get("SYNTH_PSF_XY_UM", "0.55"))    # PSF lateral (um) ~ confocal/light-sheet
    psf_z_ratio = float(os.environ.get("SYNTH_PSFZ", "2.5"))     # PSF axial maior (anisotropia optica real)
    # SEPARACAO CONSCIENTE DE COLISAO: duas celulas so' sao aceitas se a distancia centro-a-centro
    # (um) >= FATOR * (raio_i + raio_j). Assim elas ENCOSTAM mas nao se INTERPENETRAM. Antes o min_sep
    # era FIXO em 7um -> celulas de ~10um de diametro se sobrepunham ~3um, e ao aumentar o tamanho a
    # sobreposicao explodia (celulas se batiam, o DoG as fundia -> vizinho-mais-proximo estragado).
    # fator 1.0 = tangentes (sem overlap); <1 = leve encoste (tecido denso real); >1 = folga.
    csf = float(os.environ.get("SYNTH_COLLISION_FACTOR", "1.0"))

    Z, Y, X = zshape, xyshape, xyshape
    vol = np.clip(rng.normal(medium, medium*0.25, (Z, Y, X)), 0, 1).astype(np.float32)  # LIQUIDO escuro

    # posicoes + RAIOS das celulas. cens reais OU colocacao (AGRUPADA ou uniforme) consciente de tamanho.
    # CLUSTERING (SYNTH_CLUSTER=1, default): o embriao real NAO tem celulas aleatorias uniformes -- elas
    # formam GRUPOS/aglomerados (regioes densas e vazias). Processo de Thomas: sorteia CENTROS de cluster
    # e coloca a maioria das celulas em torno deles (gaussiana), + uma fracao uniforme de fundo. Isso da'
    # a estrutura espacial (tipos de agrupamento) que a colocacao aleatoria uniforme nunca reproduz.
    place_mode = os.environ.get("SYNTH_PLACE_MODE", "cluster")     # cluster (Thomas) | shell (casca) | uniform
    if cens_native is not None and len(cens_native) >= 5:
        pos = np.atleast_2d(cens_native).astype(np.float32)
        radii = rng.uniform(r_lo, r_hi, len(pos)).astype(np.float32)
    elif place_mode == "shell":
        # CASCA: geometria (curvatura/espessura) medida no real, perturbada -> layout coerente e INEDITO
        pos, radii = sample_shell_positions(
            n_cells, (Z, Y, X), rng, shell=shell, r_lo=r_lo, r_hi=r_hi, csf=csf,
            jitter=float(os.environ.get("SYNTH_SHELL_JITTER", "0.25")),
            dens_contrast=float(os.environ.get("SYNTH_SHELL_DENS", "1.0")))
    else:
        clustered = os.environ.get("SYNTH_CLUSTER", "1") == "1"
        ncl = max(3, int(n_cells * float(os.environ.get("SYNTH_CLUSTER_FRAC_N", "0.10"))))   # nº de aglomerados
        cfrac = float(os.environ.get("SYNTH_CLUSTER_FRAC", "0.72"))               # fracao de celulas em cluster
        spread_um = float(os.environ.get("SYNTH_CLUSTER_SPREAD_UM", "14.0"))      # raio do aglomerado (um)
        centers = rng.uniform([6, 16, 16], [Z-6, Y-16, X-16], size=(ncl, 3)).astype(np.float32) if clustered else None
        sp_vox = spread_um / VOXEL_NATIVE                                         # spread por eixo (voxels)
        pos, radii = [], []; tries = 0
        while len(pos) < n_cells and tries < n_cells * 60:
            tries += 1
            if clustered and rng.random() < cfrac:                               # celula DENTRO de um aglomerado
                cc = centers[rng.integers(ncl)]
                p = cc + rng.normal(0, sp_vox, 3).astype(np.float32)
                p = np.clip(p, [4, 12, 12], [Z-4, Y-12, X-12])
            else:                                                                # celula de FUNDO (uniforme)
                p = rng.uniform([4, 12, 12], [Z-4, Y-12, X-12])
            r = float(rng.uniform(r_lo, r_hi))
            if pos:
                d = np.linalg.norm((np.array(pos)-p) * VOXEL_NATIVE, axis=1)      # um, centro-a-centro
                thr = csf * (r + np.array(radii))
                # `min_sep_um` era um parametro MORTO (aceito na assinatura e nunca usado -> quem
                # passava recebia no-op silencioso). Agora e' um PISO absoluto de separacao (um).
                if min_sep_um: thr = np.maximum(thr, float(min_sep_um))
                if (d < thr).any(): continue                                      # colide -> rejeita
            pos.append(p); radii.append(r)
        pos = np.array(pos, np.float32); radii = np.array(radii, np.float32)

    # EMISSAO/ESPALHAMENTO: cada nucleo emite luz que se espalha no meio (cauda da PSF + scattering no
    # liquido) -> halo dim que preenche o espaco entre celulas com o brilho ROXO do real. Modelado como
    # uma 2a camada: o CENTRO da celula deposita uma fonte pontual que depois e' borrada LARGO e somada
    # (acumula onde ha muitas celulas = glow; escuro onde nao ha). O nucleo NITIDO vai por maximo.
    halo_amp = float(os.environ.get("SYNTH_HALO_AMP", "0.18"))       # BAIXO de proposito: e' o halo (nao o brilho)
    #                                                                  que preenchia as folgas e fundia as celulas
    halo_um = float(os.environ.get("SYNTH_HALO_UM", "2.6"))   # 3.5 deixava o piso longe da
    #                                                     celula em 0.24 vs 0.16 do real          # alcance do espalhamento (um)
    emis = np.zeros((Z, Y, X), np.float32)                          # camada de emissao (soma)
    cens = []
    # BRILHO por celula: VARIADO e (na maioria) ABAIXO da saturacao. Antes era uniform(0.72,1.0) e apos
    # PSF+emissao TODAS batiam em 1.0 -> distribuicao de pico degenerada (pico unico em 1.0), diferente
    # do real que tem brilho variado (cauda longa). Log-normal-ish: media modesta, cauda p/ cima.
    # BRILHO: o REAL SATURA MUITO (~41% das celulas com pico ~1.0, medido no histograma da galeria).
    # O erro NAO era brilho alto -- era a SEPARACAO. O aspecto "massa branca fundida" vinha do HALO
    # preenchendo as folgas entre celulas, nao do brilho. Baixar o brilho p/ 0.35-0.78 deixou o synth
    # APAGADO (2.9% de saturacao vs 41% do real). Config certa = brilho ALTO (casa a saturacao real)
    # + halo BAIXO (mantem as folgas escuras, celulas distintas): 26.5% sat, recall 0.905.
    b_lo = float(os.environ.get("SYNTH_CELL_BRIGHT_LO", "0.42"))
    b_hi = float(os.environ.get("SYNTH_CELL_BRIGHT_HI", "0.92"))
    for p, r_um in zip(pos, radii):                                 # raio JA sorteado na colocacao (consistente)
        bright = float(np.clip(rng.normal((b_lo+b_hi)/2, (b_hi-b_lo)/3), b_lo, b_hi))   # variacao real (nao satura tudo)
        pr = np.round(p); frac = np.asarray(p, float) - pr                              # deslocamento SUB-VOXEL
        esf = _sphere_native(float(r_um), rng, sharp=sharp, frac=frac) * bright         # nucleo NITIDO (sub-voxel)
        rz, ry, rx = np.array(esf.shape) // 2
        z, y, x = pr.astype(int)
        z0, z1 = max(0, z-rz), min(Z, z+rz+1); y0, y1 = max(0, y-ry), min(Y, y+ry+1); x0, x1 = max(0, x-rx), min(X, x+rx+1)
        if z1 <= z0 or y1 <= y0 or x1 <= x0: continue
        es = esf[z0-(z-rz):z0-(z-rz)+(z1-z0), y0-(y-ry):y0-(y-ry)+(y1-y0), x0-(x-rx):x0-(x-rx)+(x1-x0)]
        vol[z0:z1, y0:y1, x0:x1] = np.maximum(vol[z0:z1, y0:y1, x0:x1], es)     # nucleo = fonte pontual
        if 0 <= z < Z and 0 <= y < Y and 0 <= x < X:
            # deposita a luz emitida no centro. Usa o BRILHO da celula (nao `es.max()`, que vem do patch
            # ja CORTADO na borda -> celulas de borda emitiam menos que identicas no interior).
            emis[z, y, x] += bright * halo_amp
        cens.append([float(p[0]), float(p[1]), float(p[2])])        # centroide VERDADEIRO (float), nao arredondado
    # espalha a emissao LARGO (anisotropico) e soma ao volume -> glow roxo entre celulas
    sz2 = halo_um * psf_z_ratio / VOXEL_NATIVE[0]; sy2 = halo_um / VOXEL_NATIVE[1]; sx2 = halo_um / VOXEL_NATIVE[2]
    emis = gaussian_filter(emis, (sz2, sy2, sx2))
    # Normaliza pela RESPOSTA DE UMA FONTE (constante do kernel), nao pelo maximo global. Com o max
    # global, o glow de cada celula passava a depender do aglomerado mais brilhante do volume -> dois
    # volumes com densidades diferentes ganhavam brilho por celula diferente. Assim o halo ACUMULA com
    # a densidade (denso brilha, esparso fica escuro), que e' o comportamento fisico desejado.
    kern_peak = 1.0 / ((2.0*np.pi)**1.5 * sz2 * sy2 * sx2 + 1e-12)
    emis = np.clip(emis / max(kern_peak, 1e-12), 0.0, 1.0)
    vol = vol + emis                                                             # ADICIONA a luz emitida (acumula)

    # PSF OPTICA anisotropica na grade nativa (sigma em voxels = sigma_um / voxel_um)
    sz = psf_xy * psf_z_ratio / VOXEL_NATIVE[0]; sy = psf_xy / VOXEL_NATIVE[1]; sx = psf_xy / VOXEL_NATIVE[2]
    vol = np.clip(gaussian_filter(vol, (sz, sy, sx)), 0, 1)
    # ruido de camera (shot + leitura) na resolucao nativa
    pe = float(os.environ.get("SYNTH_POISSON_PE", "300")); rd = float(os.environ.get("SYNTH_READNOISE", "0.006"))
    vol = np.clip(rng.poisson(np.clip(vol, 0, 1)*pe)/pe + rng.normal(0, rd, vol.shape), 0, 1).astype(np.float32)

    # POOL 4x em XY. ATENCAO -- INCONSISTENCIA REAL CORRIGIDA: este gerador poolava por MEDIA, mas o
    # pipeline que PONTUA (`dog_max_infer.pool_xy`) poola por STRIDE (`[::1,::4,::4]`). Treinar/medir
    # num pooling e' fazer deploy noutro: a media suaviza o ruido (SNR maior) e o stride nao, entao o
    # dado sintetico chegava mais limpo do que o detector realmente ve. Default = STRIDE (= deploy).
    if os.environ.get("SYNTH_POOL_MODE", "stride") == "mean":
        vol_p = vol[:, :(Y//POOL)*POOL, :(X//POOL)*POOL].reshape(Z, Y//POOL, POOL, X//POOL, POOL).mean((2, 4)).astype(np.float32)
    else:
        vol_p = vol[::1, ::POOL, ::POOL].astype(np.float32)
    cens_p = (np.array(cens, np.float32) / np.array([1, POOL, POOL])) if cens else np.zeros((0, 3))
    return vol, vol_p, np.array(cens, np.float32), cens_p


def run_native_compare():
    """Compara REAL vs SYNTH nas 2 resolucoes (nativa e pooled), no MESMO enquadramento. Usa os
    helpers do gerador base (find_train_dir/read_frame/norm/dog_detect_precise) via namespace."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    td = find_train_dir(); assert td, "train dir nao encontrado"                   # noqa: F821
    vids = sorted(p.stem for p in td.glob("*.zarr"))
    name = vids[0]; zp = td / f"{name}.zarr"; gp = geff_for(td, name)               # noqa: F821
    nodes = read_geff_nodes(gp); t = sorted(nodes)[len(nodes)//2]                   # noqa: F821
    real_native = norm(read_frame(zp, t)[0])                                        # noqa: F821  (Z,256,256)
    real_pooled = real_native[:, :(real_native.shape[1]//POOL)*POOL, :(real_native.shape[2]//POOL)*POOL]
    real_pooled = real_pooled.reshape(real_native.shape[0], real_native.shape[1]//POOL, POOL,
                                      real_native.shape[2]//POOL, POOL).mean((2, 4)).astype(np.float32)
    # densidade real (DoG no pooled real)
    n_real = len(dog_detect_precise(real_pooled)[0])                               # noqa: F821
    print(f"  real {name} t={t}: nativo {real_native.shape} -> pooled {real_pooled.shape} | {n_real} celulas (DoG)", flush=True)
    sn, sp, cn, cp = gen_volume_native(zshape=real_native.shape[0], xyshape=real_native.shape[1],
                                       n_cells=int(n_real*1.3), rng=np.random.default_rng(1))
    n_syn = len(dog_detect_precise(sp)[0])                                         # noqa: F821
    print(f"  synth: gerado {sn.shape} -> pooled {sp.shape} | {n_syn} celulas (DoG) vs {n_real} real", flush=True)

    fig, ax = plt.subplots(2, 4, figsize=(19, 10))
    panels = [("REAL nativo MIP-Z", real_native.max(0), "#2166ac"), ("REAL nativo corte", real_native[real_native.shape[0]//2], "#2166ac"),
              ("REAL pooled MIP-Z (DoG)", real_pooled.max(0), "#2166ac"), ("REAL pooled corte", real_pooled[real_pooled.shape[0]//2], "#2166ac"),
              ("SYNTH nativo MIP-Z", sn.max(0), "#b2182b"), ("SYNTH nativo corte", sn[sn.shape[0]//2], "#b2182b"),
              ("SYNTH pooled MIP-Z (DoG)", sp.max(0), "#b2182b"), ("SYNTH pooled corte", sp[sp.shape[0]//2], "#b2182b")]
    for a, (ttl, img, col) in zip(ax.ravel(), panels):
        a.imshow(img, cmap="magma", vmin=0, vmax=1); a.set_title(ttl, fontsize=11, weight="bold", color=col); a.axis("off")
    plt.suptitle(f"REAL vs SYNTH nas 2 resolucoes -- gerado NATIVO e poolado como o real ({n_syn} vs {n_real} cel)",
                 weight="bold", fontsize=15)
    plt.tight_layout(); plt.savefig(NFIG / "native_compare.png", dpi=110, bbox_inches="tight"); plt.close()   # noqa: F821
    print(f"  figura -> {NFIG/'native_compare.png'}", flush=True)                   # noqa: F821


def _cell_radius_um(vol_pooled, dets):
    """Raio APARENTE (meia-altura) de cada celula detectada, em um. Mede o tamanho que o DoG ve."""
    EFF = VOXEL_NATIVE[0]  # pooled ~ isotropico 1.625 um/voxel
    out = []
    for c in np.atleast_2d(dets).astype(int):
        z, y, x = c[:3]
        sub = vol_pooled[z, max(0,y-6):y+7, max(0,x-6):x+7]
        if sub.size < 9: continue
        pk = float(sub.max())
        if pk <= 1e-3: continue
        area = float((sub > 0.5*pk).sum())          # pixels acima da meia-altura
        out.append(np.sqrt(area/np.pi) * EFF)        # raio equivalente (um)
    return np.array(out, np.float32)


def run_native_similarity():
    from scipy.stats import ks_2samp
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    td = find_train_dir(); assert td, "train dir nao encontrado"                    # noqa: F821
    vids = sorted(p.stem for p in td.glob("*.zarr"))
    # amostra REAL de varios videos/frames -> distribuicoes robustas
    real_szs, real_ints, real_nn, real_counts = [], [], [], []
    rng = np.random.default_rng(0)
    for name in vids[:12]:
        zp = td/f"{name}.zarr"; gp = geff_for(td, name)                             # noqa: F821
        if gp is None or not zp.exists(): continue
        try: nodes = read_geff_nodes(gp)                                            # noqa: F821
        except Exception: continue
        for t in sorted(nodes)[::max(1,len(nodes)//3)][:3]:
            rn = norm(read_frame(zp, t)[0])                                         # noqa: F821
            rp = rn[:, :(rn.shape[1]//POOL)*POOL, :(rn.shape[2]//POOL)*POOL].reshape(rn.shape[0], rn.shape[1]//POOL, POOL, rn.shape[2]//POOL, POOL).mean((2,4)).astype(np.float32)
            det = dog_detect_precise(rp)[0]                                         # noqa: F821
            real_counts.append(len(det)); real_szs.append(_cell_radius_um(rp, det))
            real_ints.append(rp.ravel()[::7]); real_nn.append(nn_um(det))          # noqa: F821
    real_szs = np.concatenate(real_szs); real_ints = np.concatenate(real_ints); real_nn = np.concatenate([a for a in real_nn if len(a)])
    n_real = int(np.median(real_counts))
    def sim(a, b): return 1 - float(ks_2samp(a[:6000], b[:6000]).statistic)
    # VARREDURA DE TAMANHO: acha o raio que MAXIMIZA a similaridade de tamanho com o real (data-driven,
    # nao chute). Base 3.6-5.2um; testa multiplicadores. Tambem reforca a emissao (glow) p/ intensidade.
    if os.environ.get("SYNTH_SIZE_SWEEP", "0") == "1":
        print("\n  === VARREDURA DE TAMANHO (similaridade com o real) ===", flush=True)
        best = None
        for mult in [0.75, 1.0, 1.25, 1.5, 1.75]:
            os.environ["SYNTH_CELL_RUM_LO"] = f"{3.6*mult:.2f}"; os.environ["SYNTH_CELL_RUM_HI"] = f"{5.2*mult:.2f}"
            szs = []
            for i in range(6):
                _, sp, _, _ = gen_volume_native(zshape=64, xyshape=256, n_cells=int(n_real*1.25), rng=np.random.default_rng(200+i))
                szs.append(_cell_radius_um(sp, dog_detect_precise(sp)[0]))         # noqa: F821
            szs = np.concatenate(szs); s = sim(real_szs, szs)
            print(f"    raio x{mult} ({3.6*mult:.1f}-{5.2*mult:.1f}um): sim_tamanho {s:.3f} (synth med {np.median(szs):.2f} vs real {np.median(real_szs):.2f})", flush=True)
            if best is None or s > best[1]: best = (mult, s)
        print(f"  MELHOR tamanho: x{best[0]} (sim {best[1]:.3f})", flush=True)
        os.environ["SYNTH_CELL_RUM_LO"] = f"{3.6*best[0]:.2f}"; os.environ["SYNTH_CELL_RUM_HI"] = f"{5.2*best[0]:.2f}"
    # SYNTH final com o tamanho escolhido (ou o do env)
    syn_szs, syn_ints, syn_nn, syn_counts = [], [], [], []
    for i in range(8):
        _, sp, _, _ = gen_volume_native(zshape=64, xyshape=256, n_cells=int(n_real*1.25), rng=np.random.default_rng(100+i))
        det = dog_detect_precise(sp)[0]                                            # noqa: F821
        syn_counts.append(len(det)); syn_szs.append(_cell_radius_um(sp, det))
        syn_ints.append(sp.ravel()[::7]); syn_nn.append(nn_um(det))               # noqa: F821
    syn_szs = np.concatenate(syn_szs); syn_ints = np.concatenate(syn_ints); syn_nn = np.concatenate([a for a in syn_nn if len(a)])
    axes_sim = {"tamanho_celula(um)": (sim(real_szs, syn_szs), np.median(real_szs), np.median(syn_szs)),
                "intensidade_voxel": (sim(real_ints, syn_ints), np.median(real_ints), np.median(syn_ints)),
                "vizinho_proximo(um)": (sim(real_nn, syn_nn), np.median(real_nn), np.median(syn_nn)),
                "densidade(DoG/vol)": (1 - min(1, abs(np.median(real_counts)-np.median(syn_counts))/max(np.median(real_counts),1)), np.median(real_counts), np.median(syn_counts))}
    print(f"\n  === SIMILARIDADE synth vs real (raio celula {os.environ.get('SYNTH_CELL_RUM_LO','?')}-{os.environ.get('SYNTH_CELL_RUM_HI','?')}um) ===", flush=True)
    for k,(s,r,y) in axes_sim.items():
        print(f"    {k:<22}: {s:.3f}  (real {r:.2f} vs synth {y:.2f})", flush=True)
    media = float(np.mean([v[0] for v in axes_sim.values()]))
    print(f"    {'MEDIA':<22}: {media:.3f}", flush=True)
    # figura: histogramas comparados
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
    ax[0].hist(real_szs, bins=30, range=(0,6), density=True, alpha=.55, label="real", color="#3182bd"); ax[0].hist(syn_szs, bins=30, range=(0,6), density=True, alpha=.55, label="synth", color="#e6550d")
    ax[0].set_title(f"TAMANHO da celula (um)  sim {axes_sim['tamanho_celula(um)'][0]:.2f}", fontsize=11, weight="bold"); ax[0].legend()
    ax[1].hist(real_ints, bins=40, range=(0,1), density=True, alpha=.55, label="real", color="#3182bd"); ax[1].hist(syn_ints, bins=40, range=(0,1), density=True, alpha=.55, label="synth", color="#e6550d")
    ax[1].set_title(f"intensidade de voxel  sim {axes_sim['intensidade_voxel'][0]:.2f}", fontsize=11, weight="bold"); ax[1].legend()
    ax[2].hist(real_nn, bins=30, range=(0,25), density=True, alpha=.55, label="real", color="#3182bd"); ax[2].hist(syn_nn, bins=30, range=(0,25), density=True, alpha=.55, label="synth", color="#e6550d")
    ax[2].set_title(f"vizinho mais proximo (um)  sim {axes_sim['vizinho_proximo(um)'][0]:.2f}", fontsize=11, weight="bold"); ax[2].legend()
    plt.suptitle(f"SIMILARIDADE synth (raio -25%) vs real -- media {media:.3f}", weight="bold", fontsize=14)
    plt.tight_layout(); plt.savefig(NFIG / "native_similarity.png", dpi=110, bbox_inches="tight"); plt.close()   # noqa: F821
    json.dump({k: {"sim": v[0], "real": v[1], "synth": v[2]} for k,v in axes_sim.items()} | {"media": media},
              open(NFIG / "native_similarity.json", "w"), indent=2, default=float)                                # noqa: F821
    print(f"  figura -> {NFIG/'native_similarity.png'}", flush=True)


def run_layout_compare():
    """Compara a DISTRIBUICAO ESPACIAL: real vs synth com LAYOUT REAL (posicoes do DoG real) vs synth
    com cluster ALEATORIO. O layout real casa a distribuicao por CONSTRUCAO (tecido, borda, gradiente)."""
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    td = find_train_dir(); assert td, "train dir nao encontrado"                    # noqa: F821
    vids = sorted(p.stem for p in td.glob("*.zarr"))
    fig, ax = plt.subplots(2, 4, figsize=(19, 9.5))
    for r, name in enumerate(vids[:2]):
        zp = td/f"{name}.zarr"; gp = geff_for(td, name)                            # noqa: F821
        nodes = read_geff_nodes(gp); t = sorted(nodes)[len(nodes)//2]              # noqa: F821
        nat = norm(read_frame(zp, t)[0])                                           # noqa: F821
        vp = nat[:, :(nat.shape[1]//POOL)*POOL, :(nat.shape[2]//POOL)*POOL].reshape(nat.shape[0], nat.shape[1]//POOL, POOL, nat.shape[2]//POOL, POOL).mean((2,4)).astype(np.float32)
        det_p = dog_detect_precise(vp)[0]                                          # posicoes reais (pooled)   # noqa: F821
        det_native = np.atleast_2d(det_p).astype(np.float32) * np.array([1, POOL, POOL])   # -> nativo
        # synth com LAYOUT REAL (posicoes do DoG real)
        _, sp_real, _, _ = gen_volume_native(zshape=nat.shape[0], xyshape=nat.shape[1], cens_native=det_native, rng=np.random.default_rng(7))
        # synth com CLUSTER ALEATORIO (mesma contagem)
        os.environ["SYNTH_CLUSTER"] = "1"
        _, sp_rand, _, _ = gen_volume_native(zshape=nat.shape[0], xyshape=nat.shape[1], n_cells=len(det_native), rng=np.random.default_rng(7))
        for c, (ttl, img, col) in enumerate([(f"REAL ({name[:9]})", vp.max(0), "#2166ac"),
                                             ("SYNTH layout REAL", sp_real.max(0), "#1a9850"),
                                             ("SYNTH cluster aleatorio", sp_rand.max(0), "#b2182b"),
                                             ("posicoes reais (DoG)", None, "#2166ac")]):
            if img is not None:
                ax[r, c].imshow(img, cmap="magma", vmin=0, vmax=1); ax[r, c].axis("off")
            else:
                ax[r, c].scatter(det_p[:, 2], det_p[:, 1], s=8, c="#2166ac"); ax[r, c].set_xlim(0, vp.shape[2]); ax[r, c].set_ylim(vp.shape[1], 0); ax[r, c].set_aspect("equal")
            ax[r, c].set_title(ttl, fontsize=11, weight="bold", color=col)
    plt.suptitle("DISTRIBUICAO ESPACIAL: layout REAL (posicoes do DoG) casa por construcao vs cluster aleatorio", weight="bold", fontsize=14)
    plt.tight_layout(); plt.savefig(NFIG / "layout_compare.png", dpi=105, bbox_inches="tight"); plt.close()   # noqa: F821
    print(f"  figura -> {NFIG/'layout_compare.png'}", flush=True)


if os.environ.get("SYNTH_NATIVE_COMPARE", "0") == "1":
    NFIG = OUT / "figuras_native"; NFIG.mkdir(parents=True, exist_ok=True)          # noqa: F821
    run_native_compare()
    if os.environ.get("SYNTH_NATIVE_SIM", "0") == "1": run_native_similarity()
    if os.environ.get("SYNTH_NATIVE_LAYOUT", "0") == "1": run_layout_compare()

if os.environ.get("SYNTH_NATIVE_SMOKE", "0") == "1":
    v, vp, c, cp = gen_volume_native(zshape=32, xyshape=128, n_cells=60, rng=np.random.default_rng(0))
    print(f"SMOKE nativo {v.shape} range[{v.min():.3f},{v.max():.3f}] -> pooled {vp.shape} | {len(c)} celulas")
    print("SMOKE NATIVO OK")

# PUBLIC SYNTHETIC DATASET BUILDER (target ~20 GB)

NOTE: this file is written in English on purpose -- it is published on Kaggle and read by the
community, unlike the rest of this repository.

It emits three products, each carrying supervision the real ground truth does not provide:

 A) STATIC VOLUMES   -- (Z,256,256) uint16 at the microscope's NATIVE resolution, plus centroids
    with SUB-VOXEL precision. Density is sampled from the REAL measured distribution, not fixed.
 B) TIME SEQUENCES   -- (T,Z,256,256) plus a complete lineage: nodes, edges and labelled DIVISIONS.
    This is the differentiator: the real ground truth holds ~304 sparse divisions across 199
    videos; here we emit tens of thousands.
 C) METADATA         -- the physical configuration used, a manifest and the achieved statistics.

Conventions that match the official competition pipeline:
  native voxel (z,y,x) = (1.625, 0.40625, 0.40625) um
  evaluator pooling    = STRIDE 4x in XY (`vol[:, ::4, ::4]`), NOT a block mean. A block mean
                         averages noise away and would hand the user data that is *cleaner* than
                         what a detector actually sees.
Gate: DSBUILD_RUN=1. Size target: DS_TARGET_GB.

In [ ]:
import os, json, time
import numpy as np


def _to_u16(v):
    """float [0,1] -> uint16. Halves the size versus float32 with no visible loss (the real data is
    uint16 as well)."""
    return np.clip(v * 65535.0, 0, 65535).astype(np.uint16)


def _real_stats(td, vids, n_frames=2):
    """REAL per-field count distribution + tissue-shell geometry (drives the generative placement)."""
    counts, pts = [], []
    for name in vids:
        for t in range(n_frames):
            try: vp = pool_xy(norm(read_frame(td/f"{name}.zarr", t)[0]))            # noqa: F821
            except Exception: continue
            d = dog_detect_precise(vp)[0]                                           # noqa: F821
            counts.append(len(d))
            if len(d) and len(pts) < 8: pts.append(np.atleast_2d(d)*np.array([1, POOL, POOL]))  # noqa: F821
    shell = fit_shell(np.concatenate(pts)) if pts else None                         # noqa: F821
    return np.array(counts, float), shell


def run_dataset_build():
    OUTD = OUT / "biohub_synthetic"                                                 # noqa: F821
    (OUTD/"static").mkdir(parents=True, exist_ok=True)
    (OUTD/"sequences").mkdir(parents=True, exist_ok=True)
    td = find_train_dir(); assert td, "train directory not found"                   # noqa: F821
    vids = sorted(p.stem for p in td.glob("*.zarr")); by = {}
    for v in vids: by.setdefault(v.split("_")[0], []).append(v)
    embs = sorted(by)
    # Calibrate on one embryo only, leaving the other free for the user's own validation.
    calib = sorted(by[embs[0]])[:10]
    counts, shell = _real_stats(td, calib)
    print(f"=== PUBLIC DATASET | real density {counts.min():.0f}-{counts.max():.0f} "
          f"| shell {shell['kind'] if shell else '?'} ===", flush=True)

    TARGET = float(os.environ.get("DS_TARGET_GB", "18.5")) * (1024**3)   # margem sob o teto de 20 GB
    T_SEQ = int(os.environ.get("DS_SEQ_LEN", "6"))
    SEQ_FRAC = float(os.environ.get("DS_SEQ_FRAC", "0.35"))     # share of the budget spent on sequences
    BUDGET_S = float(os.environ.get("DS_BUDGET_H", "8.5")) * 3600
    os.environ.setdefault("SYNTH_PLACE_MODE", "shell")
    t0 = time.time(); total = 0
    manifest = {"static": [], "sequences": []}

    # ---------------- A) STATIC VOLUMES ----------------
    i = 0
    while total < TARGET*(1-SEQ_FRAC) and (time.time()-t0) < BUDGET_S*(1-SEQ_FRAC):
        rg = np.random.default_rng(100000+i)
        nc = int(max(20, rg.choice(counts)*1.25))
        nat, _, cen, _ = gen_volume_native(zshape=64, xyshape=256, n_cells=nc,      # noqa: F821
                                           rng=rg, shell=shell)
        fp = OUTD/"static"/f"vol_{i:05d}.npz"
        np.savez(fp, volume=_to_u16(nat), centroids=cen.astype(np.float32),
                 voxel_um=np.array([1.625, 0.40625, 0.40625], np.float32))
        sz = fp.stat().st_size; total += sz
        manifest["static"].append(dict(file=f"static/vol_{i:05d}.npz", n_cells=int(len(cen)),
                                       shape=list(nat.shape), bytes=int(sz)))
        i += 1
        if i % 25 == 0:
            with open(OUTD/"progress.log", "a") as fh:
                fh.write(f"static {i} | {total/1024**3:.2f} GB | {(time.time()-t0)/60:.0f} min\n")
        if i % 400 == 0:
            print(f"  static {i} | {total/1024**3:.2f} GB", flush=True)
    print(f"  A) {i} static volumes, {total/1024**3:.2f} GB", flush=True)

    # ---------------- B) TIME SEQUENCES WITH LINEAGE ----------------
    # Motion is calibrated on the REAL lineage edges of the .geff files: median step 1.86 um/frame
    # (p90 3.83), lag-1 directional persistence +0.30, sister separation ~7.24 um. Division is
    # DELIBERATELY over-sampled: it is the supervision that is scarce in the real data (~0.26% of
    # nodes) and the main reason this dataset exists.
    STEP_MED, PERSIST, SISTER = 1.86, 0.30, 7.24
    VOX = np.array([1.625, 0.40625, 0.40625])
    s_step = (STEP_MED/1.5382)*1.10
    j = 0
    while total < TARGET and (time.time()-t0) < BUDGET_S:
        rg = np.random.default_rng(500000+j)
        n0 = int(max(20, rg.choice(counts)*1.25))
        pos, radii = sample_shell_positions(n0, (64, 256, 256), rg, shell=shell,    # noqa: F821
                                            r_lo=3.5, r_hi=5.5)
        if len(pos) < 10: j += 1; continue
        # per-cell speed scale (log-normal) reproduces the heavy tail of the real step distribution
        sc = np.exp(rg.normal(0, 0.55, len(pos)))
        vel = rg.normal(0, s_step, (len(pos), 3)) * sc[:, None]
        flow = rg.normal(0, 1, 3); flow /= np.linalg.norm(flow)+1e-9   # collective tissue drift
        div_p = float(os.environ.get("DS_DIV_RATE", "0.05"))           # over-sampled (real ~0.0026)
        tracks = [dict(pos=p.copy(), vel=vel[k], sc=sc[k], tid=k) for k, p in enumerate(pos)]
        nodes, edges, divs = [], [], []
        vols = np.zeros((T_SEQ, 64, 64, 64), np.uint16)                # pooled, to fit the budget
        nid = 0; prev_ids = {}
        for t in range(T_SEQ):
            cens = np.array([tr["pos"] for tr in tracks], np.float32)
            nat, _, cc, _ = gen_volume_native(zshape=64, xyshape=256, cens_native=cens,  # noqa: F821
                                              rng=np.random.default_rng(900000+j*97+t), shell=shell)
            vols[t] = _to_u16(pool_xy(nat))                                          # noqa: F821
            cur_ids = {}
            for k, tr in enumerate(tracks):
                nodes.append([t, tr["pos"][0], tr["pos"][1], tr["pos"][2], tr["tid"]])
                cur_ids[k] = nid; nid += 1
            for k, tr in enumerate(tracks):
                if tr.get("parent_slot") is not None and tr["parent_slot"] in prev_ids:
                    edges.append([prev_ids[tr["parent_slot"]], cur_ids[k]])
                    if tr.get("is_div"): divs.append(prev_ids[tr["parent_slot"]])
            prev_ids = cur_ids
            if t == T_SEQ-1: break
            nxt = []
            for k, tr in enumerate(tracks):
                # Ornstein-Uhlenbeck step: persistent velocity + collective flow
                v = PERSIST*tr["vel"] + np.sqrt(1-PERSIST**2)*rg.normal(0, s_step*tr["sc"], 3)
                v = v + 0.35*s_step*flow
                if rg.random() < div_p:                                # DIVISION
                    d = rg.normal(0, 1, 3); d /= np.linalg.norm(d)+1e-9
                    half = 0.5*max(2.0, rg.normal(SISTER, 1.6))*d/VOX
                    for sgn in (1, -1):
                        p2 = np.clip(tr["pos"]+v+sgn*half, [4, 12, 12], [59, 243, 243])
                        nxt.append(dict(pos=p2, vel=v*0.5, sc=tr["sc"], tid=tr["tid"],
                                        parent_slot=k, is_div=True))
                else:
                    p2 = np.clip(tr["pos"]+v, [4, 12, 12], [59, 243, 243])
                    nxt.append(dict(pos=p2, vel=v, sc=tr["sc"], tid=tr["tid"],
                                    parent_slot=k, is_div=False))
            tracks = nxt
        fp = OUTD/"sequences"/f"seq_{j:04d}.npz"
        np.savez(fp, volumes=vols, nodes=np.array(nodes, np.float32),
                 edges=np.array(edges, np.int32), divisions=np.array(sorted(set(divs)), np.int32),
                 voxel_um_pooled=np.array([1.625, 1.625, 1.625], np.float32))
        sz = fp.stat().st_size; total += sz
        manifest["sequences"].append(dict(file=f"sequences/seq_{j:04d}.npz", T=int(T_SEQ),
                                          n_nodes=len(nodes), n_edges=len(edges),
                                          n_divisions=len(set(divs)), bytes=int(sz)))
        j += 1
        if j % 10 == 0:
            with open(OUTD/"progress.log", "a") as fh:
                fh.write(f"sequences {j} | {total/1024**3:.2f} GB | {(time.time()-t0)/60:.0f} min\n")
        if j % 500 == 0:
            print(f"  sequences {j} | {total/1024**3:.2f} GB", flush=True)
    print(f"  B) {j} time sequences | TOTAL {total/1024**3:.2f} GB", flush=True)

    nd = sum(m["n_divisions"] for m in manifest["sequences"])
    nn = sum(m["n_nodes"] for m in manifest["sequences"])
    meta = dict(
        n_static=i, n_sequences=j, total_gb=total/1024**3,
        seq_len=T_SEQ, total_divisions=nd, total_nodes=nn,
        division_rate=nd/max(nn, 1),
        voxel_native_um=[1.625, 0.40625, 0.40625],
        pooling="stride 4x in XY (vol[:, ::4, ::4]) -- identical to the official evaluator",
        real_count_range=[float(counts.min()), float(counts.max())],
        shell_kind=(shell or {}).get("kind"),
        motion_calibration=dict(median_step_um_per_frame=STEP_MED, lag1_persistence=PERSIST,
                                sister_separation_um=SISTER),
        note_division_rate="Divisions are deliberately over-sampled; the real ground-truth rate is "
                           "about 0.26% of nodes. Re-weight your loss if you need calibrated priors.",
        config={k: v for k, v in os.environ.items() if k.startswith("SYNTH_")},
    )
    json.dump(meta, open(OUTD/"metadata.json", "w"), indent=2, default=float)
    json.dump(manifest, open(OUTD/"manifest.json", "w"), indent=2, default=float)
    print(f"\n  labelled divisions: {nd} across {nn} nodes ({100*nd/max(nn,1):.2f}%) "
          f"-- the real ground truth holds ~0.26%", flush=True)
    print(f"  -> {OUTD}", flush=True)


if os.environ.get("DSBUILD_RUN", "0") == "1":
    run_dataset_build()